<a href="https://colab.research.google.com/github/Dssingh17/Data-Science/blob/main/LIDC_nodule_v2_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lung-nodule malignancy-risk classification (LIDC-IDRI) — v2: reviewer-complete research notebook

**Model under study:** three parallel 3D-CNN branches (32³ / 48³ / 64³) + 28 radiomics features + CBAM -> MLP -> benign / malignant (radiologist-assessed malignancy risk).

**Principle of this version.** Nothing from the previous notebook is deleted (fused model, image-only, radiomics-only, size-only, size-controlled analysis, label shuffle, Grad-CAM, input-swap, nodule-removal, error audit are all kept). Those analyses are *necessary but not sufficient*, so this notebook adds everything the final reviewer demanded, so that every reviewer point is answered by a concrete, runnable cell.

| Reviewer item | Where answered |
|---|---|
| A. External validation (frozen model, fixed threshold, LNDb / LUNGx) | §12 (+ §10 freezing) |
| B. Full component-wise ablation (scales, CBAM, radiomics) | §5, §7 |
| C. Significance tests (paired DeLong, patient bootstrap, CIs for every model, effect sizes, multi-seed) | §6 |
| D. Calibration, Brier, ECE, decision-curve, clinical operating points, threshold discussion | §8 |
| E. Stronger reference standard (pathology/follow-up subgroup, label-robustness, task renamed) | §9 |
| F. Better explainability (Grad-CAM, occlusion, integrated gradients, counterfactual patch, radiomics attribution, radiologist review pack) | §11 |
| G. Public reproducibility package (code, configs, seeds, fold IDs, README, hashes) | §14 |
| Writing guidance (cautious claims, SAP, reproducibility statement) | §0, §14, §15 |

**Run order:** top to bottom. `MODE='test'` (about 40 patients, 3 epochs) first to check the pipeline; then `MODE='full'`. Everything is resumable: patches, per-fold predictions, checkpoints and statistics are cached in `WORK_DIR` on Drive.

**Compute budget (full mode):** the ablation is 10 CNN variants x `len(SEEDS)` seeds x 5 folds (+ 2 cheap tabular models). On a T4 this is long; the order of execution is *primary seed first for all variants*, so a partial run is already usable. Reduce `CFG['SEEDS']` to 1 seed to get a first full pass.

## 0. Predefined statistical analysis plan (SAP) — fixed *before* any new result is looked at

* **Task name (use this in the paper):** *radiologist-assessed malignancy-risk classification* (labels = mean radiologist malignancy score; 1-2 benign, 4-5 malignant, 3 excluded). Not definitive cancer diagnosis.
* **Data roles.** Patient-level stratified **final holdout** (20 % of patients, fixed by `SPLIT_SEED`) and a **development set** (80 %). Model selection, early stopping and thresholds use the development set only. The holdout is evaluated once per model. Both are *internal* LIDC-IDRI evaluations — *not* independent validation.
* **Primary endpoint:** holdout AUC of the full model (`full`). **Primary comparisons (family 1, Holm-corrected, alpha = 0.05):** `full` vs `rad_only`; `full` vs `img_3s_cbam` (image-only). **Secondary family:** CBAM effect, multi-scale effect, size baseline. **Ablation family:** every ablation variant vs `full`.
* **Tests:** paired DeLong (nodule-level, independence assumed) *and* patient-cluster bootstrap (B = 2000, percentile 95 % CI, two-sided bootstrap p). The patient-cluster bootstrap is the primary inference because nodules from one patient are correlated.
* **CIs:** patient-cluster bootstrap 95 % CIs for every model, on both development out-of-fold and holdout predictions.
* **Seeds:** all CNN variants trained with `CFG['SEEDS']`; report mean ± SD across *training seeds* (separately from fold SD).
* **Thresholds:** Youden on development out-of-fold predictions only; screening (sens >= 95 %) and triage (spec >= 95 %) thresholds also from development only. **Never tuned on holdout or external data.**
* **External:** frozen five-model ensemble + frozen threshold, no retraining, no checkpoint selection, no threshold tuning (hash-verified). If the fused model is not significantly better than radiomics-only, the manuscript is repositioned as a rigorous benchmark / evaluation study (auto-generated in §15).

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
except ValueError as e:
    if "Mountpoint must not already contain files" in str(e):
        print("Google Drive is already mounted and contains files. Proceeding safely.")
    else:
        raise e

In [ ]:
from pathlib import Path

# Start with the experiment directory from your earlier code.
# Change this if your older experiment used a different folder.
BASE_DIR = Path("/content/drive/MyDrive/lidc_work")

extensions = {
    ".pt", ".pth", ".pkl", ".pickle", ".joblib",
    ".npy", ".npz", ".csv", ".json", ".yaml",
    ".yml", ".parquet", ".txt"
}

if not BASE_DIR.exists():
    print("Directory not found:", BASE_DIR)
else:
    for path in sorted(BASE_DIR.rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"{size_mb:8.2f} MB  {path.relative_to(BASE_DIR)}")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE_DIR = Path("/content/drive/MyDrive/lidc_work")

# 1. Inspect the most important CSV files
csv_files = [
    "eligible_nodules.csv",
    "final_holdout_patients.csv",
    "holdout_per_nodule.csv",
    "final_holdout_bootstrap_ci.csv",
    "image_only_holdout_analysis.csv",
    "size_controlled_analysis.csv",
    "results_full.csv",
    "results_image_only.csv",
    "results_label_shuffle.csv",
    "results_test.csv",
]

for name in csv_files:
    path = BASE_DIR / name

    if path.exists():
        try:
            df = pd.read_csv(path)
            print(f"\n{'=' * 60}")
            print(f"{name}: {df.shape}")
            print("Columns:", df.columns.tolist())
            print(df.head(3).to_string(index=False))
        except pd.errors.EmptyDataError:
            print(f"\n{'=' * 60}")
            print(f"{name}: File is empty (0 bytes) and could not be parsed.")
    else:
        print(f"\nNot found: {name}")


# 2. Inspect saved model-checkpoint structure
checkpoint_files = [
    BASE_DIR / "best_fold0.pt",
    BASE_DIR / "image_only_dev" / "best_fold0.pt",
]

for path in checkpoint_files:
    print(f"\n{'=' * 60}")
    print("Checkpoint:", path)

    if not path.exists():
        print("Not found")
        continue

    try:
        try:
            checkpoint = torch.load(
                path, map_location="cpu", weights_only=True
            )
        except TypeError:
            checkpoint = torch.load(path, map_location="cpu")

        if isinstance(checkpoint, dict):
            print("Top-level keys:", list(checkpoint.keys()))

            for key, value in checkpoint.items():
                if isinstance(value, torch.Tensor):
                    print(key, "tensor shape:", tuple(value.shape))
                elif isinstance(value, dict):
                    print(key, "dictionary entries:", len(value))
        else:
            print("Checkpoint object type:", type(checkpoint))

    except Exception as e:
        print("Could not inspect checkpoint:", repr(e))


# 3. Inspect the structure of one saved CT patch
patch_dir = BASE_DIR / "patches"
patches = sorted(patch_dir.glob("*.npz")) if patch_dir.exists() else []

if patches:
    sample_path = patches[0]
    print(f"\n{'=' * 60}")
    print("Sample patch:", sample_path.name)

    try:
        with np.load(sample_path, allow_pickle=False) as data:
            for key in data.files:
                arr = data[key]
                print(key, "shape:", arr.shape, "dtype:", arr.dtype)
    except Exception as e:
        print("Could not inspect patch:", repr(e))
else:
    print("\nNo .npz patches found in", patch_dir)

### Cohort Rebuilding and Target Selection
Since our Drive working directory is missing the preprocessed data, we will query the local `pylidc` database, compile the list of nodules that meet our criteria (at least 3 annotators, diameter $\ge$ 3mm, and clear benign/malignant consensus), and cache this list as `eligible_nodules.csv` in our Drive folder.

In [37]:
import os
import pandas as pd
import pylidc as pl
import numpy as np

# Ensure the pylidc configuration file points to the raw directory
raw_dir = '/content/lidc_raw'
os.makedirs(raw_dir, exist_ok=True)
pylidcrc_path = os.path.expanduser('~/.pylidcrc')
with open(pylidcrc_path, 'w') as f:
    f.write(f'[dicom]\npath = {raw_dir}\nwarn = False\n')

# Ensure default configuration values exist
min_annotators = CFG.get('MIN_ANNOTATORS', 3) if 'CFG' in globals() else 3
min_diam = CFG.get('MIN_DIAM_MM', 3.0) if 'CFG' in globals() else 3.0
benign_max = CFG.get('BENIGN_MAX', 2.0) if 'CFG' in globals() else 2.0
malignant_min = CFG.get('MALIGNANT_MIN', 4.0) if 'CFG' in globals() else 4.0

# Ensure target directory exists
work_dir = '/content/drive/MyDrive/lidc_work'
os.makedirs(work_dir, exist_ok=True)
cache_file = os.path.join(work_dir, 'eligible_nodules.csv')

print("Querying pylidc database to build the cohort...")
pl._session.rollback()

rows = []
scans = pl.query(pl.Scan).all()

for scan in scans:
    try:
        clusters = scan.cluster_annotations(verbose=False)
    except Exception:
        continue
    for ni, nod in enumerate(clusters):
        if len(nod) < min_annotators:
            continue
        diam = float(np.mean([a.diameter for a in nod]))
        if diam < min_diam:
            continue
        m = float(np.mean([a.malignancy for a in nod]))
        if m <= benign_max:
            lab = 0
        elif m >= malignant_min:
            lab = 1
        else:
            continue # Indeterminate cases (score 3) are excluded

        rows.append(dict(
            patient=scan.patient_id,
            series=scan.series_instance_uid,
            nod=ni,
            label=lab,
            score=m,
            diam=diam,
            n_ann=len(nod)
        ))

if len(rows) > 0:
    ELIG = pd.DataFrame(rows)
else:
    # Fallback to avoid empty DataFrame errors
    ELIG = pd.DataFrame(columns=['patient', 'series', 'nod', 'label', 'score', 'diam', 'n_ann'])

ELIG.to_csv(cache_file, index=False)

print(f"Cohort compiled successfully! Saved {len(ELIG)} eligible nodules across {ELIG['patient'].nunique() if len(ELIG) > 0 else 0} patients to {cache_file}.")
if len(ELIG) > 0:
    display(ELIG.head())
else:
    print("Warning: Still found 0 nodules. Checking pylidc local database content...")
    print(f"Total scans in pylidc database: {pl.query(pl.Scan).count()}")

Querying pylidc database to build the cohort...
Cohort compiled successfully! Saved 0 eligible nodules across 0 patients to /content/drive/MyDrive/lidc_work/eligible_nodules.csv.
Total scans in pylidc database: 1018


### On-Demand DICOM Download and Patch Preprocessing
Now we will set up the pipeline to download the scans matching our cohort from the Google Cloud IDC (Imaging Data Commons) public bucket using `idc-index`, extract $64^3$ voxel CT patches centered on each target nodule, calculate their radiomics features, and save them as compressed `.npz` files to your Drive.

In [38]:
from idc_index import IDCClient
import shutil
import time
import os
import numpy as np
import pylidc as pl

# Configure IDC downloader
idc_client = IDCClient()
raw_download_dir = '/content/lidc_raw'
work_dir = '/content/drive/MyDrive/lidc_work'
mode = CFG.get('MODE', 'test') if 'CFG' in globals() else 'test'
patch_dir = os.path.join(work_dir, 'patches_test' if mode == 'test' else 'patches')
os.makedirs(patch_dir, exist_ok=True)

# Define configuration fallback limits for HU
HU_MIN, HU_MAX = -1000, 400

if len(ELIG) == 0:
    print("Warning: The cohort DataFrame (ELIG) is empty. Please check the pylidc dataset and configurations.")
else:
    # For testing, we process a small subset (e.g., 5 patients) first
    test_patients = ELIG.groupby('patient').filter(lambda x: len(x) > 0)['patient'].unique()[:5]
    jobs_subset = ELIG[ELIG['patient'].isin(test_patients)]

    print(f"Processing patches for {len(jobs_subset)} nodules across {jobs_subset.patient.nunique()} patients...")

    t0 = time.time()
    saved_count = 0

    for series_uid, group in jobs_subset.groupby('series'):
        patient_id = group.iloc[0]['patient']
        npz_paths = [os.path.join(patch_dir, f"{r.patient}_{r.series[-8:]}_n{r.nod:02d}.npz") for r in group.itertuples()]

        # Check if patches already exist to make this resumeable
        if all(os.path.exists(p) for p in npz_paths):
            continue

        print(f"\nDownloading series {series_uid[-8:]} for patient {patient_id}...")
        try:
            idc_client.download_from_selection(
                downloadDir=raw_download_dir,
                seriesInstanceUID=series_uid,
                quiet=True
            )

            # Retrieve pylidc scan object
            scan = pl.query(pl.Scan).filter(pl.Scan.series_instance_uid == series_uid).one()
            vol = scan.to_volume(verbose=False).transpose(2, 0, 1).astype(np.float32)
            spacing = np.array([scan.slice_spacing, scan.pixel_spacing, scan.pixel_spacing])
            nodule_clusters = scan.cluster_annotations(verbose=False)

            for r, out_path in zip(group.itertuples(), npz_paths):
                if os.path.exists(out_path):
                    continue

                nodule = nodule_clusters[r.nod]
                consensus_mask, consensus_bbox = consensus(nodule, clevel=0.5, ret_masks=False)

                # Create empty full volume mask and assign consensus region
                full_mask = np.zeros((vol.shape[1], vol.shape[2], vol.shape[0]), bool)
                full_mask[consensus_bbox] = consensus_mask
                full_mask = full_mask.transpose(2, 0, 1)

                # Compute patch parameters
                centroid = np.mean([ann.centroid for ann in nodule], axis=0)[[2, 0, 1]]
                hu, mask = make_patch(vol, full_mask, centroid, spacing)

                if mask.sum() == 0:
                    continue

                # Extract radiomics vector and compress
                rad_features = radiomics28(hu, mask)
                np.savez_compressed(
                    out_path,
                    p64=((hu - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float16),
                    mask=mask,
                    rad=rad_features,
                    label=r.label,
                    score=r.score,
                    patient=r.patient,
                    diam=r.diam,
                    n_ann=r.n_ann
                )
                saved_count += 1

        except Exception as e:
            print(f"Failed to process series {series_uid[-8:]}: {repr(e)}")
        finally:
            # Clean up downloaded raw DICOM slice folder immediately to preserve disk space
            shutil.rmtree(os.path.join(raw_download_dir, patient_id), ignore_errors=True)

    print(f"\nPreprocessing complete! Saved {saved_count} patches in {(time.time() - t0)/60:.1f} minutes.")

In [39]:
import os
from pathlib import Path

work_dir = Path('/content/drive/MyDrive/lidc_work')
print(f"Checking contents of: {work_dir}")

if not work_dir.exists():
    print("The directory does not exist. Please double check if Google Drive is mounted correctly.")
else:
    files_and_dirs = sorted(work_dir.rglob('*'))
    if not files_and_dirs:
        print("The directory is completely empty.")
    for path in files_and_dirs:
        # Display relative path and file size if it is a file
        rel_path = path.relative_to(work_dir)
        if path.is_file():
            size_kb = path.stat().st_size / 1024
            print(f"📄 [File] {rel_path} ({size_kb:.2f} KB)")
        elif path.is_dir():
            print(f"📁 [Dir]  {rel_path}")

Checking contents of: /content/drive/MyDrive/lidc_work
📄 [File] eligible_nodules.csv (0.04 KB)
📁 [Dir]  patches


In [40]:
!pip install -q idc-index pylidc scikit-learn "setuptools<81" SimpleITK scikit-image statsmodels

import configparser
if not hasattr(configparser, "SafeConfigParser"):          # removed in Python 3.12, pylidc still wants it
    configparser.SafeConfigParser = configparser.ConfigParser

import os, glob, shutil, time, copy, json, hashlib, pickle, platform, sys, warnings, random, itertools, zipfile, datetime
import numpy as np, pandas as pd
np.int = int                                                # pylidc still uses np.int
import pylidc as pl
from pylidc.utils import consensus
from scipy import ndimage
from scipy.stats import skew, kurtosis, rankdata, norm
from scipy.spatial.distance import pdist
from scipy.spatial import ConvexHull
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, f1_score, matthews_corrcoef, brier_score_loss
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

TASK_NAME = 'radiologist-assessed malignancy-risk classification'

CFG = dict(
    MODE='test',                     # 'test' = pipeline check | 'full' = everything
    N_TEST_PATIENTS=40,
    SEEDS=[42, 43, 44],              # CNN training seeds (first = primary seed)
    SPLIT_SEED=42,                   # fixes holdout + CV folds for ALL runs
    WORK_DIR='/content/drive/MyDrive/lidc_work',
    RAW_DIR='/content/lidc_raw',
    EXTERNAL_DIR='/content/drive/MyDrive/external_cohorts',   # put LNDb / LUNGx here (see section 12)
    MIN_ANNOTATORS=3, MIN_DIAM_MM=3.0, BENIGN_MAX=2.0, MALIGNANT_MIN=4.0,
    FEAT_DIM=128, FC=(256, 128), DROPOUT=(0.4, 0.3),
    EPOCHS=None, BATCH=16, LR=1e-3, WD=1e-4, PATIENCE=15, N_SPLITS=None, WORKERS=2,
    N_BOOT=2000,
    HOLDOUT_FRAC=0.20,
    RUN_PREPROCESS=True,
    RUN_ABLATION=True,               # section 5 (scales / CBAM / radiomics)
    RUN_SHUFFLE=True,                # label-shuffle control
    RUN_FULL_COHORT_CV=False,        # optional: paper-style 5-fold CV on all 443 nodules (internal, overlaps holdout)
    EXPL_MAX_NODULES=None,           # None = all holdout nodules
    N_REVIEW=30,                     # nodules in the radiologist review pack
)
if CFG['MODE'] == 'test':
    CFG.update(EPOCHS=3 if CFG['EPOCHS'] is None else CFG['EPOCHS'], N_SPLITS=3, SEEDS=CFG['SEEDS'][:1], N_BOOT=300, EXPL_MAX_NODULES=6, N_REVIEW=4)
else:
    CFG.update(EPOCHS=100 if CFG['EPOCHS'] is None else CFG['EPOCHS'], N_SPLITS=5)
PS = CFG['SEEDS'][0]                 # primary training seed

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    CFG['WORK_DIR'] = './lidc_work'; CFG['EXTERNAL_DIR'] = './external_cohorts'

SUB = 'full' if CFG['MODE'] == 'full' else 'test'
ROOT = os.path.join(CFG['WORK_DIR'], SUB)
PATCH_DIR = os.path.join(CFG['WORK_DIR'], 'patches' if CFG['MODE'] == 'full' else 'patches_test')
PRED_DIR, MODEL_DIR, STAT_DIR, FIG_DIR, EXPL_DIR, FROZEN_DIR, EXT_DIR, REPRO_DIR = [os.path.join(ROOT, n) for n in
    ('preds', 'models', 'stats', 'figures', 'explain', 'frozen_model', 'external_results', 'repro_package')]
for d in (CFG['WORK_DIR'], ROOT, CFG['RAW_DIR'], PATCH_DIR, PRED_DIR, MODEL_DIR, STAT_DIR, FIG_DIR, EXPL_DIR, FROZEN_DIR, EXT_DIR, REPRO_DIR):
    os.makedirs(d, exist_ok=True)
with open(os.path.expanduser('~/.pylidcrc'), 'w') as f:
    f.write(f"[dicom]\npath = {CFG['RAW_DIR']}\nwarn = False\n")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(PS)
print('Device:', DEVICE, '| mode:', CFG['MODE'], '| seeds:', CFG['SEEDS'], '| folds:', CFG['N_SPLITS'], '| epochs:', CFG['EPOCHS'])
print('Outputs ->', ROOT)

Device: cpu | mode: test | seeds: [42] | folds: 3 | epochs: 3
Outputs -> ./lidc_work/test


In [41]:
from pathlib import Path
import pandas as pd

MYDRIVE = Path("/content/drive/MyDrive")

# Check the two likely locations first.
paths = [
    MYDRIVE / "lidc_work" / "eligible_nodules.csv",
    MYDRIVE / "lidc_work_v2" / "eligible_nodules.csv",
]

found = [p for p in paths if p.is_file()]

# Search MyDrive if neither expected location contains the file.
if not found:
    found = list(MYDRIVE.rglob("eligible_nodules.csv"))

if not found:
    print("CSV not found in Google Drive.")
else:
    for p in found:
        print("Found:", p)
        print("Rows:", len(pd.read_csv(p)))

Found: /content/drive/MyDrive/lidc_work/eligible_nodules.csv
Rows: 0


In [42]:
from pathlib import Path
import os
import pandas as pd
import pylidc as pl
import numpy as np

CACHE = Path("/content/drive/MyDrive/lidc_work/eligible_nodules.csv")
CFG["WORK_DIR"] = str(CACHE.parent)
os.makedirs(CFG["WORK_DIR"], exist_ok=True)

# Helper functions to build the cohort if the cache is missing
def build_eligible_nodules():
    print("Cache not found. Querying pylidc database to build cohort (this may take a few minutes)... ")
    rows = []
    for scan in pl.query(pl.Scan).all():
        try:
            clusters = scan.cluster_annotations(verbose=False)
        except Exception:
            continue
        for ni, nod in enumerate(clusters):
            if len(nod) < CFG['MIN_ANNOTATORS']: continue
            diam = float(np.mean([a.diameter for a in nod]))
            if diam < CFG['MIN_DIAM_MM']: continue
            m = float(np.mean([a.malignancy for a in nod]))
            if m <= CFG['BENIGN_MAX']: lab = 0
            elif m >= CFG['MALIGNANT_MIN']: lab = 1
            else: continue
            rows.append(dict(patient=scan.patient_id, series=scan.series_instance_uid, nod=ni, label=lab, score=m, diam=diam, n_ann=len(nod)))
    df = pd.DataFrame(rows)
    df.to_csv(CACHE, index=False)
    print("Cohort compiled and cached successfully.")
    return df

def pick_patients(df, n, seed):
    rng = np.random.RandomState(seed)
    pat = df.groupby('patient').label.max()
    mal, ben = pat[pat == 1].index.values, pat[pat == 0].index.values
    k = n // 2
    chosen = list(rng.choice(mal, min(k, len(mal)), replace=False)) + list(rng.choice(ben, min(n - k, len(ben)), replace=False))
    return df[df.patient.isin(chosen)]

# Main driver: Load from cache or build from scratch
if CACHE.exists():
    print("Loading cached eligible nodules...")
    ELIG = pd.read_csv(CACHE)
else:
    ELIG = build_eligible_nodules()

JOBS = (
    pick_patients(ELIG, CFG["N_TEST_PATIENTS"], CFG["SPLIT_SEED"])
    if CFG["MODE"] == "test"
    else ELIG
)

print("Using cache:", CACHE)
print(f"Eligible: {len(ELIG)} nodules / {ELIG.patient.nunique()} patients")
print(f"This run: {len(JOBS)} nodules / {JOBS.patient.nunique()} patients")

Loading cached eligible nodules...
Using cache: /content/drive/MyDrive/lidc_work/eligible_nodules.csv
Eligible: 0 nodules / 0 patients
This run: 0 nodules / 0 patients


## 1. Cohort definition (offline, uses the annotation DB bundled in `pylidc`)

In [43]:
import os
from pathlib import Path

print("Configured WORK_DIR:", CFG.get("WORK_DIR"))

work_dir = Path(CFG["WORK_DIR"])
cache = work_dir / "eligible_nodules.csv"

print("Work directory exists:", work_dir.exists())
print("Eligible-nodule cache exists:", cache.exists())

if cache.exists():
    import pandas as pd
    df = pd.read_csv(cache)
    print("Cached rows:", len(df))
    print("Cached patients:", df["patient"].nunique())
    print("Columns:", df.columns.tolist())

Configured WORK_DIR: /content/drive/MyDrive/lidc_work
Work directory exists: True
Eligible-nodule cache exists: True
Cached rows: 0
Cached patients: 0
Columns: ['patient', 'series', 'nod', 'label', 'score', 'diam', 'n_ann']


In [44]:
def eligible_nodules():
    cache = os.path.join(CFG['WORK_DIR'], 'eligible_nodules.csv')
    if os.path.exists(cache): return pd.read_csv(cache)
    rows = []
    for scan in pl.query(pl.Scan).all():
        try: clusters = scan.cluster_annotations(verbose=False)
        except Exception as e:
            print('cluster failed:', scan.patient_id, repr(e)[:80]); continue
        for ni, nod in enumerate(clusters):
            if len(nod) < CFG['MIN_ANNOTATORS']: continue
            diam = float(np.mean([a.diameter for a in nod]))
            if diam < CFG['MIN_DIAM_MM']: continue
            m = float(np.mean([a.malignancy for a in nod]))
            if m <= CFG['BENIGN_MAX']: lab = 0
            elif m >= CFG['MALIGNANT_MIN']: lab = 1
            else: continue                                   # indeterminate -> excluded (documented limitation)
            rows.append(dict(patient=scan.patient_id, series=scan.series_instance_uid, nod=ni, label=lab, score=m, diam=diam, n_ann=len(nod)))
    df = pd.DataFrame(rows); df.to_csv(cache, index=False); return df

def pick_patients(df, n, seed):
    rng = np.random.RandomState(seed); pat = df.groupby('patient').label.max()
    mal, ben = pat[pat == 1].index.values, pat[pat == 0].index.values; k = n // 2
    chosen = list(rng.choice(mal, min(k, len(mal)), replace=False)) + list(rng.choice(ben, min(n - k, len(ben)), replace=False))
    return df[df.patient.isin(chosen)]

ELIG = eligible_nodules()

# Use ALL eligible nodules instead of the test subset
JOBS = ELIG.copy()

print(f"Eligible nodules: {len(ELIG)}")
print(f"Patients: {ELIG.patient.nunique()}")
print(f"Nodules to process: {len(JOBS)}")
print(f"Scans to process: {JOBS.series.nunique()}")

Eligible nodules: 0
Patients: 0
Nodules to process: 0
Scans to process: 0


## 2. Preprocessing -> small `.npz` patches (resumable; raw DICOMs deleted after each scan)

In [45]:
import os
import pandas as pd

ROOT = os.path.join(CFG["WORK_DIR"], "full_rebuild")
os.makedirs(ROOT, exist_ok=True)

# Fallback to load the split dataframes if they are not already defined in memory
if 'DF_DEV' not in globals() or 'DF_HOLD' not in globals():
    holdout_file = os.path.join(ROOT, "final_holdout_patients.csv")
    if 'DF' in globals() and os.path.exists(holdout_file):
        holdout_patients = set(pd.read_csv(holdout_file).patient.astype(str))
        DF_HOLD = DF[DF.patient.isin(holdout_patients)].copy().reset_index(drop=True)
        DF_DEV = DF[~DF.patient.isin(holdout_patients)].copy().reset_index(drop=True)
    else:
        DF_DEV, DF_HOLD = [], []

print("Training output directory:", ROOT)
print("Development nodules:", len(DF_DEV) if hasattr(DF_DEV, '__len__') else 0)
print("Holdout nodules:", len(DF_HOLD) if hasattr(DF_HOLD, '__len__') else 0)

Training output directory: /content/drive/MyDrive/lidc_work/full_rebuild
Development nodules: 0
Holdout nodules: 0


In [46]:
import os

ROOT = os.path.join(CFG["WORK_DIR"], "full_rebuild")
os.makedirs(ROOT, exist_ok=True)

HOLDOUT_FILE = os.path.join(
    ROOT, "final_holdout_patients.csv"
)

print("ROOT:", ROOT)
print("HOLDOUT_FILE:", HOLDOUT_FILE)
print("Holdout file exists:", os.path.exists(HOLDOUT_FILE))

ROOT: /content/drive/MyDrive/lidc_work/full_rebuild
HOLDOUT_FILE: /content/drive/MyDrive/lidc_work/full_rebuild/final_holdout_patients.csv
Holdout file exists: False


In [47]:
#rollback
import pylidc as pl
import traceback

# Recover the shared pylidc database session.
pl._session.rollback()
print("Session rollback completed.")

# Check whether the database can be queried again.
try:
    count = pl.query(pl.Scan).count()
    print("Database query successful. Scan count:", count)

    scan = pl.query(pl.Scan).first()
    print("Test scan:", scan.patient_id if scan else "No scans found")

    if scan is not None:
        clusters = scan.cluster_annotations(verbose=False)
        print("Clustering successful. Clusters:", len(clusters))

except Exception:
    pl._session.rollback()
    traceback.print_exc()

Session rollback completed.
Database query successful. Scan count: 1018
Test scan: LIDC-IDRI-0078
Clustering successful. Clusters: 4


In [48]:
def process_scan(series, rows):
    todo = [r for r in rows.itertuples() if not os.path.exists(npz_path(r))]
    if not todo: return 0
    scan = pl.query(pl.Scan).filter(pl.Scan.series_instance_uid == series).one()
    IDC.download_from_selection(downloadDir=CFG['RAW_DIR'], seriesInstanceUID=series, dirTemplate='%PatientID/%SeriesInstanceUID',
                                quiet=True, show_progress_bar=False)
    try:
        vol = scan.to_volume(verbose=False).transpose(2, 0, 1).astype(np.float32)          # (i,j,k) -> (z,y,x)
        sp = np.array([scan.slice_spacing, scan.pixel_spacing, scan.pixel_spacing])
        nods, saved = scan.cluster_annotations(verbose=False), 0
        for r in todo:
            nod = nods[r.nod]
            cmask, cbbox = consensus(nod, clevel=0.5, ret_masks=False)
            full = np.zeros((vol.shape[1], vol.shape[2], vol.shape[0]), bool); full[cbbox] = cmask; full = full.transpose(2, 0, 1)
            centre = np.mean([a.centroid for a in nod], axis=0)[[2, 0, 1]]
            hu, m = make_patch(vol, full, centre, sp)
            if m.sum() == 0: continue
            np.savez_compressed(npz_path(r), p64=((hu - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float16), mask=m, rad=radiomics28(hu, m),
                                label=r.label, score=r.score, patient=r.patient, diam=r.diam, n_ann=r.n_ann)
            saved += 1
        return saved
    finally:
        shutil.rmtree(os.path.join(CFG['RAW_DIR'], scan.patient_id), ignore_errors=True)

In [49]:
HU_MIN, HU_MAX, PATCH = -1000, 400, 64

def crop(v, c, half, fill):
    lo = np.round(c).astype(int) - half
    out = np.full(2 * half, fill, v.dtype)
    s, e = np.maximum(lo, 0), np.minimum(lo + 2 * half, v.shape)
    if (e > s).all():
        out[tuple(slice(a, b) for a, b in zip(s - lo, e - lo))] = v[tuple(slice(a, b) for a, b in zip(s, e))]
    return out

def fit(a, n, fill):
    return crop(a, np.array(a.shape) / 2, np.full(3, n // 2), fill) if a.shape != (n, n, n) else a

def make_patch(vol, mask, centre, sp):
    half = np.ceil((PATCH / 2 + 4) / np.asarray(sp)).astype(int)
    v = crop(vol, centre, half, HU_MIN).astype(np.float32); m = crop(mask.astype(np.uint8), centre, half, 0)
    z = np.asarray(sp, float) / 1.0
    v = ndimage.zoom(v, z, order=1); m = ndimage.zoom(m, z, order=0)
    v = fit(np.clip(v, HU_MIN, HU_MAX), PATCH, HU_MIN); m = fit(m, PATCH, 0).astype(bool)
    return v, m

RAD_NAMES = ['mean','median','std','var','min','max','range','p10','p90','entropy',
             'volume','surface','sphericity','compactness','elongation','flatness','max_diam','sv_ratio',
             'skew','kurtosis','energy','uniformity','rms','mad','grad_mean','grad_std','autocorr','cov']
RAD_GROUPS = {'first-order': list(range(0, 10)), 'shape': list(range(10, 18)), 'intensity/texture': list(range(18, 28)),
              'size-related (volume, surface, max_diam)': [RAD_NAMES.index(n) for n in ('volume', 'surface', 'max_diam')]}

def radiomics28(x, m):
    v = x[m].astype(np.float64); n = len(v)
    h, _ = np.histogram(v, bins=64); p = h[h > 0] / h.sum(); ent = -(p * np.log2(p)).sum()
    vol = float(n); surf = float(n - ndimage.binary_erosion(m).sum()) + 1e-8
    co = np.argwhere(m).astype(float); co -= co.mean(0)
    ev = np.clip(np.sort(np.linalg.eigvalsh(np.cov(co.T) + 1e-6 * np.eye(3)))[::-1], 1e-8, None)
    sub = co[np.random.RandomState(0).choice(n, min(n, 500), replace=False)]
    g = np.sqrt(sum(a ** 2 for a in np.gradient(x.astype(np.float64))))[m]
    mu, sd = v.mean(), v.std()
    f = [mu, np.median(v), sd, v.var(), v.min(), v.max(), np.ptp(v), *np.percentile(v, [10, 90]), ent,
         vol, surf, np.pi ** (1 / 3) * (6 * vol) ** (2 / 3) / surf, vol / surf ** 1.5, np.sqrt(ev[1] / ev[0]), np.sqrt(ev[2] / ev[0]),
         pdist(sub).max(), surf / vol,
         skew(v), kurtosis(v), (v ** 2).sum(), (p ** 2).sum(), np.sqrt((v ** 2).mean()), np.abs(v - mu).mean(),
         g.mean(), g.std(), np.corrcoef(v[:-1], v[1:])[0, 1], sd / (abs(mu) + 1e-8)]
    return np.nan_to_num(np.array(f, np.float32))

def npz_path(r): return os.path.join(PATCH_DIR, f"{r.patient}_{r.series[-8:]}_n{r.nod:02d}.npz")

def process_scan(series, rows):
    todo = [r for r in rows.itertuples() if not os.path.exists(npz_path(r))]
    if not todo: return 0
    scan = pl.query(pl.Scan).filter(pl.Scan.series_instance_uid == series).one()
    IDC.download_from_selection(downloadDir=CFG['RAW_DIR'], seriesInstanceUID=series, dirTemplate='%PatientID/%SeriesInstanceUID',
                                quiet=True, show_progress_bar=False)
    try:
        vol = scan.to_volume(verbose=False).transpose(2, 0, 1).astype(np.float32)          # (i,j,k) -> (z,y,x)
        sp = np.array([scan.slice_spacing, scan.pixel_spacing, scan.pixel_spacing])
        nods, saved = scan.cluster_annotations(verbose=False), 0
        for r in todo:
            nod = nods[r.nod]
            cmask, cbbox = consensus(nod, clevel=0.5, ret_masks=False)
            full = np.zeros((vol.shape[1], vol.shape[2], vol.shape[0]), bool); full[cbbox] = cmask; full = full.transpose(2, 0, 1)
            centre = np.mean([a.centroid for a in nod], axis=0)[[2, 0, 1]]
            hu, m = make_patch(vol, full, centre, sp)
            if m.sum() == 0: continue
            np.savez_compressed(npz_path(r), p64=((hu - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float16), mask=m, rad=radiomics28(hu, m),
                                label=r.label, score=r.score, patient=r.patient, diam=r.diam, n_ann=r.n_ann)
            saved += 1
        return saved
    finally:
        shutil.rmtree(os.path.join(CFG['RAW_DIR'], scan.patient_id), ignore_errors=True)

if CFG["RUN_PREPROCESS"]:
    from idc_index import IDCClient
    import traceback

    IDC = IDCClient()

    t0 = time.time()
    failed = []
    total = 0

    # Reset any failed transaction left by the interrupted run.
    try:
        pl._session.rollback()
        print("Database session rollback completed.")
    except Exception as e:
        print("Initial rollback warning:", repr(e))

    groups = list(JOBS.groupby("series"))

    for i, (series, rows) in enumerate(groups, 1):
        try:
            total += process_scan(series, rows)

        except Exception as e:
            # Record the failed scan.
            failed.append({
                "patient": rows.patient.iloc[0],
                "series": series,
                "error": repr(e)
            })

            # Print the full traceback for the first failure.
            if len(failed) == 1:
                print("\nFirst preprocessing failure:")
                traceback.print_exc()

            # Recover the SQLAlchemy session before the next scan.
            try:
                pl._session.rollback()
            except Exception as rollback_error:
                print("Rollback failed:", repr(rollback_error))

        if i % 5 == 0 or i == len(groups):
            print(
                f"[{i}/{len(groups)}] "
                f"saved {total} | "
                f"failed scans {len(failed)} | "
                f"{(time.time() - t0) / 60:.1f} min"
            )

    # Save the failure log so it remains available after a runtime reset.
    if failed:
        failure_path = os.path.join(
            CFG["WORK_DIR"],
            "preprocessing_failures.csv"
        )

        pd.DataFrame(failed).to_csv(failure_path, index=False)
        print("\nFailure log saved:", failure_path)

    print(
        "\nPatch files on disk:",
        len(glob.glob(os.path.join(PATCH_DIR, "*.npz")))
    )
    print("Total scans failed:", len(failed))

Database session rollback completed.

Patch files on disk: 0
Total scans failed: 0


In [50]:
import os

# Keep the existing manifest and old patches safe.
# Create a separate directory for regenerated patches.
PATCH_DIR = os.path.join(
    CFG["WORK_DIR"],
    "patches_full_rebuild"
)
os.makedirs(PATCH_DIR, exist_ok=True)

# Use the full eligible-nodule list, not the test subset.
JOBS = ELIG.copy()

def npz_path(r):
    return os.path.join(
        PATCH_DIR,
        f"{r.patient}_{str(r.series)[-8:]}_n{int(r.nod):02d}.npz"
    )

print("Patch output directory:", PATCH_DIR)
print("Eligible nodules:", len(ELIG))
print("Patients:", ELIG["patient"].nunique())
print("Nodules queued:", len(JOBS))
print("Scans queued:", JOBS["series"].nunique())
print("Test subset active for processing:", len(JOBS) != len(ELIG))

Patch output directory: /content/drive/MyDrive/lidc_work/patches_full_rebuild
Eligible nodules: 0
Patients: 0
Nodules queued: 0
Scans queued: 0
Test subset active for processing: False


In [51]:
# Sequential Preprocessor Driver
import traceback
import time
import pylidc as pl

print("=== Sequential Preprocessor Driver ===")
if 'JOBS' not in globals() or len(JOBS) == 0:
    print("Warning: JOBS dataframe is empty or undefined. Please rebuild the cohort first.")
else:
    # Ensure the pylidc database transaction is clean
    pl._session.rollback()

    t0 = time.time()
    failed = []
    total_processed = 0
    groups = list(JOBS.groupby('series'))
    print(f"Found {len(groups)} series groups to process in JOBS.\n")

    # Process each series sequentially using the correct loop variables
    for i, (series_uid, rows_group) in enumerate(groups, 1):
        try:
            print(f"[{i}/{len(groups)}] Processing Series: {series_uid[-8:]} (Patient: {rows_group.patient.iloc[0]})", end=' ')
            saved_count = process_scan(series_uid, rows_group)
            total_processed += saved_count
            print(f"-> saved {saved_count} patches.")
        except Exception as e:
            failed.append({
                'series': series_uid,
                'patient': rows_group.patient.iloc[0],
                'error': repr(e)
            })
            print(f"\nError processing series {series_uid[-8:]}: {repr(e)}")
            pl._session.rollback()

    print(f"\nFinished! Successfully processed and saved {total_processed} patches.")
    print(f"Failed series: {len(failed)}")

NameError: name 'series' is not defined

In [52]:
import time
import traceback

# Ensure the pylidc database transaction is clean
pl._session.rollback()

# Initialize counters and failure tracking
t0 = time.time()
failed = []
total_processed = 0

groups = list(JOBS.groupby('series'))
print(f"Found {len(groups)} series groups to process.")

# Iterate through each group to define 'series' and 'rows' dynamically
for i, (series_uid, rows_group) in enumerate(groups, 1):
    try:
        print(f"\nProcessing [{i}/{len(groups)}] Series: {series_uid[-8:]} (Patient: {rows_group.patient.iloc[0]})")
        saved_count = process_scan(series_uid, rows_group)
        total_processed += saved_count
    except Exception as e:
        failed.append({
            'series': series_uid,
            'patient': rows_group.patient.iloc[0],
            'error': repr(e)
        })
        print(f"Error processing series {series_uid[-8:]}: {repr(e)}")
        pl._session.rollback()

print(f"\nFinished! Successfully processed and saved {total_processed} patches.")
print(f"Failed series: {len(failed)}")

Found 0 series groups to process.

Finished! Successfully processed and saved 0 patches.
Failed series: 0


In [53]:
import os
import pandas as pd
import pylidc as pl
import numpy as np

# Force a complete rebuild of the eligible nodules cohort from local database
cache_file = os.path.join(CFG['WORK_DIR'], 'eligible_nodules.csv')
print(f"Rebuilding cohort to: {cache_file}")

pl._session.rollback()
rows = []
scans = pl.query(pl.Scan).all()

for scan in scans:
    try:
        clusters = scan.cluster_annotations(verbose=False)
    except Exception:
        continue
    for ni, nod in enumerate(clusters):
        if len(nod) < CFG['MIN_ANNOTATORS']:
            continue
        diam = float(np.mean([a.diameter for a in nod]))
        if diam < CFG['MIN_DIAM_MM']:
            continue
        m = float(np.mean([a.malignancy for a in nod]))
        if m <= CFG['BENIGN_MAX']:
            lab = 0
        elif m >= CFG['MALIGNANT_MIN']:
            lab = 1
        else:
            continue # Indeterminate are excluded

        rows.append(dict(
            patient=scan.patient_id,
            series=scan.series_instance_uid,
            nod=ni,
            label=lab,
            score=m,
            diam=diam,
            n_ann=len(nod)
        ))

ELIG = pd.DataFrame(rows)
os.makedirs(os.path.dirname(cache_file), exist_ok=True)
ELIG.to_csv(cache_file, index=False)
JOBS = ELIG.copy()

print(f"Successfully built cohort! Saved {len(ELIG)} nodules across {ELIG.patient.nunique()} patients.")

Rebuilding cohort to: /content/drive/MyDrive/lidc_work/eligible_nodules.csv
Successfully built cohort! Saved 444 nodules across 331 patients.


### Cohort-wide Preprocessing and Patch Generation
Using the dynamic sequential preprocessor loop, we will process all series groups matching our active cohort `JOBS` to download raw scans, extract $64^3$ patches, calculate radiomics, and cache the resulting `.npz` files in `patches_full_rebuild`.

In [ ]:
import time
import traceback

# Ensure the pylidc database transaction is clean
pl._session.rollback()

# Initialize counters and failure tracking
t0 = time.time()
failed = []
total_processed = 0

groups = list(JOBS.groupby('series'))
print(f"Found {len(groups)} series groups to process.")

for i, (series_uid, rows_group) in enumerate(groups, 1):
    try:
        print(f"\nProcessing [{i}/{len(groups)}] Series: {series_uid[-8:]} (Patient: {rows_group.patient.iloc[0]})", end=' ')
        saved_count = process_scan(series_uid, rows_group)
        total_processed += saved_count
        print(f"-> saved {saved_count} patches.")
    except Exception as e:
        failed.append({
            'series': series_uid,
            'patient': rows_group.patient.iloc[0],
            'error': repr(e)
        })
        print(f"\nError processing series {series_uid[-8:]}: {repr(e)}")
        pl._session.rollback()

print(f"\nFinished! Successfully processed and saved {total_processed} patches.")
print(f"Failed series: {len(failed)}")
if failed:
    import pandas as pd
    pd.DataFrame(failed).to_csv(os.path.join(CFG['WORK_DIR'], 'preprocessing_failures.csv'), index=False)

Found 331 series groups to process.

Processing [1/331] Series: 70740405 (Patient: LIDC-IDRI-1002) -> saved 1 patches.

Processing [2/331] Series: 40925249 (Patient: LIDC-IDRI-0187) -> saved 1 patches.

Processing [3/331] Series: 69304872 (Patient: LIDC-IDRI-0184) -> saved 1 patches.

Processing [4/331] Series: 68101375 (Patient: LIDC-IDRI-0965) -> saved 1 patches.

Processing [5/331] Series: 26303943 (Patient: LIDC-IDRI-0890) -> saved 1 patches.

Processing [6/331] Series: 65997306 (Patient: LIDC-IDRI-0481) -> saved 3 patches.

Processing [7/331] Series: 09808059 (Patient: LIDC-IDRI-0447) -> saved 1 patches.

Processing [8/331] Series: 42738609 (Patient: LIDC-IDRI-0606) -> saved 1 patches.

Processing [9/331] Series: 70245028 (Patient: LIDC-IDRI-0305) -> saved 1 patches.

Processing [10/331] Series: 12736928 (Patient: LIDC-IDRI-0354) -> saved 1 patches.

Processing [11/331] Series: 21315800 (Patient: LIDC-IDRI-0772) -> saved 2 patches.

Processing [12/331] Series: 42211437 (Patient: L

### Section 3: Build the Analysis Table & Stratified Splits
This section compiles the newly preprocessed patches, performs an exhaustive validation of the dataset arrays, and establishes the deterministic, patient-stratified train/holdout splits based on our predefined configuration.

In [7]:
import sys
import subprocess
import os

# ---- Inline installation check for pylidc and dependencies ----
try:
    import pylidc as pl
except ModuleNotFoundError:
    print("Required dependencies not found. Installing packages...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pylidc", "scikit-learn", "SimpleITK", "scikit-image"], check=True)
    import pylidc as pl

import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt

# ---- Ensure CFG and helper definitions exist in the local session ----
if 'CFG' not in globals():
    CFG = {
        'WORK_DIR': '/content/drive/MyDrive/lidc_work',
        'MODE': 'test',
        'MIN_ANNOTATORS': 3,
        'MIN_DIAM_MM': 3.0,
        'BENIGN_MAX': 2.0,
        'MALIGNANT_MIN': 4.0,
        'SPLIT_SEED': 42,
        'HOLDOUT_FRAC': 0.20
    }

# Ensure ELIG is loaded/defined from cache or database if missing
if 'ELIG' not in globals():
    cache_path = os.path.join(CFG['WORK_DIR'], 'eligible_nodules.csv')
    if os.path.exists(cache_path):
        ELIG = pd.read_csv(cache_path)
    else:
        print('Cache not found. Rebuilding cohort from pylidc database...')
        pl._session.rollback()
        rows = []
        for scan in pl.query(pl.Scan).all():
            try:
                clusters = scan.cluster_annotations(verbose=False)
            except Exception:
                continue
            for ni, nod in enumerate(clusters):
                if len(nod) < CFG['MIN_ANNOTATORS']: continue
                diam = float(np.mean([a.diameter for a in nod]))
                if diam < CFG['MIN_DIAM_MM']: continue
                m_score = float(np.mean([a.malignancy for a in nod]))
                if m_score <= CFG['BENIGN_MAX']: lab = 0
                elif m_score >= CFG['MALIGNANT_MIN']: lab = 1
                else: continue
                rows.append(dict(patient=scan.patient_id, series=scan.series_instance_uid, nod=ni, label=lab, score=m_score, diam=diam, n_ann=len(nod)))
        ELIG = pd.DataFrame(rows)
        os.makedirs(CFG['WORK_DIR'], exist_ok=True)
        ELIG.to_csv(cache_path, index=False)
        print('Cohort compiled and cached successfully.')

# Ensure the active patch directory is targeted
PATCH_DIR = os.path.join(CFG['WORK_DIR'], 'patches_full_rebuild')

def npz_path(r):
    return os.path.join(PATCH_DIR, f"{r.patient}_{r.series[-8:]}_n{r.nod:02d}.npz")

def solidity_of(m):
    co = np.argwhere(m)
    if len(co) < 10: return np.nan
    try: return float(min(1.0, m.sum() / ConvexHull(co).volume))
    except Exception: return np.nan

def scan_meta(series_list):
    out = {}
    for s in series_list:
        try:
            sc = pl.query(pl.Scan).filter(pl.Scan.series_instance_uid == s).first()
            out[s] = dict(slice_thk=float(sc.slice_thickness), pix=float(sc.pixel_spacing), contrast=int(bool(sc.contrast_used)))
        except Exception: out[s] = dict(slice_thk=np.nan, pix=np.nan, contrast=-1)
    return out

# Compile metadata from the successfully generated patches
recs = []
for r in ELIG.itertuples():
    fp = npz_path(r)
    if not os.path.exists(fp): continue
    z = np.load(fp)
    recs.append(dict(file=fp, patient=str(r.patient), series=r.series, nod=int(r.nod), label=int(r.label), score=float(r.score), diam=float(r.diam),
                     n_ann=int(r.n_ann), rad=z['rad'].astype(np.float32), mask_vox=int(z['mask'].sum()), solidity=solidity_of(z['mask']),
                     flat=float(z['p64'].astype(np.float32).std()) < 1e-3))

DF = pd.DataFrame(recs)

if DF.empty:
    print(f"WARNING: No preprocessed patch files found in {PATCH_DIR}.")
else:
    SM = scan_meta(DF.series.unique())
    for k in ('slice_thk', 'pix', 'contrast'): DF[k] = [SM[s][k] for s in DF.series]
    print(f"{len(DF)} nodules | {DF.patient.nunique()} patients | benign {(DF.label==0).sum()} / malignant {(DF.label==1).sum()} | flat patches {int(DF.flat.sum())} (must be 0)")

    # Generate stratified patient split
    from sklearn.model_selection import StratifiedShuffleSplit
    HOLDOUT_FILE = os.path.join(CFG["WORK_DIR"], "full_rebuild", "final_holdout_patients.csv")
    os.makedirs(os.path.dirname(HOLDOUT_FILE), exist_ok=True)

    if os.path.exists(HOLDOUT_FILE):
        holdout_patients = set(pd.read_csv(HOLDOUT_FILE).patient.astype(str))
    else:
        pdf = DF.groupby('patient').label.max().reset_index()
        _, hi = next(StratifiedShuffleSplit(1, test_size=CFG['HOLDOUT_FRAC'], random_state=CFG['SPLIT_SEED']).split(pdf.patient, pdf.label))
        holdout_patients = set(pdf.iloc[hi].patient.astype(str))
        pd.DataFrame({'patient': sorted(holdout_patients)}).to_csv(HOLDOUT_FILE, index=False)

    DF_HOLD = DF[DF.patient.isin(holdout_patients)].copy().reset_index(drop=True)
    DF_DEV = DF[~DF.patient.isin(holdout_patients)].copy().reset_index(drop=True)

    assert not (set(DF_DEV.patient) & set(DF_HOLD.patient)), 'Patient leakage detected!'
    y_dev, y_hold = DF_DEV.label.values.astype(int), DF_HOLD.label.values.astype(int)
    print(f"\nDevelopment: {len(DF_DEV)} nodules / {DF_DEV.patient.nunique()} patients (malignant {y_dev.sum()})")
    print(f"Holdout    : {len(DF_HOLD)} nodules / {DF_HOLD.patient.nunique()} patients (malignant {y_hold.sum()})")

## 3. Build the analysis table, sanity check, and the patient-level development / holdout split
Extra per-nodule metadata used by the subgroup analyses (§9): approximate solidity (from the consensus mask) and scan protocol (slice thickness, contrast) taken from the `pylidc` annotation database.

In [ ]:
import os
import pandas as pd

print("Eligible nodules:", len(ELIG))
print("Unique eligible patients:", ELIG.patient.nunique())

ELIG = ELIG.copy()
ELIG["patch_exists"] = ELIG.apply(
    lambda r: os.path.exists(npz_path(r)),
    axis=1
)

print("\nPatch status:")
print(ELIG["patch_exists"].value_counts().rename({
    True: "Patch exists",
    False: "Patch missing"
}))

print("\nEligible nodules by label:")
print(ELIG.groupby("label").size())

print("\nPatch availability by label:")
print(pd.crosstab(ELIG["label"], ELIG["patch_exists"]))

print("\nEligible nodules without patches:")
print(
    ELIG.loc[
        ~ELIG["patch_exists"],
        ["patient", "series", "nod", "label", "diam", "n_ann"]
    ].head(15).to_string(index=False)
)

In [ ]:
r = next(ELIG[ELIG["patch_exists"]].itertuples())
fp = npz_path(r)

with np.load(fp) as z:
    print("Example file:", fp)
    print("Saved arrays:", z.files)

    for key in ["p64", "mask", "rad", "label", "score"]:
        if key in z:
            arr = z[key]
            print(
                f"{key}: shape={arr.shape}, "
                f"dtype={arr.dtype}, "
                f"value={arr.item() if arr.ndim == 0 else 'array'}"
            )

In [ ]:
import os
import numpy as np
import pandas as pd

# Use the main patch directory, not patches_test
PATCH_DIR = os.path.join(CFG["WORK_DIR"], "patches")

def npz_path(r):
    return os.path.join(
        PATCH_DIR,
        f"{r.patient}_{str(r.series)[-8:]}_n{int(r.nod):02d}.npz"
    )

problems = []
valid_count = 0

for r in ELIG.itertuples(index=False):
    fp = npz_path(r)

    if not os.path.isfile(fp):
        problems.append((r.patient, r.nod, "missing file"))
        continue

    try:
        with np.load(fp) as z:
            required = {"p64", "mask", "rad", "label"}
            absent = required - set(z.files)

            if absent:
                problems.append((r.patient, r.nod, f"missing arrays: {absent}"))
                continue

            p64 = z["p64"]
            mask = z["mask"]
            rad = z["rad"]

            if p64.shape != (64, 64, 64):
                problems.append((r.patient, r.nod, "incorrect p64 shape"))
            elif mask.shape != (64, 64, 64):
                problems.append((r.patient, r.nod, "incorrect mask shape"))
            elif rad.shape != (28,):
                problems.append((r.patient, r.nod, "incorrect radiomics shape"))
            elif not np.isfinite(p64).all() or not np.isfinite(rad).all():
                problems.append((r.patient, r.nod, "non-finite values"))
            elif int(z["label"].item()) != int(r.label):
                problems.append((r.patient, r.nod, "label mismatch"))
            elif int(mask.sum()) < 20:
                problems.append((r.patient, r.nod, "mask too small"))
            else:
                valid_count += 1

    except Exception as e:
        problems.append((r.patient, r.nod, f"read error: {e}"))

print("Eligible nodules:", len(ELIG))
print("Valid patches:", valid_count)
print("Problems found:", len(problems))

for issue in problems[:20]:
    print(issue)

In [ ]:
CFG["MODE"] = "full"
JOBS = ELIG.copy()

print("Eligible nodules:", len(ELIG))
print("Nodules queued:", len(JOBS))
print("Unique scans queued:", JOBS.series.nunique())
print("PATCH_DIR:", PATCH_DIR)

In [ ]:
import pandas as pd
import os
import pylidc as pl
import numpy as np

cache_file = os.path.join(CFG['WORK_DIR'], 'eligible_nodules.csv')

def build_eligible_nodules(path):
    print("Querying pylidc database to build cohort (this may take a few minutes)... ")
    pl._session.rollback()
    rows = []
    for scan in pl.query(pl.Scan).all():
        try:
            clusters = scan.cluster_annotations(verbose=False)
        except Exception:
            continue
        for ni, nod in enumerate(clusters):
            if len(nod) < CFG['MIN_ANNOTATORS']: continue
            diam = float(np.mean([a.diameter for a in nod]))
            if diam < CFG['MIN_DIAM_MM']: continue
            m = float(np.mean([a.malignancy for a in nod]))
            if m <= CFG['BENIGN_MAX']: lab = 0
            elif m >= CFG['MALIGNANT_MIN']: lab = 1
            else: continue
            rows.append(dict(patient=scan.patient_id, series=scan.series_instance_uid, nod=ni, label=lab, score=m, diam=diam, n_ann=len(nod)))
    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print("Cohort compiled and cached successfully.")
    return df

# Safely load or rebuild cache
if os.path.exists(cache_file) and os.path.getsize(cache_file) > 0:
    try:
        ELIG = pd.read_csv(cache_file)
        JOBS = ELIG.copy()
        print(f"Successfully loaded {len(ELIG)} eligible nodules from cache!")
    except pd.errors.EmptyDataError:
        print("Cache file is empty. Rebuilding...")
        ELIG = build_eligible_nodules(cache_file)
        JOBS = ELIG.copy()
else:
    ELIG = build_eligible_nodules(cache_file)
    JOBS = ELIG.copy()

In [ ]:
import sys
import subprocess

# ---- Inline installation check for pylidc and required packages ----
try:
    import pylidc as pl
except ModuleNotFoundError:
    print("pylidc not found. Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pylidc", "SimpleITK", "scikit-image", "idc-index", "setuptools<81"], check=True)
    import pylidc as pl

import os
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt

# ---- Ensure CFG is defined in case of runtime resets ----
if 'CFG' not in globals():
    CFG = {
        'WORK_DIR': '/content/drive/MyDrive/lidc_work',
        'MODE': 'test', # 'test' = pipeline check | 'full' = everything
        'MIN_ANNOTATORS': 3,
        'MIN_DIAM_MM': 3.0,
        'BENIGN_MAX': 2.0,
        'MALIGNANT_MIN': 4.0,
        'SPLIT_SEED': 42,
        'HOLDOUT_FRAC': 0.20
    }

def solidity_of(m):
    co = np.argwhere(m)
    if len(co) < 10: return np.nan
    try: return float(min(1.0, m.sum() / ConvexHull(co).volume))      # approximate (voxel-centre hull)
    except Exception: return np.nan

def scan_meta(series_list):
    out = {}
    for s in series_list:
        try:
            sc = pl.query(pl.Scan).filter(pl.Scan.series_instance_uid == s).first()
            out[s] = dict(slice_thk=float(sc.slice_thickness), pix=float(sc.pixel_spacing), contrast=int(bool(sc.contrast_used)))
        except Exception: out[s] = dict(slice_thk=np.nan, pix=np.nan, contrast=-1)
    return out

# ---- Ensure ELIG is loaded/defined from cache or database if missing ----
if 'ELIG' not in globals():
    cache_path = os.path.join(CFG['WORK_DIR'], 'eligible_nodules.csv')
    if os.path.exists(cache_path):
        print('Loading ELIG from cache...')
        ELIG = pd.read_csv(cache_path)
    else:
        print('Cache not found. Querying pylidc database to build cohort...')
        pl._session.rollback()
        rows = []
        for scan in pl.query(pl.Scan).all():
            try:
                clusters = scan.cluster_annotations(verbose=False)
            except Exception:
                continue
            for ni, nod in enumerate(clusters):
                if len(nod) < CFG['MIN_ANNOTATORS']: continue
                diam = float(np.mean([a.diameter for a in nod]))
                if diam < CFG['MIN_DIAM_MM']: continue
                m_score = float(np.mean([a.malignancy for a in nod]))
                if m_score <= CFG['BENIGN_MAX']: lab = 0
                elif m_score >= CFG['MALIGNANT_MIN']: lab = 1
                else: continue
                rows.append(dict(patient=scan.patient_id, series=scan.series_instance_uid, nod=ni, label=lab, score=m_score, diam=diam, n_ann=len(nod)))
        ELIG = pd.DataFrame(rows)
        os.makedirs(CFG['WORK_DIR'], exist_ok=True)
        ELIG.to_csv(cache_path, index=False)
        print('Cohort compiled and cached successfully.')

# Fallback definition for npz_path in case it was not defined upstream
if 'npz_path' not in globals():
    PATCH_DIR = os.path.join(CFG['WORK_DIR'], 'patches' if CFG['MODE'] == 'full' else 'patches_test')
    def npz_path(r):
        return os.path.join(PATCH_DIR, f"{r.patient}_{r.series[-8:]}_n{r.nod:02d}.npz")

recs = []
for r in ELIG.itertuples():
    fp = npz_path(r)
    if not os.path.exists(fp): continue
    z = np.load(fp)
    recs.append(dict(file=fp, patient=str(r.patient), series=r.series, nod=int(r.nod), label=int(r.label), score=float(r.score), diam=float(r.diam),
                     n_ann=int(r.n_ann), rad=z['rad'].astype(np.float32), mask_vox=int(z['mask'].sum()), solidity=solidity_of(z['mask']),
                     flat=float(z['p64'].astype(np.float32).std()) < 1e-3))

DF = pd.DataFrame(recs)

if DF.empty:
    print(f"WARNING: No preprocessed patch files found in {PATCH_DIR}.")
    print("Please ensure you have run the preprocessing stage (section 2) for the current mode before running this cell.")
else:
    SM = scan_meta(DF.series.unique())
    for k in ('slice_thk', 'pix', 'contrast'): DF[k] = [SM[s][k] for s in DF.series]
    print(f"{len(DF)} nodules | {DF.patient.nunique()} patients | benign {(DF.label==0).sum()} / malignant {(DF.label==1).sum()} | flat patches {int(DF.flat.sum())} (must be 0)")
    missing = ELIG[~ELIG.apply(lambda r: os.path.exists(npz_path(r)), axis=1)] if CFG['MODE'] == 'full' else pd.DataFrame()
    print('Eligible nodules without patch (annotation lost in resampling etc.):', len(missing))

    fig, ax = plt.subplots(2, 6, figsize=(18, 6))
    for a, f_ in zip(ax.T, DF.sample(min(6, len(DF)), random_state=0).file):
        z = np.load(f_); p, m = z['p64'].astype(np.float32), z['mask']
        for r_, sl in enumerate([p[32], p[:, 32]]):
            a[r_].imshow(sl, cmap='gray', vmin=0, vmax=1); a[r_].contour((m[32] if r_ == 0 else m[:, 32]).astype(float), [0.5], colors='r'); a[r_].axis('off')
        a[0].set_title(f"{'malignant' if z['label'] else 'benign'}  score {float(z['score']):.1f}")
    plt.suptitle('Sanity check: red outline = radiologist consensus mask must sit on the nodule'); plt.show()

    # ---- fixed patient-level development / holdout split (deterministic from SPLIT_SEED; IDs exported in section 14)
    from sklearn.model_selection import StratifiedShuffleSplit
    HOLDOUT_FILE = os.path.join(
        CFG["WORK_DIR"],
        "full_rebuild",
        "final_holdout_patients.csv"
    )
    os.makedirs(os.path.dirname(HOLDOUT_FILE), exist_ok=True)
    if os.path.exists(HOLDOUT_FILE):
        holdout_patients = set(pd.read_csv(HOLDOUT_FILE).patient.astype(str))
    else:
        pdf = DF.groupby('patient').label.max().reset_index()
        _, hi = next(StratifiedShuffleSplit(1, test_size=CFG['HOLDOUT_FRAC'], random_state=CFG['SPLIT_SEED']).split(pdf.patient, pdf.label))
        holdout_patients = set(pdf.iloc[hi].patient.astype(str))
        pd.DataFrame({'patient': sorted(holdout_patients)}).to_csv(HOLDOUT_FILE, index=False)
    DF_HOLD = DF[DF.patient.isin(holdout_patients)].copy().reset_index(drop=True)
    DF_DEV = DF[~DF.patient.isin(holdout_patients)].copy().reset_index(drop=True)
    assert not (set(DF_DEV.patient) & set(DF_HOLD.patient)), 'patient leakage between development and holdout!'
    y_dev, y_hold = DF_DEV.label.values.astype(int), DF_HOLD.label.values.astype(int)
    print(f"Development: {len(DF_DEV)} nodules / {DF_DEV.patient.nunique()} patients (malignant {y_dev.sum()})")
    print(f"Holdout    : {len(DF_HOLD)} nodules / {DF_HOLD.patient.nunique()} patients (malignant {y_hold.sum()})  <- INTERNAL evaluation, not independent validation")

## 4. Model, data pipeline and training engine
One generic engine trains *any* variant (scale subset, CBAM on/off, radiomics on/off) with any training seed on any dataframe, caches every fold to disk, and always:
* fits the radiomics scaler on the **training fold only**;
* early-stops / selects the checkpoint on the **validation** part only;
* picks the fold threshold on **validation** only;
* predicts the development out-of-fold part and the holdout in the same pass (holdout never influences anything).

In [ ]:
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy import ndimage
from sklearn.model_selection import StratifiedGroupKFold

def make_folds(df, k, seed):
    """
    Outer grouped CV:
      - one outer fold is held out for OOF testing
      - the remaining patients form the training/validation pool

    Inner grouped split:
      - approximately 80% for training
      - approximately 20% for validation

    Patients never overlap between train, validation, and test.
    """
    y = df["label"].to_numpy()
    groups = df["patient"].astype(str).to_numpy()

    outer_cv = StratifiedGroupKFold(
        n_splits=k,
        shuffle=True,
        random_state=seed
    )

    folds = []

    for f, (trainval_idx, test_idx) in enumerate(
        outer_cv.split(np.zeros(len(df)), y, groups)
    ):
        inner_df = df.iloc[trainval_idx]

        inner_cv = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=seed + f + 1
        )

        train_local, val_local = next(
            inner_cv.split(
                np.zeros(len(inner_df)),
                inner_df["label"].to_numpy(),
                inner_df["patient"].astype(str).to_numpy()
            )
        )

        train_idx = trainval_idx[train_local]
        val_idx = trainval_idx[val_local]

        train_pat = set(groups[train_idx])
        val_pat = set(groups[val_idx])
        test_pat = set(groups[test_idx])

        assert train_pat.isdisjoint(val_pat)
        assert train_pat.isdisjoint(test_pat)
        assert val_pat.isdisjoint(test_pat)

        folds.append((train_idx, val_idx, test_idx))

    # Every development nodule must appear in exactly one test fold.
    all_test_idx = np.concatenate([test for _, _, test in folds])
    assert len(all_test_idx) == len(df)
    assert len(np.unique(all_test_idx)) == len(df)

    return folds

def augment(p):                      # ONE random transform of the 64^3 patch; 48^3 / 32^3 are centre crops of the result
    r = np.random
    if r.rand() < .5: p = ndimage.rotate(p, r.uniform(-15, 15), axes=[(0, 1), (0, 2), (1, 2)][r.randint(3)], reshape=False, order=1, mode='nearest')
    for a in range(3):
        if r.rand() < .5: p = np.flip(p, a)
    if r.rand() < .5: p = ndimage.shift(p, r.randint(-5, 6, 3), order=1, mode='nearest')
    if r.rand() < .5: p = p + r.uniform(-.05, .05)
    return np.clip(p, 0, 1)

class NoduleDS(Dataset):
    def __init__(self, df, idx, rad, aug): self.f, self.y, self.rad, self.aug = df.file.values[idx], df.label.values[idx], rad, aug
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        p = np.load(self.f[i])['p64'].astype(np.float32)
        if self.aug: p = augment(p)
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a))[None]
        return t(p[16:48, 16:48, 16:48]), t(p[8:56, 8:56, 8:56]), t(p), torch.from_numpy(self.rad[i]), int(self.y[i])

class CBAM3D(nn.Module):
    def __init__(self, c, r=8):
        super().__init__(); h = max(c // r, 4)
        self.mlp = nn.Sequential(nn.Linear(c, h), nn.ReLU(True), nn.Linear(h, c)); self.conv = nn.Conv3d(2, 1, 7, padding=3)
    def forward(self, x):
        x = x * torch.sigmoid(self.mlp(x.mean((2, 3, 4))) + self.mlp(x.amax((2, 3, 4))))[:, :, None, None, None]
        return x * torch.sigmoid(self.conv(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)))

class Branch(nn.Module):
    def __init__(self, cbam, fd):
        super().__init__(); L = []
        for ci, co in [(1, 16), (16, 32), (32, 64)]: L += [nn.Conv3d(ci, co, 3, padding=1), nn.BatchNorm3d(co), nn.ReLU(True), nn.MaxPool3d(2)]
        self.conv, self.att, self.fc = nn.Sequential(*L), (CBAM3D(64) if cbam == 'branch' else nn.Identity()), nn.Linear(64, fd)
    def forward(self, x):
        self.fmap = self.att(self.conv(x)); return self.fc(self.fmap.mean((2, 3, 4)))

class Net(nn.Module):
    # scales: any non-empty subset of (32, 48, 64); radiomics: use the 28-d vector; cbam: 'branch' | 'fused' | 'none'
    def __init__(self, scales=(32, 48, 64), radiomics=True, cbam='branch', fd=None):
        super().__init__()
        # Resolve parameters dynamically, falling back to typical defaults if global CFG misses them
        config = globals().get('CFG', {})
        fd = fd or config.get('FEAT_DIM', 128)
        fc_dims = config.get('FC', (256, 128))
        drop_rates = config.get('DROPOUT', (0.4, 0.3))

        self.scales, self.use_rad = tuple(scales), radiomics
        self.b = nn.ModuleList([Branch(cbam, fd) for _ in self.scales])
        d = fd * len(self.scales) + (28 if radiomics else 0)
        self.gate = nn.Sequential(nn.Linear(d, d // 4), nn.ReLU(True), nn.Linear(d // 4, d), nn.Sigmoid()) if cbam == 'fused' else None
        h1, h2 = fc_dims
        d1, d2 = drop_rates
        self.head = nn.Sequential(nn.Linear(d, h1), nn.ReLU(True), nn.Dropout(d1), nn.Linear(h1, h2), nn.ReLU(True), nn.Dropout(d2), nn.Linear(h2, 2))

    def forward(self, p32, p48, p64, rad):
        ins = {32: p32, 48: p48, 64: p64}
        z = torch.cat([b(ins[s]) for b, s in zip(self.b, self.scales)] + ([rad] if self.use_rad else []), 1)
        return self.head(z * self.gate(z) if self.gate is not None else z)

_m = Net(); print(f"full model: {sum(p.numel() for p in _m.parameters()):,} params | fused dim = {_m.head[0].in_features} (3x128 + 28 = 412)"); del _m

In [ ]:
# ---- every model that is trained / compared in this study ----------------------------------------------------
IV, IM = RAD_NAMES.index('volume'), RAD_NAMES.index('max_diam')
VARIANTS = {
    # --- proposed model and its image / radiomics ablations
    'full':           dict(kind='cnn', scales=(32, 48, 64), radiomics=True,  cbam='branch'),   # full proposed model
    'full_no_cbam':   dict(kind='cnn', scales=(32, 48, 64), radiomics=True,  cbam='none'),     # tests attention contribution
    'img_3s_cbam':    dict(kind='cnn', scales=(32, 48, 64), radiomics=False, cbam='branch'),   # full model WITHOUT radiomics
    'img_3s_nocbam':  dict(kind='cnn', scales=(32, 48, 64), radiomics=False, cbam='none'),     # multiscale benefit without attention
    # --- scale ablation (image-only, CBAM as in the full model, so ONLY the scale set varies)
    'img_32':         dict(kind='cnn', scales=(32,),        radiomics=False, cbam='branch'),
    'img_48':         dict(kind='cnn', scales=(48,),        radiomics=False, cbam='branch'),
    'img_64':         dict(kind='cnn', scales=(64,),        radiomics=False, cbam='branch'),
    'img_32_48':      dict(kind='cnn', scales=(32, 48),     radiomics=False, cbam='branch'),
    'img_32_64':      dict(kind='cnn', scales=(32, 64),     radiomics=False, cbam='branch'),
    'img_48_64':      dict(kind='cnn', scales=(48, 64),     radiomics=False, cbam='branch'),
    # --- tabular baselines (logistic regression, same folds)
    'rad_only':       dict(kind='tab', cols=list(range(28))),
    'size_only':      dict(kind='tab', cols=[IV, IM]),
}
CORE = ['full', 'img_3s_cbam', 'rad_only', 'size_only']
ABLATION_TABLE = [   # (row label, variant key, purpose)  -- exactly the reviewer's table
    ('32^3 only',                         'img_32',        'Tests fine local texture'),
    ('48^3 only',                         'img_48',        'Tests intermediate morphology'),
    ('64^3 only',                         'img_64',        'Tests broader context'),
    ('32^3 + 48^3',                       'img_32_48',     'Tests two-scale fusion'),
    ('32^3 + 64^3',                       'img_32_64',     'Tests fine + broad fusion'),
    ('48^3 + 64^3',                       'img_48_64',     'Tests intermediate + broad fusion'),
    ('32+48+64, no CBAM (image-only)',    'img_3s_nocbam', 'Tests multiscale benefit without attention'),
    ('32+48+64 + CBAM + radiomics (FULL)', 'full',         'Full proposed model'),
    ('Full model without radiomics',      'img_3s_cbam',   'Tests image-only contribution'),
    ('Full model without CBAM',           'full_no_cbam',  'Tests attention contribution'),
    ('Radiomics-only',                    'rad_only',      'Tests handcrafted-feature baseline'),
]

def metrics(y, p, t=0.5):
    pred = (p >= t).astype(int); tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    d = lambda a, b: a / b if b else np.nan
    return dict(acc=(tp+tn)/len(y), sens=d(tp, tp+fn), spec=d(tn, tn+fp), prec=d(tp, tp+fp), f1=f1_score(y, pred, zero_division=0),
                auc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan, mcc=matthews_corrcoef(y, pred), thr=t)

def youden_thr(y, p):
    if len(set(y)) < 2: return 0.5
    fpr, tpr, th = roc_curve(y, p); return float(np.clip(th[np.argmax(tpr - fpr)], 0, 1))

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for p32, p48, p64, rad, y in loader:
        P += F.softmax(model(*[a.to(DEVICE) for a in (p32, p48, p64, rad)]), 1)[:, 1].cpu().tolist(); Y += y.tolist()
    return np.array(P), np.array(Y)

def model_dir(tag, vname, seed):
    d = os.path.join(MODEL_DIR, tag, vname, f"s{seed}"); os.makedirs(d, exist_ok=True); return d

def _load_fold_cache(cache):
    z = np.load(cache); return dict(te=z['te'], p_te=z['p_te'], p_hold=z['p_hold'], val_auc=float(z['val_auc']), thr=float(z['thr']))

def train_cnn_fold(df, df_hold, f, vname, seed, tag):
    d = model_dir(tag, vname, seed); cache = os.path.join(d, f"fold{f}.npz")
    if os.path.exists(cache): return _load_fold_cache(cache)
    spec = VARIANTS[vname]
    tr, va, te = make_folds(df, CFG['N_SPLITS'], CFG['SPLIT_SEED'])[f]
    sc = StandardScaler().fit(np.stack(df.rad.values[tr]))                       # radiomics scaler: TRAIN fold only
    R = lambda d_, i: np.clip(sc.transform(np.stack(d_.rad.values[i])), -5, 5).astype(np.float32)
    def mk(d_, i, aug):
        return DataLoader(NoduleDS(d_, i, R(d_, i), aug), CFG['BATCH'], shuffle=aug, num_workers=CFG['WORKERS'],
                          drop_last=aug and len(i) % CFG['BATCH'] == 1,
                          worker_init_fn=lambda w: np.random.seed((torch.initial_seed() + w) % 2**32))
    Ltr, Lva, Lte = mk(df, tr, True), mk(df, va, False), mk(df, te, False)
    set_seed(seed); model = Net(scales=spec['scales'], radiomics=spec['radiomics'], cbam=spec['cbam']).to(DEVICE)
    cnt = np.bincount(df.label.values[tr], minlength=2).clip(1); w = torch.tensor(len(tr) / (2 * cnt), dtype=torch.float32, device=DEVICE)
    lossf, opt = nn.CrossEntropyLoss(weight=w), torch.optim.Adam(model.parameters(), CFG['LR'], weight_decay=CFG['WD'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'max', factor=0.5, patience=max(3, CFG['PATIENCE'] // 3))
    amp = DEVICE.type == 'cuda'; scaler = torch.amp.GradScaler('cuda', enabled=amp)
    best, best_state, bad = -1, None, 0
    for ep in range(1, CFG['EPOCHS'] + 1):
        model.train()
        for p32, p48, p64, rad, y in Ltr:
            opt.zero_grad()
            with torch.autocast('cuda', enabled=amp):
                loss = lossf(model(*[a.to(DEVICE) for a in (p32, p48, p64, rad)]), y.to(DEVICE))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        pv, yv = predict(model, Lva); auc = roc_auc_score(yv, pv) if len(set(yv)) > 1 else 0.5
        sched.step(auc)
        if auc > best: best, best_state, bad = auc, copy.deepcopy(model.state_dict()), 0       # selection on VALIDATION only
        else: bad += 1
        if bad >= CFG['PATIENCE']: break
    model.load_state_dict(best_state)
    pv, yv = predict(model, Lva); thr = youden_thr(yv, pv)                                      # fold threshold from VALIDATION only
    pt, _ = predict(model, Lte)
    if df_hold is not None and len(df_hold):
        ph, _ = predict(model, mk(df_hold, np.arange(len(df_hold)), False))
    else: ph = np.zeros(0)
    torch.save(best_state, os.path.join(d, f"fold{f}.pt")); pickle.dump(sc, open(os.path.join(d, f"scaler_fold{f}.pkl"), 'wb'))
    np.savez(cache, te=te, p_te=pt, p_hold=ph, val_auc=best, thr=thr)
    return dict(te=te, p_te=pt, p_hold=ph, val_auc=best, thr=thr)

def train_tab_fold(df, df_hold, f, vname, tag):
    d = model_dir(tag, vname, 0); cache = os.path.join(d, f"fold{f}.npz")
    if os.path.exists(cache): return _load_fold_cache(cache)
    cols = VARIANTS[vname]['cols']; tr, va, te = make_folds(df, CFG['N_SPLITS'], CFG['SPLIT_SEED'])[f]
    X, y = np.stack(df.rad.values)[:, cols], df.label.values
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X[tr], y[tr])
    pv = pipe.predict_proba(X[va])[:, 1]; pt = pipe.predict_proba(X[te])[:, 1]
    ph = pipe.predict_proba(np.stack(df_hold.rad.values)[:, cols])[:, 1] if df_hold is not None and len(df_hold) else np.zeros(0)
    thr = youden_thr(y[va], pv); auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.5
    pickle.dump(pipe, open(os.path.join(d, f"pipe_fold{f}.pkl"), 'wb'))
    np.savez(cache, te=te, p_te=pt, p_hold=ph, val_auc=auc, thr=thr)
    return dict(te=te, p_te=pt, p_hold=ph, val_auc=auc, thr=thr)

def run_variant(vname, seed, df, df_hold, tag='dev'):
    # returns dict(oof, hold, thr, fold_thr, val_auc); cached per fold and per variant/seed
    seed = 0 if VARIANTS[vname]['kind'] == 'tab' else seed
    out = os.path.join(PRED_DIR, f"{tag}__{vname}__s{seed}.npz")
    if os.path.exists(out):
        z = np.load(out); return {k: z[k] for k in z.files}
    oof, hold, fthr, vauc = np.full(len(df), np.nan), [], [], []
    for f in range(CFG['N_SPLITS']):
        r = train_tab_fold(df, df_hold, f, vname, tag) if VARIANTS[vname]['kind'] == 'tab' else train_cnn_fold(df, df_hold, f, vname, seed, tag)
        oof[r['te']] = r['p_te']; fthr.append(r['thr']); vauc.append(r['val_auc'])
        if len(r['p_hold']): hold.append(r['p_hold'])
        if VARIANTS[vname]['kind'] == 'cnn': print(f"  [{tag}] {vname} seed {seed} fold {f}: val AUC {r['val_auc']:.4f}")
    assert np.isfinite(oof).all(), 'missing out-of-fold predictions'
    res = dict(oof=oof, hold=np.mean(hold, 0) if hold else np.zeros(0), thr=youden_thr(df.label.values, oof), fold_thr=np.array(fthr), val_auc=np.array(vauc))
    np.savez(out, **res); return res
print('Engine ready.')

## 5. Run all experiments (core models + complete ablation, several training seeds)
The *primary seed* is run first for every variant, then the extra seeds. `RUNS[variant][seed]` holds out-of-fold development predictions (`oof`), holdout predictions (`hold`, mean of the 5 fold models) and the development-chosen threshold (`thr`).

In [ ]:
ROOT = "/content/drive/MyDrive/lidc_work/full_rebuild"

RUN_DIR = os.path.join(
    ROOT, "training_20261020_01"
)

MODEL_DIR = os.path.join(RUN_DIR, "models")
PRED_DIR = os.path.join(RUN_DIR, "preds")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

print("ROOT:", ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("PRED_DIR:", PRED_DIR)

In [ ]:
def pred_set(v, mode='primary'):
    rs = RUNS[v]
    if mode == 'primary':
        r = rs[PS] if PS in rs else rs[sorted(rs)[0]]; return r['oof'], r['hold']
    return np.mean([r['oof'] for r in rs.values()], 0), np.mean([r['hold'] for r in rs.values()], 0)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

PATCH_DIR = "/content/drive/MyDrive/lidc_work/patches_full_rebuild"

patch_files = sorted(
    glob.glob(os.path.join(PATCH_DIR, "*.npz"))
)

if len(patch_files) != 443:
    raise RuntimeError(
        f"Expected 443 patch files, found {len(patch_files)}"
    )

rows = []

for f in patch_files:
    with np.load(f, allow_pickle=False) as z:
        rows.append({
            "file": f,
            "nodule_id": os.path.splitext(os.path.basename(f))[0],
            "patient": str(z["patient"].item()),
            "label": int(z["label"].item()),
            "score": float(z["score"].item()),
            "diam": float(z["diam"].item()),
            "n_ann": int(z["n_ann"].item()),
            "rad": z["rad"].astype(np.float32).copy(),
        })

DF = pd.DataFrame(rows)

print("DF rows:", len(DF))
print("Unique patients:", DF["patient"].nunique())
print("\nLabel counts:")
print(DF["label"].value_counts().sort_index())

wrong_paths = DF[
    ~DF["file"].str.startswith(PATCH_DIR + os.sep)
]
missing_files = DF[
    ~DF["file"].apply(os.path.isfile)
]

print("\nRows pointing outside PATCH_DIR:", len(wrong_paths))
print("Missing files:", len(missing_files))
print("Duplicate nodule IDs:", DF["nodule_id"].duplicated().sum())

assert len(DF) == 443
assert len(wrong_paths) == 0
assert len(missing_files) == 0
assert DF["nodule_id"].is_unique

print("\nPASS: DF is rebuilt from patches_full_rebuild.")

In [ ]:
import os
import glob

PATCH_DIR = "/content/drive/MyDrive/lidc_work/patches_full_rebuild"

patch_files = glob.glob(os.path.join(PATCH_DIR, "*.npz"))
patch_by_name = {os.path.basename(p): p for p in patch_files}

print("New patch files:", len(patch_files))
print("Unique patch filenames:", len(patch_by_name))
print("DF rows:", len(DF))

# Match using filenames, preserving all existing DF columns
df_names = DF["file"].map(os.path.basename)
missing_files = sorted(set(df_names) - set(patch_by_name))

print("DF filenames missing from rebuilt directory:", len(missing_files))

if missing_files:
    print("Examples:", missing_files[:10])
    raise RuntimeError("Some DF files have no matching rebuilt patch.")

# Update paths only after all filenames have matched
DF = DF.copy()
DF["file"] = df_names.map(patch_by_name)

assert DF["file"].apply(os.path.isfile).all()
assert DF["file"].str.startswith(PATCH_DIR + os.sep).all()

print("\nPASS: All DF paths now point to patches_full_rebuild.")
print("DF rows:", len(DF))
print("Unique patients:", DF["patient"].nunique())
print("Columns preserved:", DF.columns.tolist())

In [ ]:
import os
import numpy as np

PATCH_DIR = "/content/drive/MyDrive/lidc_work/patches_full_rebuild"

out_path = os.path.join(
    PATCH_DIR,
    "LIDC-IDRI-0487_56821227_n05.npz"
)

row = target.iloc[0]

assert not os.path.exists(out_path), "Patch already exists; do not overwrite."
assert hu.shape == (64, 64, 64)
assert m.shape == (64, 64, 64)
assert int(m.sum()) == 18, f"Unexpected mask size: {m.sum()}"
assert np.isfinite(hu).all()

rad = np.asarray(radiomics28(hu, m), dtype=np.float32)

assert rad.shape == (28,), f"Unexpected radiomics shape: {rad.shape}"
assert np.isfinite(rad).all(), "Radiomics contains non-finite values."

np.savez_compressed(
    out_path,
    p64=((hu - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float16),
    mask=m.astype(bool),
    rad=rad,
    label=int(row["label"]),
    score=float(row["score"]),
    patient=str(row["patient"]),
    diam=float(row["diam"]),
    n_ann=int(row["n_ann"]),
)

# Verify what was saved
with np.load(out_path, allow_pickle=False) as z:
    print("Saved:", out_path)
    print("Patch shape:", z["p64"].shape)
    print("Mask voxels:", int(z["mask"].sum()))
    print("Radiomics shape:", z["rad"].shape)
    print("Label:", int(z["label"]))
    print("Patient:", z["patient"].item())

In [ ]:
print("ROOT:", ROOT)
print("PATCH_DIR:", PATCH_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("PRED_DIR:", PRED_DIR)

print("\nConfiguration:")
print("N_SPLITS:", CFG["N_SPLITS"])
print("SEEDS:", CFG["SEEDS"])
print("RUN_ABLATION:", CFG["RUN_ABLATION"])
print("Variants:", list(VARIANTS))

In [ ]:
from pathlib import Path
import numpy as np

# Check for stale or partial training artifacts
model_files = [
    p for p in Path(MODEL_DIR).rglob("*")
    if p.is_file()
]
pred_files = [
    p for p in Path(PRED_DIR).rglob("*")
    if p.is_file()
]

print("Model files already present:", len(model_files))
print("Prediction files already present:", len(pred_files))

# Do not silently reuse cached results
assert len(model_files) == 0 and len(pred_files) == 0, (
    "Output directories are not empty. Inspect their contents before training."
)

# Validate the 3-fold split
folds = make_folds(DF_DEV, CFG["N_SPLITS"], CFG["SPLIT_SEED"])

assert len(folds) == CFG["N_SPLITS"]

all_test_indices = []

for f, (tr, va, te) in enumerate(folds):
    train_patients = set(DF_DEV.iloc[tr]["patient"])
    val_patients   = set(DF_DEV.iloc[va]["patient"])
    test_patients  = set(DF_DEV.iloc[te]["patient"])

    assert train_patients.isdisjoint(val_patients)
    assert train_patients.isdisjoint(test_patients)
    assert val_patients.isdisjoint(test_patients)

    all_test_indices.extend(te.tolist())

    print(
        f"Fold {f}: train={len(tr)}, val={len(va)}, test={len(te)} | "
        f"patients={len(train_patients)}/{len(val_patients)}/{len(test_patients)}"
    )

# Every development nodule must appear in exactly one test fold
assert sorted(all_test_indices) == list(range(len(DF_DEV)))

print("\nPRE-FLIGHT PASSED")
print("Ready to start the fresh training run.")

In [ ]:
RUNS = {v: {} for v in VARIANTS}
todo_variants = list(VARIANTS) if CFG['RUN_ABLATION'] else CORE
t0 = time.time()
for seed in CFG['SEEDS']:
    for v in todo_variants:
        if VARIANTS[v]['kind'] == 'tab' and seed != PS: continue
        key = 0 if VARIANTS[v]['kind'] == 'tab' else seed
        if key in RUNS[v]: continue
        print(f"=== {v} | seed {seed} ===")
        RUNS[v][key] = run_variant(v, seed, DF_DEV, DF_HOLD, tag='dev')
    print(f"--- seed {seed} finished | {(time.time()-t0)/60:.1f} min")
AVAILABLE = [v for v in VARIANTS if RUNS[v]]
print('Variants available:', AVAILABLE)

def pred_set(v, mode='primary'):
    rs = RUNS[v]
    if mode == 'primary':
        r = rs[PS] if PS in rs else rs[sorted(rs)[0]]; return r['oof'], r['hold']
    return np.mean([r['oof'] for r in rs.values()], 0), np.mean([r['hold'] for r in rs.values()], 0)

def dev_thr(v, mode='primary'): return youden_thr(y_dev, pred_set(v, mode)[0])

# headline (reproduces the previous notebook's final-holdout evaluation for the fused model)
oof_f, hold_f = pred_set('full'); FINAL_THRESHOLD = dev_thr('full')
print('\nFULL model | dev out-of-fold AUC %.4f | Youden thr (dev only) %.4f' % (roc_auc_score(y_dev, oof_f), FINAL_THRESHOLD))
print('fold-wise validation thresholds:', np.round(RUNS['full'][PS]['fold_thr'], 4), '(instability discussed in section 8)')
print('HOLDOUT (internal):', {k: round(float(v), 4) for k, v in metrics(y_hold, hold_f, FINAL_THRESHOLD).items()})

In [ ]:
from pathlib import Path
import numpy as np

print("=== Aggregate prediction files ===")
pred_files = sorted(Path(PRED_DIR).glob("*.npz"))

for path in pred_files:
    with np.load(path, allow_pickle=False) as data:
        print(f"\n{path.name}")
        for key in data.files:
            print(f"  {key}: shape={data[key].shape}, "
                  f"dtype={data[key].dtype}")

print("\nAggregate NPZ files:", len(pred_files))

print("\n=== Saved model and fold-cache files ===")
model_root = Path(MODEL_DIR)

for suffix in ("*.pt", "*.pkl", "*.npz"):
    matches = list(model_root.rglob(suffix))
    print(f"{suffix}: {len(matches)}")

print("\n=== Expected variants ===")
print("Configured:", len(VARIANTS))
print("Names:", list(VARIANTS))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)

pred_root = Path(PRED_DIR)

y_dev = DF_DEV["label"].to_numpy(dtype=int)
y_hold = DF_HOLD["label"].to_numpy(dtype=int)

assert len(y_dev) == 363, f"Unexpected development size: {len(y_dev)}"
assert len(y_hold) == 81, f"Unexpected holdout size: {len(y_hold)}"
assert set(np.unique(y_dev)).issubset({0, 1})
assert set(np.unique(y_hold)).issubset({0, 1})

results = []

files = sorted(pred_root.glob("dev__*__s*.npz"))
assert len(files) == 12, f"Expected 12 aggregate files, found {len(files)}"

for path in files:
    with np.load(path, allow_pickle=False) as data:
        oof = np.asarray(data["oof"], dtype=float)
        hold = np.asarray(data["hold"], dtype=float)
        threshold = float(data["thr"])
        fold_auc = np.asarray(data["val_auc"], dtype=float)

    variant = path.name[len("dev__"):].rsplit("__s", 1)[0]

    # Validate predictions before evaluating them
    assert oof.shape == y_dev.shape, (
        f"{variant}: OOF shape {oof.shape} does not match labels {y_dev.shape}"
    )
    assert hold.shape == y_hold.shape, (
        f"{variant}: holdout shape {hold.shape} does not match labels {y_hold.shape}"
    )
    assert np.isfinite(oof).all() and np.isfinite(hold).all(), (
        f"{variant}: non-finite prediction values"
    )
    assert np.all((oof >= 0) & (oof <= 1)), f"{variant}: OOF values outside [0,1]"
    assert np.all((hold >= 0) & (hold <= 1)), f"{variant}: holdout values outside [0,1]"
    assert np.isfinite(threshold) and 0 <= threshold <= 1

    # Ranking metrics
    dev_auc = roc_auc_score(y_dev, oof)
    hold_auc = roc_auc_score(y_hold, hold)

    # Threshold was selected on development data, not holdout
    pred_hold = (hold >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_hold, pred_hold, labels=[0, 1]
    ).ravel()

    results.append({
        "variant": variant,
        "dev_oof_auc": dev_auc,
        "mean_fold_val_auc": np.mean(fold_auc),
        "fold_val_auc_sd": np.std(fold_auc, ddof=1),
        "holdout_auc": hold_auc,
        "threshold_dev": threshold,
        "holdout_accuracy": accuracy_score(y_hold, pred_hold),
        "holdout_sensitivity": recall_score(
            y_hold, pred_hold, zero_division=0
        ),
        "holdout_specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "holdout_precision": precision_score(
            y_hold, pred_hold, zero_division=0
        ),
        "holdout_f1": f1_score(y_hold, pred_hold, zero_division=0),
        "holdout_mcc": matthews_corrcoef(y_hold, pred_hold),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    })

results_df = pd.DataFrame(results).sort_values(
    "dev_oof_auc", ascending=False
).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

print("Development class counts:")
print(pd.Series(y_dev).value_counts().sort_index().to_dict())
print("Holdout class counts:")
print(pd.Series(y_hold).value_counts().sort_index().to_dict())

print("\n=== All variants: development and holdout results ===")
display(results_df.round(4))

In [ ]:
import inspect

print("=" * 70)
print("1. MODEL VARIANT CONFIGURATION")
print("=" * 70)

for name, config in VARIANTS.items():
    print(f"{name}: {config}")

print("\n" + "=" * 70)
print("2. RADIOMICS FEATURE EXTRACTION")
print("=" * 70)

for name in ["radiomics28"]:
    obj = globals().get(name)
    if obj is not None:
        print(inspect.getsource(obj))

print("\n" + "=" * 70)
print("3. TABULAR TRAINING PIPELINE")
print("=" * 70)

for name in ["train_tab_fold"]:
    obj = globals().get(name)
    if obj is not None:
        print(inspect.getsource(obj))

print("\n" + "=" * 70)
print("4. TABULAR MODEL DEFINITION")
print("=" * 70)

for name in ["TabNet", "TabularNet", "TabMLP", "MLP"]:
    obj = globals().get(name)
    if obj is not None and inspect.isclass(obj):
        print(f"\nClass: {name}")
        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Source unavailable.")

print("\n" + "=" * 70)
print("5. FEATURE AND LABEL STRUCTURE")
print("=" * 70)

print("DF columns:", DF.columns.tolist())
print("Radiomics feature length:", len(DF.iloc[0]["rad"]))
print("Label counts:", DF["label"].value_counts().sort_index().to_dict())

print("\nExample metadata fields (first 3 rows):")
display(DF[["patient", "nod", "label", "score", "diam", "n_ann"]].head(3))

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y_dev = DF_DEV["label"].to_numpy(dtype=int)
diam = DF_DEV[["diam"]].to_numpy(dtype=float)

folds = make_folds(
    DF_DEV,
    CFG["N_SPLITS"],
    CFG["SPLIT_SEED"]
)

oof_diameter = np.full(len(DF_DEV), np.nan)
fold_aucs = []

for f, (tr, va, te) in enumerate(folds):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000)
    )

    # Train only on the training patients for this fold
    model.fit(diam[tr], y_dev[tr])

    # Predict each outer test fold exactly once
    oof_diameter[te] = model.predict_proba(diam[te])[:, 1]

    fold_auc = roc_auc_score(
        y_dev[te],
        oof_diameter[te]
    )
    fold_aucs.append(fold_auc)
    print(f"Fold {f} diameter-only test AUC: {fold_auc:.4f}")

assert np.isfinite(oof_diameter).all()

print("\nDiameter-only OOF AUC:",
      round(roc_auc_score(y_dev, oof_diameter), 4))
print("Mean fold test AUC:",
      round(float(np.mean(fold_aucs)), 4))
print("Fold AUC standard deviation:",
      round(float(np.std(fold_aucs, ddof=1)), 4))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01"
)

model_dir = run_dir / "models"
pred_dir = run_dir / "preds"

print("Run directory exists:", run_dir.exists())
print("Model directory exists:", model_dir.exists())
print("Prediction directory exists:", pred_dir.exists())

if pred_dir.exists():
    files = sorted(pred_dir.glob("dev__*__s*.npz"))
    print("Aggregate prediction files:", len(files))
    for f in files:
        print(f.name)

if model_dir.exists():
    print("Saved model weights:", len(list(model_dir.rglob("*.pt"))))
    print("Saved scalers:", len(list(model_dir.rglob("*.pkl"))))
    print("Saved fold caches:", len(list(model_dir.rglob("*.npz"))))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PATCH_DIR = Path(
    "/content/drive/MyDrive/lidc_work/patches_full_rebuild"
)

files = sorted(PATCH_DIR.glob("*.npz"))
print("Patch files found:", len(files))

rows = []
problems = []

for path in files:
    try:
        with np.load(path, allow_pickle=False) as z:
            p = z["p64"]
            mask = z["mask"]
            rad = z["rad"]
            label = int(z["label"])
            score = float(z["score"])
            diam = float(z["diam"])
            patient = str(z["patient"].item())
            voxels = int(mask.sum())

        if p.shape != (64, 64, 64):
            problems.append((path.name, "wrong patch shape"))
        if mask.shape != (64, 64, 64):
            problems.append((path.name, "wrong mask shape"))
        if rad.shape != (28,):
            problems.append((path.name, "wrong radiomics shape"))
        if not np.isfinite(p).all() or not np.isfinite(rad).all():
            problems.append((path.name, "non-finite values"))
        if voxels == 0:
            problems.append((path.name, "empty mask"))

        expected_label = (
            0 if score <= 2 else 1 if score >= 4 else -1
        )
        if label != expected_label:
            problems.append((path.name, "label-score mismatch"))

        rows.append({
            "file": path.name,
            "patient": patient,
            "label": label,
            "score": score,
            "diam": diam,
            "mask_voxels": voxels,
        })

    except Exception as e:
        problems.append((path.name, repr(e)))

audit = pd.DataFrame(rows)

print("Readable patches:", len(audit))
print("Unique patients:", audit["patient"].nunique())
print("Label counts:",
      audit["label"].value_counts().sort_index().to_dict())
print("Masks below 20 voxels:",
      int((audit["mask_voxels"] < 20).sum()))
print("Empty masks:",
      int((audit["mask_voxels"] == 0).sum()))
print("Problems:", len(problems))

print("\nMasks below 20 voxels:")
display(
    audit.loc[
        audit["mask_voxels"] < 20,
        ["file", "label", "score", "diam", "mask_voxels"]
    ]
)

if problems:
    print("\nFirst 30 problems:")
    for problem in problems[:30]:
        print(problem)

assert len(files) == 444, "Unexpected patch count"
assert len(audit) == 444, "Some patches could not be read"

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, brier_score_loss

# ------------------------------------------------------------
# 1. Paths: use the fresh run, not the older frozen-model folder
# ------------------------------------------------------------
WORK = Path("/content/drive/MyDrive/lidc_work")
RUN = WORK / "full_rebuild" / "training_20261020_01"

MODEL_DIR = RUN / "models" / "dev" / "full" / "s42"
DEV_PRED = RUN / "preds" / "dev__full__s42.npz"

EXT_ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MANIFEST = EXT_ROOT / "manifest.csv"
PATCH_DIR = WORK / "external_validation" / "LNDb" / "patches"

OUT = RUN / "external_validation_fresh" / (
    "LNDb_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)

for p in [MODEL_DIR, DEV_PRED, MANIFEST, PATCH_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"Required path not found: {p}")

# ------------------------------------------------------------
# 2. Verify exact case-to-patch matching before inference
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST)
required = {"case_id", "patient_id", "label", "reference_type"}

if not required.issubset(df.columns):
    raise ValueError(f"Manifest missing columns: {required - set(df.columns)}")

df["case_id"] = df["case_id"].astype(str).str.strip()

if df["case_id"].duplicated().any():
    raise ValueError("Duplicate case_id values in manifest.")

if df["label"].isna().any():
    raise ValueError("Missing labels in manifest.")

y = df["label"].to_numpy(dtype=int)

if not set(np.unique(y)).issubset({0, 1}) or len(np.unique(y)) != 2:
    raise ValueError(f"Expected both binary labels 0/1; found {np.unique(y)}")

expected = {f"{case_id}.npz" for case_id in df["case_id"]}
actual = {p.name for p in PATCH_DIR.glob("*.npz")}

missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("Manifest cases:", len(df))
print("Patch files:", len(actual))
print("Unique patients:", df["patient_id"].nunique())
print("Label counts:", df["label"].value_counts().sort_index().to_dict())
print("Missing patches:", len(missing))
print("Unmatched patch files:", len(extra))

if missing or extra or len(df) != 521:
    raise RuntimeError(
        "Case-to-patch matching failed. Stopping before evaluation. "
        f"First missing={missing[:5]}, first extra={extra[:5]}"
    )

# Use the threshold chosen from fresh LIDC development OOF predictions.
with np.load(DEV_PRED, allow_pickle=False) as z:
    threshold = float(z["thr"])

print("Frozen LIDC development OOF threshold:", threshold)

# ------------------------------------------------------------
# 3. Recreate the exact full-model architecture
# ------------------------------------------------------------
# These are the architecture settings from the saved notebook.
CFG = {"FEAT_DIM": 128, "FC": (256, 128), "DROPOUT": (0.4, 0.3)}

class CBAM3D(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        h = max(c // r, 4)
        self.mlp = nn.Sequential(
            nn.Linear(c, h),
            nn.ReLU(True),
            nn.Linear(h, c)
        )
        self.conv = nn.Conv3d(2, 1, 7, padding=3)

    def forward(self, x):
        x = x * torch.sigmoid(
            self.mlp(x.mean((2, 3, 4))) +
            self.mlp(x.amax((2, 3, 4)))
        )[:, :, None, None, None]

        return x * torch.sigmoid(
            self.conv(torch.cat([
                x.mean(1, keepdim=True),
                x.amax(1, keepdim=True)
            ], 1))
        )

class Branch(nn.Module):
    def __init__(self, cbam, fd):
        super().__init__()
        layers = []
        for ci, co in [(1, 16), (16, 32), (32, 64)]:
            layers += [
                nn.Conv3d(ci, co, 3, padding=1),
                nn.BatchNorm3d(co),
                nn.ReLU(True),
                nn.MaxPool3d(2)
            ]

        self.conv = nn.Sequential(*layers)
        self.att = CBAM3D(64) if cbam == "branch" else nn.Identity()
        self.fc = nn.Linear(64, fd)

    def forward(self, x):
        x = self.att(self.conv(x))
        return self.fc(x.mean((2, 3, 4)))

class Net(nn.Module):
    def __init__(
        self,
        scales=(32, 48, 64),
        radiomics=True,
        cbam="branch",
        fd=None
    ):
        super().__init__()
        fd = fd or CFG["FEAT_DIM"]
        self.scales = tuple(scales)
        self.use_rad = radiomics
        self.b = nn.ModuleList([
            Branch(cbam, fd) for _ in self.scales
        ])

        dim = fd * len(self.scales) + (28 if radiomics else 0)
        self.gate = None

        h1, h2 = CFG["FC"]
        d1, d2 = CFG["DROPOUT"]

        self.head = nn.Sequential(
            nn.Linear(dim, h1),
            nn.ReLU(True),
            nn.Dropout(d1),
            nn.Linear(h1, h2),
            nn.ReLU(True),
            nn.Dropout(d2),
            nn.Linear(h2, 2)
        )

    def forward(self, p32, p48, p64, rad):
        inputs = {32: p32, 48: p48, 64: p64}
        pieces = [
            branch(inputs[scale])
            for branch, scale in zip(self.b, self.scales)
        ]

        if self.use_rad:
            pieces.append(rad)

        return self.head(torch.cat(pieces, 1))

# ------------------------------------------------------------
# 4. Load fresh fold models and their own saved scalers
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Inference device:", device)

models = []
hash_rows = []

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for fold in range(3):
    ckpt = MODEL_DIR / f"fold{fold}.pt"
    scaler_path = MODEL_DIR / f"scaler_fold{fold}.pkl"

    if not ckpt.exists() or not scaler_path.exists():
        raise FileNotFoundError(
            f"Missing fold checkpoint or scaler for fold {fold}"
        )

    try:
        state = torch.load(ckpt, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(ckpt, map_location="cpu")

    model = Net(
        scales=(32, 48, 64),
        radiomics=True,
        cbam="branch"
    ).to(device)

    # Strict loading stops if the architecture and weights disagree.
    model.load_state_dict(state, strict=True)
    model.eval()

    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)

    models.append((model, scaler))

    hash_rows.append({
        "file": str(ckpt),
        "sha256": sha256_file(ckpt)
    })
    hash_rows.append({
        "file": str(scaler_path),
        "sha256": sha256_file(scaler_path)
    })

print("Loaded and verified all three fold models.")

# ------------------------------------------------------------
# 5. Predict in batches; no augmentation or training
# ------------------------------------------------------------
fold_probs = np.zeros((3, len(df)), dtype=np.float64)
batch_size = 16

with torch.inference_mode():
    for fold, (model, scaler) in enumerate(models):
        print(f"Predicting with fold {fold}...")

        for start in range(0, len(df), batch_size):
            end = min(start + batch_size, len(df))
            patches = []
            radiomics = []

            for case_id in df["case_id"].iloc[start:end]:
                path = PATCH_DIR / f"{case_id}.npz"

                with np.load(path, allow_pickle=False) as data:
                    p = data["p64"].astype(np.float32)
                    r = data["rad"].astype(np.float32)

                if p.shape != (64, 64, 64):
                    raise ValueError(f"Unexpected patch shape in {path}: {p.shape}")
                if r.shape != (28,):
                    raise ValueError(f"Unexpected radiomics shape in {path}: {r.shape}")
                if not np.isfinite(p).all() or not np.isfinite(r).all():
                    raise ValueError(f"Non-finite values in {path}")

                patches.append(p)
                radiomics.append(r)

            P = np.stack(patches)
            R = np.stack(radiomics)

            # Apply the matching fold's training-fitted scaler.
            R = np.clip(scaler.transform(R), -5, 5).astype(np.float32)

            x = torch.from_numpy(P).to(device)[:, None]
            r = torch.from_numpy(R).to(device)

            logits = model(
                x[:, :, 16:48, 16:48, 16:48],
                x[:, :, 8:56, 8:56, 8:56],
                x,
                r
            )

            fold_probs[fold, start:end] = (
                F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            )

# Match the ensemble method: average the three fold probabilities.
prob = fold_probs.mean(axis=0)
pred = (prob >= threshold).astype(int)

# ------------------------------------------------------------
# 6. Calculate external metrics at the frozen threshold
# ------------------------------------------------------------
tn = int(((y == 0) & (pred == 0)).sum())
fp = int(((y == 0) & (pred == 1)).sum())
fn = int(((y == 1) & (pred == 0)).sum())
tp = int(((y == 1) & (pred == 1)).sum())

sensitivity = tp / (tp + fn) if tp + fn else float("nan")
specificity = tn / (tn + fp) if tn + fp else float("nan")
precision = tp / (tp + fp) if tp + fp else float("nan")
accuracy = (tp + tn) / len(y)
f1 = (
    2 * precision * sensitivity / (precision + sensitivity)
    if np.isfinite(precision) and np.isfinite(sensitivity)
    and precision + sensitivity > 0
    else 0.0
)
mcc_den = np.sqrt(
    (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
)
mcc = ((tp * tn - fp * fn) / mcc_den) if mcc_den else 0.0

metrics = {
    "n_nodules": len(y),
    "n_patients": int(df["patient_id"].nunique()),
    "n_reference_positive": int(y.sum()),
    "n_reference_negative": int((y == 0).sum()),
    "auc": float(roc_auc_score(y, prob)),
    "threshold_from_lidc_development_oof": threshold,
    "accuracy": accuracy,
    "sensitivity": sensitivity,
    "specificity": specificity,
    "precision": precision,
    "f1": f1,
    "mcc": mcc,
    "brier_raw_probabilities": float(brier_score_loss(y, prob)),
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
    "reference_type": "; ".join(sorted(
        df["reference_type"].dropna().astype(str).unique()
    ))
}

predictions = df.copy()
predictions["prob_fold0"] = fold_probs[0]
predictions["prob_fold1"] = fold_probs[1]
predictions["prob_fold2"] = fold_probs[2]
predictions["probability_mean"] = prob
predictions["prediction_at_frozen_threshold"] = pred

# ------------------------------------------------------------
# 7. Save to a new directory; never overwrite earlier reports
# ------------------------------------------------------------
OUT.mkdir(parents=True, exist_ok=False)

predictions.to_csv(OUT / "predictions.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUT / "metrics.csv", index=False)
pd.DataFrame(hash_rows).to_csv(OUT / "checkpoint_hashes.csv", index=False)

statement = f"""
Fresh checkpoint external evaluation: LNDb
Run directory: {RUN}
Patch directory: {PATCH_DIR}
Manifest: {MANIFEST}

Number of nodules: {len(df)}
Number of patients: {df['patient_id'].nunique()}
Reference labels: {metrics['reference_type']}

Model: fresh full model, three saved fold checkpoints, seed 42.
Inference: each fold checkpoint used with its corresponding saved scaler.
Ensemble: mean of the three fold positive-class probabilities.
Threshold: {threshold:.10f}, loaded from the fresh LIDC development OOF prediction artifact.
No retraining, calibration fitting, threshold selection, or tuning was performed on LNDb.
Predictions and metrics are saved separately from the previous LNDb reports.

Interpretation: evaluation against the LNDb manifest's radiologist-assessed
risk-proxy labels; these are not necessarily confirmed cancer diagnoses.
"""
(OUT / "evaluation_statement.txt").write_text(statement.strip() + "\n")

print("\n=== FRESH LNDb EVALUATION ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

print("\nResults saved to:", OUT)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    precision_score, f1_score, matthews_corrcoef,
    brier_score_loss
)

RUN = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01"
)

files = sorted(
    (RUN / "external_validation_fresh").rglob("predictions.csv"),
    key=lambda p: p.stat().st_mtime
)
if not files:
    raise FileNotFoundError("Could not find fresh LNDb predictions.csv")

pred_file = files[-1]
df = pd.read_csv(pred_file)

required = {
    "patient_id", "label", "probability_mean",
    "prediction_at_frozen_threshold"
}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")

y = df["label"].to_numpy(dtype=int)
p = df["probability_mean"].to_numpy(dtype=float)
pred = df["prediction_at_frozen_threshold"].to_numpy(dtype=int)

# Patient-level clusters
patient_indices = {
    patient: np.flatnonzero(df["patient_id"].to_numpy() == patient)
    for patient in df["patient_id"].unique()
}
patients = list(patient_indices)
rng = np.random.default_rng(20261021)

def calculate_metrics(idx):
    yy, pp, pr = y[idx], p[idx], pred[idx]

    if len(np.unique(yy)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(yy, pp)

    return {
        "auc": auc,
        "accuracy": accuracy_score(yy, pr),
        "sensitivity": recall_score(yy, pr, zero_division=0),
        "specificity": recall_score(1 - yy, 1 - pr, zero_division=0),
        "precision": precision_score(yy, pr, zero_division=0),
        "f1": f1_score(yy, pr, zero_division=0),
        "mcc": matthews_corrcoef(yy, pr),
        "brier": brier_score_loss(yy, pp),
    }

point = calculate_metrics(np.arange(len(df)))

B = 2000
samples = {k: [] for k in point}
for _ in range(B):
    chosen = rng.choice(patients, size=len(patients), replace=True)
    # Repeated patient selections intentionally repeat their nodules.
    idx = np.concatenate([patient_indices[g] for g in chosen])
    m = calculate_metrics(idx)
    for k, value in m.items():
        if np.isfinite(value):
            samples[k].append(value)

rows = []
for metric, estimate in point.items():
    vals = np.asarray(samples[metric])
    rows.append({
        "metric": metric,
        "estimate": estimate,
        "ci_lo": np.quantile(vals, 0.025),
        "ci_hi": np.quantile(vals, 0.975),
        "valid_bootstrap_replicates": len(vals),
    })

result = pd.DataFrame(rows)
out = pred_file.parent / "patient_cluster_bootstrap_ci.csv"
result.to_csv(out, index=False)

print("Predictions:", pred_file)
print("\nPatient-cluster bootstrap 95% confidence intervals:")
print(result.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSaved:", out)

In [ ]:
from pathlib import Path
import requests
import pandas as pd
import io

ROOT = Path("/content/drive/MyDrive/external_cohorts/LUNGx")
LABEL_DIR = ROOT / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)

OUT = LABEL_DIR / "TestSet_NoduleData_PublicRelease_wTruth.xlsx"

URL = (
    "https://wiki.cancerimagingarchive.net/download/attachments/"
    "19039197/TestSet_NoduleData_PublicRelease_wTruth.xlsx"
    "?api=v2&modificationDate=1474635989693&version=1"
)

if OUT.exists():
    print("Using existing file:", OUT)
    content = OUT.read_bytes()
else:
    print("Downloading official LUNGx test-set reference spreadsheet...")
    response = requests.get(URL, timeout=90)
    response.raise_for_status()
    content = response.content

    # Reject an HTML error/login page rather than saving it as an Excel file.
    if not content.startswith(b"PK"):
        raise RuntimeError(
            "The response does not look like an XLSX file. "
            "No file was saved. Check the download URL or access."
        )

    OUT.write_bytes(content)
    print("Saved:", OUT)

xls = pd.ExcelFile(io.BytesIO(content))
print("\nSheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(io.BytesIO(content), sheet_name=sheet)
    print(f"\n=== Sheet: {sheet} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(8).to_string(index=False))

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pydicom

ROOT = Path("/content/drive/MyDrive/external_cohorts/LUNGx")
XLSX = ROOT / "labels" / "TestSet_NoduleData_PublicRelease_wTruth.xlsx"
META = ROOT / "images" / "metadata" / "metadata.csv"
OUT = ROOT / "labels" / "lungx_test_manifest_world_coordinates.csv"

# --------------------------------------------------
# 1. Load and validate the 73 test nodule annotations
# --------------------------------------------------
labels = pd.read_excel(XLSX)
metadata = pd.read_csv(META)

labels["Scan Number"] = labels["Scan Number"].astype("string").str.strip()
labels["Final Diagnosis"] = (
    labels["Final Diagnosis"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
labels["Nodule Number"] = pd.to_numeric(
    labels["Nodule Number"], errors="coerce"
)
labels["Nodule Center Image"] = pd.to_numeric(
    labels["Nodule Center Image"], errors="coerce"
)

valid = (
    labels["Scan Number"].str.fullmatch(r"LUNGx-CT\d+", na=False)
    & labels["Nodule Number"].notna()
    & labels["Nodule Center Image"].notna()
    & labels["Nodule Center x,y Position*"].notna()
    & labels["Final Diagnosis"].notna()
)
labels = labels.loc[valid].copy()

if len(labels) != 73 or labels["Scan Number"].nunique() != 60:
    raise RuntimeError("Expected 73 nodules across 60 test scans.")

if labels.duplicated(["Scan Number", "Nodule Number"]).any():
    raise RuntimeError("Duplicate scan/nodule identifiers found.")

def binary_label(diagnosis):
    text = str(diagnosis).strip().casefold()
    if text == "benign nodule":
        return 0
    if "malignant" in text or "cancer" in text:
        return 1
    raise ValueError(f"Unrecognized diagnosis: {diagnosis!r}")

labels["label"] = labels["Final Diagnosis"].map(binary_label)

if labels["label"].value_counts().to_dict() != {0: 37, 1: 36}:
    raise RuntimeError("Unexpected binary-label counts.")

metadata["PatientID"] = metadata["PatientID"].astype(str).str.strip()
metadata = metadata.set_index("PatientID", verify_integrity=True)

# --------------------------------------------------
# 2. Convert pixel coordinates using the stated DICOM
#    slice's ImagePositionPatient, orientation and spacing
# --------------------------------------------------
records = []

for scan_id, rows in labels.groupby("Scan Number", sort=True):
    if scan_id not in metadata.index:
        raise FileNotFoundError(f"No DICOM metadata row for {scan_id}")

    meta = metadata.loc[scan_id]
    series_dir = Path(str(meta["S5cmdManifestPath"]))

    if not series_dir.is_dir():
        raise FileNotFoundError(f"DICOM series directory missing: {series_dir}")

    # Build instance-number lookup for this scan.
    by_instance = {}
    series_uids = set()

    for file_path in series_dir.rglob("*.dcm"):
        try:
            ds = pydicom.dcmread(
                str(file_path),
                stop_before_pixels=True,
                specific_tags=[
                    "InstanceNumber", "ImagePositionPatient",
                    "ImageOrientationPatient", "PixelSpacing",
                    "Rows", "Columns", "SeriesInstanceUID"
                ]
            )
            if not hasattr(ds, "InstanceNumber"):
                continue

            instance = int(ds.InstanceNumber)
            if instance in by_instance:
                raise RuntimeError(
                    f"{scan_id}: duplicate InstanceNumber {instance}"
                )

            required_tags = [
                "ImagePositionPatient", "ImageOrientationPatient",
                "PixelSpacing", "Rows", "Columns", "SeriesInstanceUID"
            ]
            if not all(hasattr(ds, t) for t in required_tags):
                raise RuntimeError(
                    f"Required DICOM geometry missing: {file_path}"
                )

            by_instance[instance] = {
                "path": str(file_path),
                "ipp": np.asarray(ds.ImagePositionPatient, dtype=float),
                "iop": np.asarray(ds.ImageOrientationPatient, dtype=float),
                "spacing": np.asarray(ds.PixelSpacing, dtype=float),
                "rows": int(ds.Rows),
                "cols": int(ds.Columns),
                "series_uid": str(ds.SeriesInstanceUID),
                "slice_thickness_mm": float(
                    getattr(ds, "SliceThickness", np.nan)
                )
            }
            series_uids.add(str(ds.SeriesInstanceUID))

        except Exception as e:
            if isinstance(e, RuntimeError):
                raise
            raise RuntimeError(
                f"Could not inspect DICOM header {file_path}: {e}"
            ) from e

    if len(series_uids) != 1:
        raise RuntimeError(
            f"{scan_id}: expected one DICOM series, found {len(series_uids)}"
        )

    expected_uid = str(meta["SeriesInstanceUID"])
    if expected_uid not in series_uids:
        raise RuntimeError(
            f"{scan_id}: DICOM SeriesInstanceUID does not match metadata"
        )

    for _, row in rows.iterrows():
        image_number = int(row["Nodule Center Image"])

        if image_number not in by_instance:
            raise RuntimeError(
                f"{scan_id}: no DICOM with InstanceNumber={image_number}"
            )

        d = by_instance[image_number]

        xy_match = re.fullmatch(
            r"\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*",
            str(row["Nodule Center x,y Position*"])
        )
        if not xy_match:
            raise ValueError(
                f"Invalid pixel coordinate: "
                f"{row['Nodule Center x,y Position*']!r}"
            )

        pixel_x, pixel_y = map(float, xy_match.groups())

        if not (
            1 <= pixel_x <= d["cols"]
            and 1 <= pixel_y <= d["rows"]
        ):
            raise RuntimeError(
                f"{scan_id} nodule {row['Nodule Number']}: "
                "pixel coordinate outside the image"
            )

        # Spreadsheet coordinates are one-based:
        # x = column, y = row.
        col_index = pixel_x - 1.0
        row_index = pixel_y - 1.0

        ipp = d["ipp"]
        iop = d["iop"]
        spacing = d["spacing"]

        # DICOM mapping:
        # first IOP vector advances along columns;
        # second IOP vector advances along rows.
        world = (
            ipp
            + col_index * spacing[1] * iop[:3]
            + row_index * spacing[0] * iop[3:]
        )

        records.append({
            "case_id": f"{scan_id}_N{int(row['Nodule Number']):02d}",
            "patient_id": scan_id,
            "image_path": str(series_dir),
            "instance_number": image_number,
            "pixel_x": pixel_x,
            "pixel_y": pixel_y,
            "world_x": float(world[0]),
            "world_y": float(world[1]),
            "world_z": float(world[2]),
            "label": int(row["label"]),
            "reference_diagnosis": str(row["Final Diagnosis"]),
            "reference_type": "LUNGx published test-set diagnosis",
            "diameter_mm": np.nan,  # Not present in the reference spreadsheet
            "scanner": str(meta.get("ManufacturerModelName", "")),
            "slice_thickness_mm": d["slice_thickness_mm"],
            "series_instance_uid": d["series_uid"],
            "dicom_file": d["path"]
        })

manifest = pd.DataFrame(records).sort_values(
    ["patient_id", "instance_number", "case_id"]
).reset_index(drop=True)

if len(manifest) != 73 or manifest["case_id"].duplicated().any():
    raise RuntimeError("Final manifest row count or uniqueness check failed.")

if not np.isfinite(
    manifest[["world_x", "world_y", "world_z"]].to_numpy()
).all():
    raise RuntimeError("Non-finite physical coordinates found.")

OUT.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(OUT, index=False)

print("Manifest created:", OUT)
print("Nodules:", len(manifest))
print("Patients:", manifest["patient_id"].nunique())
print("Labels:", manifest["label"].value_counts().sort_index().to_dict())
print("Missing diameters:", int(manifest["diameter_mm"].isna().sum()))
print("\nSample manifest rows:")
print(
    manifest[
        [
            "case_id", "instance_number", "pixel_x", "pixel_y",
            "world_x", "world_y", "world_z", "label", "diameter_mm"
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

WORK = Path("/content/drive/MyDrive/lidc_work")
RUN = WORK / "full_rebuild" / "training_20261020_01"

ELIGIBLE = WORK / "eligible_nodules.csv"
HOLDOUT = WORK / "full_rebuild" / "final_holdout_patients.csv"
PATCH_DIR = WORK / "patches_full_rebuild"
PREDICTIONS = RUN / "preds" / "dev__full__s42.npz"

eligible = pd.read_csv(ELIGIBLE)
hold = pd.read_csv(HOLDOUT)

eligible["patient"] = eligible["patient"].astype(str).str.strip()
hold["patient"] = hold["patient"].astype(str).str.strip()

holdout_ids = set(hold["patient"])
is_holdout = eligible["patient"].isin(holdout_ids)

print("=== Correct fresh-run split ===")
print("Eligible nodules:", len(eligible))
print("Eligible patients:", eligible["patient"].nunique())
print("Holdout patients:", len(holdout_ids))
print("Holdout nodules:", int(is_holdout.sum()))
print("Development nodules:", int((~is_holdout).sum()))
print("Patient overlap:", len(
    set(eligible.loc[~is_holdout, "patient"]) & holdout_ids
))

print("\n=== Fresh patch counts ===")
patch_rows = []
for p in PATCH_DIR.glob("*.npz"):
    with np.load(p, allow_pickle=False) as z:
        patch_rows.append({
            "file": p.name,
            "patient": str(np.asarray(z["patient"]).item()).strip()
        })

patches = pd.DataFrame(patch_rows)
print("Patch files:", len(patches))
print("Unique patch patients:", patches["patient"].nunique())
print("Holdout patches:", int(patches["patient"].isin(holdout_ids).sum()))

print("\n=== Fresh prediction counts ===")
with np.load(PREDICTIONS, allow_pickle=False) as z:
    print("OOF predictions:", len(z["oof"]))
    print("Holdout predictions:", len(z["hold"]))

print("\n=== Diameter distribution from the correct development split ===")
dev = eligible.loc[~is_holdout].copy()
dev["diam"] = pd.to_numeric(dev["diam"], errors="coerce")
print(dev["diam"].describe(
    percentiles=[.10, .25, .50, .75, .90, .95]
).to_string())

assert len(dev) == 363, "Development count does not match fresh-run OOF predictions."
assert int(is_holdout.sum()) == 81, "Holdout count does not match fresh-run holdout predictions."
assert set(dev["patient"]).isdisjoint(holdout_ids)

print("\nPASS: patient split counts agree with the fresh-run prediction artifact.")

In [ ]:
array_index_by_case = dict(zip(
    audit["case_id"],
    audit["array_slice_index"].astype(int)
))

In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import numpy as np
import pandas as pd
import SimpleITK as sitk

from scipy import ndimage
from scipy.stats import skew, kurtosis
from scipy.spatial.distance import pdist

# ------------------------------------------------------------
# 1. Paths and development-only mask parameters
# ------------------------------------------------------------
WORK = Path("/content/drive/MyDrive/lidc_work")
RUN = WORK / "full_rebuild" / "training_20261020_01"

LUNGX_ROOT = Path("/content/drive/MyDrive/external_cohorts/LUNGx")
MANIFEST = (
    LUNGX_ROOT / "labels" /
    "lungx_test_manifest_world_coordinates.csv"
)
ELIGIBLE = WORK / "eligible_nodules.csv"
HOLDOUT = WORK / "full_rebuild" / "final_holdout_patients.csv"

for path in [MANIFEST, ELIGIBLE, HOLDOUT]:
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")

eligible = pd.read_csv(ELIGIBLE)
holdout_df = pd.read_csv(HOLDOUT)

eligible["patient"] = eligible["patient"].astype(str).str.strip()
holdout_ids = set(holdout_df["patient"].astype(str).str.strip())

dev = eligible.loc[
    ~eligible["patient"].isin(holdout_ids)
].copy()
dev["diam"] = pd.to_numeric(dev["diam"], errors="coerce")
dev = dev.loc[dev["diam"].notna() & (dev["diam"] > 0)]

if len(dev) != 363:
    raise RuntimeError(
        f"Expected 363 development nodules, found {len(dev)}"
    )

diam_median = float(dev["diam"].median())
diam_q75 = float(dev["diam"].quantile(0.75))

SEARCH_RADIUS_MM = 10.0
FALLBACK_RADIUS_MM = 3.5

print("LIDC development nodules used:", len(dev))
print(f"Development diameter median: {diam_median:.4f} mm")
print(f"Development diameter 75th percentile: {diam_q75:.4f} mm")
print(f"Fixed mask search radius: {SEARCH_RADIUS_MM:.4f} mm")
print(f"Fixed fallback sphere radius: {FALLBACK_RADIUS_MM:.4f} mm")

# Save outputs separately from the existing LNDb evaluation.
OUT = (
    RUN / "external_validation_fresh" /
    ("LUNGx_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
)
PATCH_OUT = OUT / "patches"
PATCH_OUT.mkdir(parents=True, exist_ok=False)

# ------------------------------------------------------------
# 2. Validate the LUNGx manifest
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST)

required = {
    "case_id", "patient_id", "image_path",
    "world_x", "world_y", "world_z", "label"
}
if not required.issubset(manifest.columns):
    raise ValueError(
        f"Manifest is missing columns: {required - set(manifest.columns)}"
    )

if len(manifest) != 73 or manifest["patient_id"].nunique() != 60:
    raise RuntimeError("Expected 73 nodules across 60 LUNGx scans.")

if manifest["case_id"].duplicated().any():
    raise RuntimeError("Duplicate LUNGx case IDs.")

if manifest["label"].value_counts().to_dict() != {0: 37, 1: 36}:
    raise RuntimeError("Unexpected LUNGx reference-label counts.")

# ------------------------------------------------------------
# 3. Reproduce the training patch preprocessing
# ------------------------------------------------------------
HU_MIN, HU_MAX, PATCH = -1000, 400, 64

def crop(v, c, half, fill):
    lo = np.round(c).astype(int) - half
    out = np.full(2 * half, fill, dtype=v.dtype)

    s = np.maximum(lo, 0)
    e = np.minimum(lo + 2 * half, v.shape)

    if (e > s).all():
        out[
            tuple(slice(a, b) for a, b in zip(s - lo, e - lo))
        ] = v[tuple(slice(a, b) for a, b in zip(s, e))]

    return out

def fit(a, n, fill):
    if a.shape == (n, n, n):
        return a
    return crop(
        a,
        np.array(a.shape) / 2,
        np.full(3, n // 2),
        fill
    )

def make_patch(vol, mask, centre, spacing_zyx):
    half = np.ceil(
        (PATCH / 2 + 4) / np.asarray(spacing_zyx)
    ).astype(int)

    v = crop(vol, centre, half, HU_MIN).astype(np.float32)
    m = crop(mask.astype(np.uint8), centre, half, 0)

    zoom = np.asarray(spacing_zyx, dtype=float) / 1.0
    v = ndimage.zoom(v, zoom, order=1)
    m = ndimage.zoom(m, zoom, order=0)

    v = fit(np.clip(v, HU_MIN, HU_MAX), PATCH, HU_MIN)
    m = fit(m, PATCH, 0).astype(bool)

    return v, m

RAD_NAMES = [
    "mean", "median", "std", "var", "min", "max", "range",
    "p10", "p90", "entropy", "volume", "surface", "sphericity",
    "compactness", "elongation", "flatness", "max_diam", "sv_ratio",
    "skew", "kurtosis", "energy", "uniformity", "rms", "mad",
    "grad_mean", "grad_std", "autocorr", "cov"
]

def radiomics28(x, m):
    v = x[m].astype(np.float64)
    n = len(v)

    if n < 10:
        raise ValueError(f"Mask too small for radiomics: {n} voxels")

    h, _ = np.histogram(v, bins=64)
    p = h[h > 0] / h.sum()
    ent = -(p * np.log2(p)).sum()

    vol = float(n)
    surf = float(n - ndimage.binary_erosion(m).sum()) + 1e-8

    co = np.argwhere(m).astype(float)
    co -= co.mean(0)

    ev = np.clip(
        np.sort(np.linalg.eigvalsh(
            np.cov(co.T) + 1e-6 * np.eye(3)
        ))[::-1],
        1e-8,
        None
    )

    sub = co[
        np.random.RandomState(0).choice(
            n, min(n, 500), replace=False
        )
    ]

    g = np.sqrt(sum(a ** 2 for a in np.gradient(
        x.astype(np.float64)
    )))[m]

    mu, sd = v.mean(), v.std()

    f = [
        mu, np.median(v), sd, v.var(), v.min(), v.max(),
        np.ptp(v), *np.percentile(v, [10, 90]), ent,
        vol, surf,
        np.pi ** (1 / 3) * (6 * vol) ** (2 / 3) / surf,
        vol / surf ** 1.5,
        np.sqrt(ev[1] / ev[0]),
        np.sqrt(ev[2] / ev[0]),
        pdist(sub).max(),
        surf / vol,
        skew(v), kurtosis(v), (v ** 2).sum(), (p ** 2).sum(),
        np.sqrt((v ** 2).mean()), np.abs(v - mu).mean(),
        g.mean(), g.std(),
        np.corrcoef(v[:-1], v[1:])[0, 1],
        sd / (abs(mu) + 1e-8)
    ]

    return np.nan_to_num(np.asarray(f, dtype=np.float32))

# ------------------------------------------------------------
# 4. Diameter-independent adaptation of the existing auto-mask
# ------------------------------------------------------------
def auto_mask_lungx(hu):
    # Patch is already resampled to 1 mm isotropic spacing.
    zz, yy, xx = np.indices(hu.shape)

    r = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    fallback = r <= FALLBACK_RADIUS_MM

    labels, n = ndimage.label(
        (hu > -500) & (r <= SEARCH_RADIUS_MM)
    )

    if n == 0:
        return fallback, "fallback_no_components"

    central = labels[28:37, 28:37, 28:37]
    ids = central[central > 0]

    if len(ids) == 0:
        return fallback, "fallback_no_central_component"

    selected = int(np.bincount(ids).argmax())

    candidate = ndimage.binary_fill_holes(
        ndimage.binary_opening(
            labels == selected,
            iterations=1
        )
    )

    if candidate.sum() < 20:
        return fallback, "fallback_component_too_small"

    return candidate.astype(bool), "threshold_component"

# ------------------------------------------------------------
# 5. Read each scan once and create the patches
# ------------------------------------------------------------
def read_ct(series_dir):
    reader = sitk.ImageSeriesReader()
    filenames = reader.GetGDCMSeriesFileNames(str(series_dir))

    if not filenames:
        raise FileNotFoundError(
            f"No DICOM files found in {series_dir}"
        )

    reader.SetFileNames(filenames)
    return reader.Execute()

records = []
errors = []

for patient_id, group in manifest.groupby("patient_id", sort=True):
    series_dir = Path(str(group.iloc[0]["image_path"]))

    try:
        img = read_ct(series_dir)
        vol = sitk.GetArrayFromImage(img).astype(np.float32)
        spacing_zyx = np.asarray(img.GetSpacing(), dtype=float)[::-1]

        for row in group.itertuples(index=False):
            try:
                # SimpleITK index is (x,y,z); patch center must be (z,y,x).
                # Use the verified DICOM slice mapping instead of the physical-coordinate transform.
                z_idx = int(array_index_by_case[row.case_id])

                # LUNGx pixel coordinates are one-based; NumPy indices are zero-based.
                x_idx = float(row.pixel_x) - 1.0
                y_idx = float(row.pixel_y) - 1.0

                # Patch center must be in (z, y, x) order.
                center_zyx = np.asarray([z_idx, y_idx, x_idx], dtype=float)

                if not np.isfinite(center_zyx).all():
                    raise ValueError("Non-finite center index")

                if not (
                    0 <= center_zyx[0] < vol.shape[0]
                    and 0 <= center_zyx[1] < vol.shape[1]
                    and 0 <= center_zyx[2] < vol.shape[2]
                ):
                    raise ValueError(f"Center outside CT volume: {center_zyx}")

                dummy_mask = np.zeros(vol.shape, dtype=bool)
                hu, _ = make_patch(
                    vol, dummy_mask, center_zyx, spacing_zyx
                )

                mask, method = auto_mask_lungx(hu)
                if int(mask.sum()) < 20:
                    raise ValueError(
                        f"Generated mask unexpectedly small: {mask.sum()}"
                    )

                rad = radiomics28(hu, mask)
                if rad.shape != (28,) or not np.isfinite(rad).all():
                    raise ValueError("Invalid radiomics vector")

                p64 = (
                    (hu - HU_MIN) / (HU_MAX - HU_MIN)
                ).astype(np.float16)

                patch_path = PATCH_OUT / f"{row.case_id}.npz"

                np.savez_compressed(
                    patch_path,
                    p64=p64,
                    rad=rad,
                    mask=mask.astype(bool),
                    label=int(row.label),
                    patient=str(row.patient_id),
                    mask_method=method
                )

                records.append({
                    "case_id": row.case_id,
                    "patient_id": row.patient_id,
                    "label": int(row.label),
                    "patch_file": str(patch_path),
                    "mask_voxels": int(mask.sum()),
                    "mask_fraction": float(mask.mean()),
                    "mask_method": method,
                    "radiomics_features": int(len(rad))
                })

            except Exception as exc:
                errors.append({
                    "case_id": row.case_id,
                    "patient_id": patient_id,
                    "error": repr(exc)
                })

        del vol, img
        gc.collect()

    except Exception as exc:
        for row in group.itertuples(index=False):
            errors.append({
                "case_id": row.case_id,
                "patient_id": patient_id,
                "error": repr(exc)
            })

# ------------------------------------------------------------
# 6. Save an audit record and report quality checks
# ------------------------------------------------------------
summary = pd.DataFrame(records)
error_df = pd.DataFrame(errors)

summary.to_csv(OUT / "mask_summary.csv", index=False)
error_df.to_csv(OUT / "patch_generation_errors.csv", index=False)
manifest.to_csv(OUT / "manifest_used.csv", index=False)

statement = f"""
LUNGx patch-generation record
Reference manifest: {MANIFEST}
Output directory: {OUT}

LIDC development nodules used to define mask parameters: {len(dev)}
Development diameter median: {diam_median:.6f} mm
Development diameter 75th percentile: {diam_q75:.6f} mm
Fixed search radius: {SEARCH_RADIUS_MM:.6f} mm
Fixed fallback sphere radius: {FALLBACK_RADIUS_MM:.6f} mm

CT patch preprocessing follows the saved training make_patch convention:
HU clipping [{HU_MIN}, {HU_MAX}], 1-mm isotropic resampling, and 64x64x64 patches.

Automatic mask generation uses a fixed search radius and a fixed fallback
radius because individual LUNGx diameters are not supplied in the test manifest.
This is an external preprocessing adaptation, not identical to the diameter-
dependent LNDb implementation.

No LUNGx diagnoses were used to select mask parameters or train the model.
No model training or threshold tuning was performed.
"""
(OUT / "preprocessing_statement.txt").write_text(
    statement.strip() + "\n"
)

print("\n=== LUNGx patch-generation summary ===")
print("Expected nodules:", len(manifest))
print("Successfully generated:", len(summary))
print("Errors:", len(error_df))
print("Output directory:", OUT)

if len(summary):
    print("\nMask methods:")
    print(summary["mask_method"].value_counts().to_string())

    print("\nMask voxel distribution:")
    print(summary["mask_voxels"].describe(
        percentiles=[.1, .25, .5, .75, .9]
    ).to_string())

    print("\nSample cases:")
    print(summary.head(12).to_string(index=False))

if len(error_df):
    print("\nFirst errors:")
    print(error_df.head(10).to_string(index=False))

if len(summary) != 73 or len(error_df):
    print(
        "\nSTOP: not all 73 patches were generated successfully. "
        "Do not evaluate the model yet."
    )
else:
    print(
        "\nAll 73 patches generated. Review mask counts and fallback "
        "frequency before running model inference."
    )

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

ROOT = Path("/content/drive/MyDrive/external_cohorts/LUNGx")
MANIFEST = (
    ROOT / "labels" /
    "lungx_test_manifest_world_coordinates.csv"
)

df = pd.read_csv(MANIFEST)
df = df[df["patient_id"] == "LUNGx-CT017"]

if len(df) != 2:
    raise RuntimeError(
        f"Expected two CT017 nodules, found {len(df)}"
    )

for _, row in df.iterrows():
    print("\n" + "=" * 65)
    print("Case:", row["case_id"])

    series_dir = Path(row["image_path"])
    dicom_path = Path(row["dicom_file"])

    if not series_dir.is_dir():
        raise FileNotFoundError(series_dir)
    if not dicom_path.is_file():
        raise FileNotFoundError(dicom_path)

    # Header from the exact DICOM file used to derive the coordinate.
    ds = pydicom.dcmread(dicom_path, stop_before_pixels=True)

    ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
    iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
    spacing = np.asarray(ds.PixelSpacing, dtype=float)

    col = float(row["pixel_x"]) - 1.0
    r = float(row["pixel_y"]) - 1.0

    world_from_header = (
        ipp
        + col * spacing[1] * iop[:3]
        + r * spacing[0] * iop[3:]
    )

    world_saved = np.array([
        row["world_x"], row["world_y"], row["world_z"]
    ], dtype=float)

    print("Series directory:", series_dir)
    print("Selected DICOM:", dicom_path.name)
    print("InstanceNumber:", getattr(ds, "InstanceNumber", None))
    print("Pixel x,y:", row["pixel_x"], row["pixel_y"])
    print("DICOM rows, columns:", ds.Rows, ds.Columns)
    print("ImagePositionPatient:", ipp)
    print("ImageOrientationPatient:", iop)
    print("PixelSpacing:", spacing)
    print("SeriesInstanceUID:", ds.SeriesInstanceUID)

    print("\nSaved manifest world XYZ:", world_saved)
    print("Recomputed world XYZ:", world_from_header)
    print(
        "Coordinate difference (mm):",
        float(np.linalg.norm(world_saved - world_from_header))
    )

    # Enumerate the available DICOM series in this directory.
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(series_dir))

    print("\nSimpleITK series IDs:", series_ids)
    print("Expected SeriesInstanceUID:", str(ds.SeriesInstanceUID))

    if not series_ids or str(ds.SeriesInstanceUID) not in series_ids:
        print("ERROR: selected DICOM series is not listed by SimpleITK.")
        continue

    filenames = reader.GetGDCMSeriesFileNames(
        str(series_dir), str(ds.SeriesInstanceUID)
    )

    resolved_selected = dicom_path.resolve()
    resolved_series_files = {
        Path(f).resolve() for f in filenames
    }

    print("Files in selected series:", len(filenames))
    print(
        "Selected DICOM included in SimpleITK series:",
        resolved_selected in resolved_series_files
    )

    reader.SetFileNames(filenames)
    image = reader.Execute()

    index = image.TransformPhysicalPointToContinuousIndex(
        tuple(world_from_header.tolist())
    )
    index_ipp = image.TransformPhysicalPointToContinuousIndex(
        tuple(ipp.tolist())
    )

    print("\nSimpleITK volume size (x,y,z):", image.GetSize())
    print("Volume origin:", image.GetOrigin())
    print("Volume spacing:", image.GetSpacing())
    print("Volume direction:", image.GetDirection())
    print("Continuous index of nodule center (x,y,z):", index)
    print("Continuous index of selected slice origin:", index_ipp)

    size = np.asarray(image.GetSize(), dtype=float)
    in_bounds = all(
        -0.5 <= index[i] < size[i] - 0.5
        for i in range(3)
    )
    print("Center within volume bounds:", in_bounds)

    del image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

patch_file = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/external_validation_fresh/"
    "LUNGx_20261021_132921/patches/LUNGx-CT017_N02.npz"
)

with np.load(patch_file) as data:
    print("Available arrays:", data.files)

    img = data["p64"].astype(np.float32)
    mask = data["mask"].astype(bool)

print("\nPatch diagnostics")
print("Shape:", img.shape)
print("Minimum:", np.nanmin(img))
print("Maximum:", np.nanmax(img))
print("Mean:", np.nanmean(img))
print("Standard deviation:", np.nanstd(img))
print("Percentiles (1, 10, 50, 90, 99):",
      np.nanpercentile(img, [1, 10, 50, 90, 99]))
print("Unique values (first 20):", np.unique(img)[:20])
print("Mask voxels:", mask.sum())
print("Image contains NaN:", np.isnan(img).any())

# Display using the patch's own intensity range.
z, y, x = np.round(np.argwhere(mask).mean(axis=0)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

views = [
    (img[z, :, :], mask[z, :, :], f"Axial z={z}"),
    (img[:, y, :], mask[:, y, :], f"Coronal y={y}"),
    (img[:, :, x], mask[:, :, x], f"Sagittal x={x}"),
]

vmin, vmax = np.nanpercentile(img, [1, 99])

for ax, (image_slice, mask_slice, title) in zip(axes, views):
    ax.imshow(image_slice, cmap="gray", vmin=vmin, vmax=vmax)

    if mask_slice.any():
        ax.contour(mask_slice, levels=[0.5],
                   colors="red", linewidths=1)

    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

patch_file = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/external_validation_fresh/"
    "LUNGx_20261021_132921/patches/LUNGx-CT017_N02.npz"
)

with np.load(patch_file) as data:
    img = data["p64"].astype(np.float32)
    mask = data["mask"].astype(bool)

# The patch is ordered (z, y, x); its intended center is approximately (32, 32, 32).
c = 32

views = [
    (img[c, :, :], mask[c, :, :], (c, c), "Axial z=32"),
    (img[:, c, :], mask[:, c, :], (c, c), "Coronal y=32"),
    (img[:, :, c], mask[:, :, c], (c, c), "Sagittal x=32"),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (image_slice, mask_slice, center, title) in zip(axes, views):
    ax.imshow(image_slice, cmap="gray", vmin=0, vmax=1)
    if mask_slice.any():
        ax.contour(mask_slice, levels=[0.5], colors="red", linewidths=1)
    ax.plot(center[1], center[0], marker="+", color="yellow", markersize=12)
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(f"{patch_file.stem} — red: mask, yellow: patch center")
plt.tight_layout()
plt.show()

print("Mask voxels:", int(mask.sum()))
print("Mask contains patch center:", bool(mask[c, c, c]))
print("Center intensity:", float(img[c, c, c]))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

ROOT = Path("/content/drive/MyDrive/external_cohorts/LUNGx")
MANIFEST = ROOT / "labels" / "lungx_test_manifest_world_coordinates.csv"

df = pd.read_csv(MANIFEST)
results = []
errors = []

header_tags = [
    "InstanceNumber", "ImagePositionPatient",
    "ImageOrientationPatient", "PixelSpacing",
    "Rows", "Columns", "SeriesInstanceUID"
]

def read_header(path):
    return pydicom.dcmread(
        str(path),
        stop_before_pixels=True,
        specific_tags=header_tags
    )

for patient_id, group in df.groupby("patient_id", sort=True):
    series_dir = Path(group.iloc[0]["image_path"])

    try:
        series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(
            str(series_dir)
        )
        expected_uid = str(group.iloc[0]["series_instance_uid"])

        if not series_ids or expected_uid not in series_ids:
            raise RuntimeError(
                f"Expected series {expected_uid} not found: {series_ids}"
            )

        files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(
            str(series_dir), expected_uid
        )
        file_index = {
            Path(f).resolve(): i for i, f in enumerate(files)
        }

        for row in group.itertuples(index=False):
            target = Path(row.dicom_file).resolve()

            if target not in file_index:
                errors.append({
                    "case_id": row.case_id,
                    "error": "Target DICOM not present in GDCM sequence"
                })
                continue

            k = file_index[target]
            ds = read_header(target)

            issues = []

            if int(ds.InstanceNumber) != int(row.instance_number):
                issues.append("InstanceNumber mismatch")

            if str(ds.SeriesInstanceUID) != expected_uid:
                issues.append("SeriesInstanceUID mismatch")

            x, y = float(row.pixel_x), float(row.pixel_y)
            if not (1 <= x <= int(ds.Columns) and
                    1 <= y <= int(ds.Rows)):
                issues.append("Pixel coordinate out of bounds")

            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            normal = np.cross(iop[:3], iop[3:])
            normal /= np.linalg.norm(normal)

            def projection(path):
                h = read_header(path)
                pos = np.asarray(h.ImagePositionPatient, dtype=float)
                return float(pos @ normal)

            current = projection(target)
            prev_step = np.nan
            next_step = np.nan

            if k > 0:
                prev_pos = projection(files[k - 1])
                prev_step = current - prev_pos

            if k + 1 < len(files):
                next_pos = projection(files[k + 1])
                next_step = next_pos - current

            # Check local order and uniform spacing near the target.
            for step in (prev_step, next_step):
                if np.isfinite(step) and step <= 0:
                    issues.append("Slice order does not increase physically")
                elif np.isfinite(step) and not np.isclose(
                    step, 1.0, atol=0.05
                ):
                    issues.append(f"Unexpected local slice spacing: {step:.4f} mm")

            if issues:
                errors.append({
                    "case_id": row.case_id,
                    "error": "; ".join(issues)
                })

            results.append({
                "case_id": row.case_id,
                "patient_id": patient_id,
                "array_slice_index": k,
                "instance_number": int(ds.InstanceNumber),
                "pixel_x": x,
                "pixel_y": y,
                "rows": int(ds.Rows),
                "columns": int(ds.Columns),
                "pixel_spacing": tuple(
                    float(v) for v in ds.PixelSpacing
                ),
                "previous_slice_step_mm": prev_step,
                "next_slice_step_mm": next_step,
                "local_spacing_ok": all(
                    not np.isfinite(s) or
                    np.isclose(s, 1.0, atol=0.05)
                    for s in (prev_step, next_step)
                )
            })

    except Exception as exc:
        errors.append({
            "case_id": patient_id,
            "error": repr(exc)
        })

audit = pd.DataFrame(results)
errors_df = pd.DataFrame(errors)

print("=== DICOM-to-array mapping audit ===")
print("Reference nodules:", len(df))
print("Successfully mapped:", len(audit))
print("Scans covered:", audit["patient_id"].nunique() if len(audit) else 0)
print("Errors:", len(errors_df))

if len(audit):
    print("\nUnique image dimensions:")
    print(audit[["rows", "columns"]].drop_duplicates().to_string(index=False))

    print("\nUnique pixel spacings:")
    print(audit["pixel_spacing"].value_counts().to_string())

    print("\nArray-slice index range:")
    print(audit["array_slice_index"].describe().to_string())

    print("\nLocal spacing checks passed:",
          int(audit["local_spacing_ok"].sum()), "/", len(audit))

if len(errors_df):
    print("\nFirst errors:")
    print(errors_df.head(20).to_string(index=False))

# Save only this audit report to a new file.
audit_path = ROOT / "labels" / "lungx_dicom_array_mapping_audit.csv"
audit.to_csv(audit_path, index=False)
print("\nAudit report saved:", audit_path)

In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom

from scipy import ndimage
from scipy.stats import skew, kurtosis
from scipy.spatial.distance import pdist

# ============================================================
# 1. Paths and fixed parameters from LIDC development data
# ============================================================
WORK = Path("/content/drive/MyDrive/lidc_work")
RUN = WORK / "full_rebuild" / "training_20261020_01"

LUNGX_ROOT = Path("/content/drive/MyDrive/external_cohorts/LUNGx")
MANIFEST_PATH = (
    LUNGX_ROOT / "labels" /
    "lungx_test_manifest_world_coordinates.csv"
)
AUDIT_PATH = (
    LUNGX_ROOT / "labels" /
    "lungx_dicom_array_mapping_audit.csv"
)

ELIGIBLE_PATH = WORK / "eligible_nodules.csv"
HOLDOUT_PATH = WORK / "full_rebuild" / "final_holdout_patients.csv"

for path in [
    MANIFEST_PATH, AUDIT_PATH, ELIGIBLE_PATH, HOLDOUT_PATH
]:
    if not path.exists():
        raise FileNotFoundError(f"Required file is missing: {path}")

# Use the same 363-nodule development split, excluding the actual
# 81-nodule holdout. Do not derive mask parameters from LUNGx labels.
eligible = pd.read_csv(ELIGIBLE_PATH)
holdout_df = pd.read_csv(HOLDOUT_PATH)

eligible["patient"] = eligible["patient"].astype(str).str.strip()
holdout_ids = set(holdout_df["patient"].astype(str).str.strip())

dev = eligible.loc[
    ~eligible["patient"].isin(holdout_ids)
].copy()
dev["diam"] = pd.to_numeric(dev["diam"], errors="coerce")
dev = dev.loc[dev["diam"].notna() & (dev["diam"] > 0)]

if len(dev) != 363:
    raise RuntimeError(f"Expected 363 development nodules; found {len(dev)}")

HU_MIN, HU_MAX, PATCH = -1000, 400, 64

diam_median = float(dev["diam"].median())
diam_q75 = float(dev["diam"].quantile(0.75))

SEARCH_RADIUS_MM = max(diam_q75 * 0.75 + 1.0, 4.0)
FALLBACK_RADIUS_MM = max(diam_median / 2.0, 2.5)

print(f"Development nodules: {len(dev)}")
print(f"Development median diameter: {diam_median:.4f} mm")
print(f"Development 75th percentile: {diam_q75:.4f} mm")
print(f"Fixed search radius: {SEARCH_RADIUS_MM:.4f} mm")
print(f"Fixed fallback radius: {FALLBACK_RADIUS_MM:.4f} mm")

# All output goes into a separate new folder.
OUT = (
    RUN / "external_validation_fresh" /
    ("LUNGx_corrected_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
)
PATCH_OUT = OUT / "patches"
PATCH_OUT.mkdir(parents=True, exist_ok=False)

# ============================================================
# 2. Validate manifest and DICOM-array mapping
# ============================================================
manifest = pd.read_csv(MANIFEST_PATH)
audit = pd.read_csv(AUDIT_PATH)

if len(manifest) != 73 or manifest["case_id"].duplicated().any():
    raise RuntimeError("Manifest must have 73 unique nodule IDs.")

if len(audit) != 73 or audit["case_id"].duplicated().any():
    raise RuntimeError("Mapping audit must contain 73 unique nodule IDs.")

if set(manifest["case_id"]) != set(audit["case_id"]):
    raise RuntimeError("Manifest and mapping-audit case IDs do not match.")

if manifest["label"].value_counts().to_dict() != {0: 37, 1: 36}:
    raise RuntimeError("Unexpected LUNGx reference-label counts.")

if not audit["local_spacing_ok"].all():
    raise RuntimeError("Not all target slices passed the local-spacing audit.")

array_index_by_case = audit.set_index("case_id")["array_slice_index"].to_dict()

# ============================================================
# 3. Exact LIDC patch preprocessing
# ============================================================
def crop(v, c, half, fill):
    lo = np.round(c).astype(int) - half
    out = np.full(2 * half, fill, dtype=v.dtype)
    s = np.maximum(lo, 0)
    e = np.minimum(lo + 2 * half, v.shape)

    if (e > s).all():
        out[
            tuple(slice(a, b) for a, b in zip(s - lo, e - lo))
        ] = v[tuple(slice(a, b) for a, b in zip(s, e))]

    return out

def fit(a, n, fill):
    if a.shape == (n, n, n):
        return a
    return crop(
        a, np.array(a.shape) / 2,
        np.full(3, n // 2), fill
    )

def make_patch(vol, mask, centre, spacing_zyx):
    half = np.ceil(
        (PATCH / 2 + 4) / np.asarray(spacing_zyx)
    ).astype(int)

    v = crop(vol, centre, half, HU_MIN).astype(np.float32)
    m = crop(mask.astype(np.uint8), centre, half, 0)

    zoom = np.asarray(spacing_zyx, dtype=float) / 1.0
    v = ndimage.zoom(v, zoom, order=1)
    m = ndimage.zoom(m, zoom, order=0)

    v = fit(np.clip(v, HU_MIN, HU_MAX), PATCH, HU_MIN)
    m = fit(m, PATCH, 0).astype(bool)

    return v, m

def radiomics28(x, m):
    v = x[m].astype(np.float64)
    n = len(v)

    if n < 10:
        raise ValueError(f"Mask too small for radiomics: {n} voxels")

    h, _ = np.histogram(v, bins=64)
    p = h[h > 0] / h.sum()
    ent = -(p * np.log2(p)).sum()

    vol = float(n)
    surf = float(n - ndimage.binary_erosion(m).sum()) + 1e-8

    co = np.argwhere(m).astype(float)
    co -= co.mean(0)

    ev = np.clip(
        np.sort(np.linalg.eigvalsh(
            np.cov(co.T) + 1e-6 * np.eye(3)
        ))[::-1],
        1e-8, None
    )

    sub = co[
        np.random.RandomState(0).choice(
            n, min(n, 500), replace=False
        )
    ]

    g = np.sqrt(sum(
        a ** 2 for a in np.gradient(x.astype(np.float64))
    ))[m]

    mu, sd = v.mean(), v.std()

    features = [
        mu, np.median(v), sd, v.var(), v.min(), v.max(),
        np.ptp(v), *np.percentile(v, [10, 90]), ent,
        vol, surf,
        np.pi ** (1 / 3) * (6 * vol) ** (2 / 3) / surf,
        vol / surf ** 1.5,
        np.sqrt(ev[1] / ev[0]),
        np.sqrt(ev[2] / ev[0]),
        pdist(sub).max(), surf / vol,
        skew(v), kurtosis(v), (v ** 2).sum(), (p ** 2).sum(),
        np.sqrt((v ** 2).mean()), np.abs(v - mu).mean(),
        g.mean(), g.std(),
        np.corrcoef(v[:-1], v[1:])[0, 1],
        sd / (abs(mu) + 1e-8)
    ]

    return np.nan_to_num(np.asarray(features, dtype=np.float32))

# Keep the chosen development-derived mask parameters fixed.
def auto_mask_lungx(hu):
    zz, yy, xx = np.indices(hu.shape)
    r = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    fallback = r <= FALLBACK_RADIUS_MM

    lab, n = ndimage.label(
        (hu > -500) & (r <= SEARCH_RADIUS_MM)
    )

    if n == 0:
        return fallback, "fallback_no_components"

    central = lab[28:37, 28:37, 28:37]
    ids = central[central > 0]

    if len(ids) == 0:
        return fallback, "fallback_no_central_component"

    selected = int(np.bincount(ids).argmax())
    candidate = ndimage.binary_fill_holes(
        ndimage.binary_opening(
            lab == selected, iterations=1
        )
    )

    if candidate.sum() < 20:
        return fallback, "fallback_component_too_small"

    return candidate.astype(bool), "threshold_component"

# ============================================================
# 4. Regenerate every patch using the DICOM file's array index
# ============================================================
def pixel_array_hu(ds):
    a = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return a * slope + intercept

records = []
errors = []

for patient_id, group in manifest.groupby("patient_id", sort=True):
    series_dir = Path(str(group.iloc[0]["image_path"]))

    try:
        series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(
            str(series_dir)
        )
        expected_uid = str(group.iloc[0]["series_instance_uid"])

        if not series_ids or expected_uid not in series_ids:
            raise RuntimeError(
                f"{patient_id}: expected series not found"
            )

        reader = sitk.ImageSeriesReader()
        filenames = reader.GetGDCMSeriesFileNames(
            str(series_dir), expected_uid
        )
        file_index = {
            Path(f).resolve(): i for i, f in enumerate(filenames)
        }

        reader.SetFileNames(filenames)
        img = reader.Execute()
        vol = sitk.GetArrayFromImage(img).astype(np.float32)

        if vol.shape[0] != len(filenames):
            raise RuntimeError(
                f"{patient_id}: volume depth and file count differ"
            )

        for row in group.itertuples(index=False):
            try:
                target_file = Path(str(row.dicom_file)).resolve()

                if target_file not in file_index:
                    raise RuntimeError(
                        "Annotated DICOM file is absent from series stack"
                    )

                z_idx = int(file_index[target_file])
                audited_idx = int(array_index_by_case[row.case_id])

                if z_idx != audited_idx:
                    raise RuntimeError(
                        f"Slice mapping discrepancy: {z_idx} vs {audited_idx}"
                    )

                ds = pydicom.dcmread(str(target_file))

                if int(ds.InstanceNumber) != int(row.instance_number):
                    raise RuntimeError("DICOM InstanceNumber mismatch")

                if str(ds.SeriesInstanceUID) != expected_uid:
                    raise RuntimeError("DICOM SeriesInstanceUID mismatch")

                # Verify the selected DICOM slice equals the array plane.
                expected_plane = pixel_array_hu(ds)
                actual_plane = vol[z_idx]

                if expected_plane.shape != actual_plane.shape:
                    raise RuntimeError("DICOM and volume slice shapes differ")

                maxdiff = float(
                    np.max(np.abs(expected_plane - actual_plane))
                )
                if maxdiff > 1e-3:
                    raise RuntimeError(
                        f"Selected DICOM slice differs from volume "
                        f"plane; max abs error={maxdiff}"
                    )

                # LUNGx coordinates are one-based pixel coordinates:
                # x is column, y is row, z is the audited array slice index.
                x_idx = float(row.pixel_x) - 1.0
                y_idx = float(row.pixel_y) - 1.0
                centre_zyx = np.array(
                    [float(z_idx), y_idx, x_idx], dtype=float
                )

                if not (
                    0 <= x_idx < vol.shape[2]
                    and 0 <= y_idx < vol.shape[1]
                    and 0 <= z_idx < vol.shape[0]
                ):
                    raise RuntimeError(
                        f"Pixel center outside volume: {centre_zyx}"
                    )

                # Axis spacing for numpy array axes (z,y,x).
                spacing_xyz = np.asarray(img.GetSpacing(), dtype=float)
                spacing_zyx = np.array([
                    spacing_xyz[2],
                    float(ds.PixelSpacing[0]),
                    float(ds.PixelSpacing[1])
                ])

                dummy_mask = np.zeros(vol.shape, dtype=bool)
                hu, _ = make_patch(
                    vol, dummy_mask, centre_zyx, spacing_zyx
                )

                if hu.shape != (64, 64, 64):
                    raise RuntimeError(f"Wrong patch shape: {hu.shape}")

                mask, method = auto_mask_lungx(hu)
                if int(mask.sum()) < 20:
                    raise RuntimeError(
                        f"Generated mask too small: {mask.sum()} voxels"
                    )

                rad = radiomics28(hu, mask)
                if rad.shape != (28,) or not np.isfinite(rad).all():
                    raise RuntimeError("Invalid 28-feature vector")

                p64 = (
                    (hu - HU_MIN) / (HU_MAX - HU_MIN)
                ).astype(np.float16)

                patch_path = PATCH_OUT / f"{row.case_id}.npz"

                np.savez_compressed(
                    patch_path,
                    p64=p64,
                    rad=rad,
                    mask=mask.astype(bool),
                    label=int(row.label),
                    patient=str(row.patient_id)
                )

                records.append({
                    "case_id": row.case_id,
                    "patient_id": row.patient_id,
                    "label": int(row.label),
                    "patch_file": str(patch_path),
                    "array_slice_index": z_idx,
                    "center_x_index": x_idx,
                    "center_y_index": y_idx,
                    "center_z_index": z_idx,
                    "selected_slice_max_abs_error": maxdiff,
                    "mask_voxels": int(mask.sum()),
                    "mask_method": method,
                    "radiomics_count": len(rad)
                })

            except Exception as exc:
                errors.append({
                    "case_id": row.case_id,
                    "patient_id": patient_id,
                    "error": repr(exc)
                })

        del vol, img
        gc.collect()

    except Exception as exc:
        for row in group.itertuples(index=False):
            if not any(e["case_id"] == row.case_id for e in errors):
                errors.append({
                    "case_id": row.case_id,
                    "patient_id": patient_id,
                    "error": repr(exc)
                })

results_df = pd.DataFrame(records)
errors_df = pd.DataFrame(errors)

results_df.to_csv(OUT / "mask_summary.csv", index=False)
errors_df.to_csv(OUT / "patch_generation_errors.csv", index=False)
manifest.to_csv(OUT / "manifest_used.csv", index=False)

print("\n=== CORRECTED LUNGx PATCH GENERATION ===")
print("Output directory:", OUT)
print("Expected cases:", len(manifest))
print("Generated patches:", len(results_df))
print("Errors:", len(errors_df))

if len(results_df):
    print("\nMask methods:")
    print(results_df["mask_method"].value_counts().to_string())

    print("\nMask voxel counts:")
    print(results_df["mask_voxels"].describe(
        percentiles=[.1, .25, .5, .75, .9]
    ).to_string())

    print("\nMaximum DICOM-to-volume slice difference:",
          results_df["selected_slice_max_abs_error"].max())

if len(errors_df):
    print("\nFirst errors:")
    print(errors_df.head(20).to_string(index=False))

if len(results_df) != 73 or len(errors_df) != 0:
    print("\nSTOP: do not evaluate the model until all cases are verified.")
else:
    print(
        "\nAll 73 patches generated with verified slice mapping. "
        "Review the mask summary before inference."
    )

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/external_validation_fresh/"
    "LUNGx_corrected_20261021_121055"
)
PATCH_DIR = OUT / "patches"
SUMMARY = pd.read_csv(OUT / "mask_summary.csv")

# Select representative cases: fallback, small, median, large,
# and both CT017 nodules.
chosen = []

fallback = SUMMARY[
    SUMMARY["mask_method"].astype(str).str.startswith("fallback")
]
if len(fallback):
    chosen.append(fallback.iloc[0])

normal = SUMMARY[
    ~SUMMARY["mask_method"].astype(str).str.startswith("fallback")
].sort_values("mask_voxels")

if len(normal):
    chosen.extend([
        normal.iloc[0],
        normal.iloc[len(normal) // 2],
        normal.iloc[-1],
    ])

chosen.extend(
    SUMMARY[SUMMARY["patient_id"] == "LUNGx-CT017"].itertuples(index=False)
)

# Deduplicate by case ID and limit the preview size.
unique = {}
for item in chosen:
    case_id = (
        item["case_id"] if isinstance(item, pd.Series)
        else item.case_id
    )
    unique[case_id] = SUMMARY.loc[
        SUMMARY["case_id"] == case_id
    ].iloc[0]

chosen_df = pd.DataFrame(list(unique.values())).head(7)

fig, axes = plt.subplots(
    len(chosen_df), 3,
    figsize=(12, 3.2 * len(chosen_df)),
    squeeze=False
)

for row_idx, row in enumerate(chosen_df.itertuples(index=False)):
    path = PATCH_DIR / f"{row.case_id}.npz"

    with np.load(path, allow_pickle=False) as z:
        p = z["p64"].astype(np.float32)
        mask = z["mask"].astype(bool)

    # Restore HU values for display.
    hu = p * 1400.0 - 1000.0

    # Select orthogonal planes through the mask centroid.
    coords = np.argwhere(mask)
    if len(coords):
        cz, cy, cx = np.rint(coords.mean(axis=0)).astype(int)
    else:
        cz = cy = cx = 32

    views = [
        ("Axial", hu[cz], mask[cz]),
        ("Coronal", hu[:, cy, :], mask[:, cy, :]),
        ("Sagittal", hu[:, :, cx], mask[:, :, cx]),
    ]

    for col_idx, (view_name, image, mask_slice) in enumerate(views):
        ax = axes[row_idx, col_idx]
        ax.imshow(
            image, cmap="gray",
            vmin=-1000, vmax=400,
            origin="lower"
        )

        if mask_slice.any() and not mask_slice.all():
            ax.contour(
                mask_slice.astype(float),
                levels=[0.5],
                colors="red",
                linewidths=1.2
            )

        # Expected crop center, shown as a yellow cross.
        if col_idx == 0:
            center_x, center_y = 32, 32
        elif col_idx == 1:
            center_x, center_y = 32, 32  # x, z
        else:
            center_x, center_y = 32, 32  # y, z

        ax.plot(
            center_x, center_y,
            marker="+", color="yellow",
            markersize=10, markeredgewidth=1.5
        )

        ax.set_title(
            f"{row.case_id}\n{view_name} | "
            f"{row.mask_voxels} mask voxels"
        )
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, brier_score_loss

# ------------------------------------------------------------
# 1. Paths: use the fresh run, not the older frozen-model folder
# ------------------------------------------------------------
WORK = Path("/content/drive/MyDrive/lidc_work")
RUN = WORK / "full_rebuild" / "training_20261020_01"

MODEL_DIR = RUN / "models" / "dev" / "full" / "s42"
DEV_PRED = RUN / "preds" / "dev__full__s42.npz"

EXT_ROOT = Path(
    "/content/drive/MyDrive/external_cohorts/LUNGx"
)

MANIFEST = (
    EXT_ROOT / "labels" /
    "lungx_test_manifest_world_coordinates.csv"
)

PATCH_DIR = (
    RUN / "external_validation_fresh" /
    "LUNGx_20261021_132921" / "patches"
)

OUT = (
    RUN / "external_validation_fresh" /
    "LUNGx_20261021_132921" / "inference"
)

for p in [MODEL_DIR, DEV_PRED, MANIFEST, PATCH_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"Required path not found: {p}")

# ------------------------------------------------------------
# 2. Verify exact case-to-patch matching before inference
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST)
required = {"case_id", "patient_id", "label", "reference_type"}

if not required.issubset(df.columns):
    raise ValueError(f"Manifest missing columns: {required - set(df.columns)}")

df["case_id"] = df["case_id"].astype(str).str.strip()

if df["case_id"].duplicated().any():
    raise ValueError("Duplicate case_id values in manifest.")

if df["label"].isna().any():
    raise ValueError("Missing labels in manifest.")

y = df["label"].to_numpy(dtype=int)

if not set(np.unique(y)).issubset({0, 1}) or len(np.unique(y)) != 2:
    raise ValueError(f"Expected both binary labels 0/1; found {np.unique(y)}")

expected = {f"{case_id}.npz" for case_id in df["case_id"]}
actual = {p.name for p in PATCH_DIR.glob("*.npz")}

missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("Manifest cases:", len(df))
print("Patch files:", len(actual))
print("Unique patients:", df["patient_id"].nunique())
print("Label counts:", df["label"].value_counts().sort_index().to_dict())
print("Missing patches:", len(missing))
print("Unmatched patch files:", len(extra))

if missing or extra or len(df) != 73:
    raise RuntimeError(
        "Case-to-patch matching failed. Stopping before evaluation. "
        f"First missing={missing[:5]}, first extra={extra[:5]}"
    )

# Use the threshold chosen from fresh LIDC development OOF predictions.
with np.load(DEV_PRED, allow_pickle=False) as z:
    threshold = float(z["thr"])

print("Frozen LIDC development OOF threshold:", threshold)

# ------------------------------------------------------------
# 3. Recreate the exact full-model architecture
# ------------------------------------------------------------
# These are the architecture settings from the saved notebook.
CFG = {"FEAT_DIM": 128, "FC": (256, 128), "DROPOUT": (0.4, 0.3)}

class CBAM3D(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        h = max(c // r, 4)
        self.mlp = nn.Sequential(
            nn.Linear(c, h),
            nn.ReLU(True),
            nn.Linear(h, c)
        )
        self.conv = nn.Conv3d(2, 1, 7, padding=3)

    def forward(self, x):
        x = x * torch.sigmoid(
            self.mlp(x.mean((2, 3, 4))) +
            self.mlp(x.amax((2, 3, 4)))
        )[:, :, None, None, None]

        return x * torch.sigmoid(
            self.conv(torch.cat([
                x.mean(1, keepdim=True),
                x.amax(1, keepdim=True)
            ], 1))
        )

class Branch(nn.Module):
    def __init__(self, cbam, fd):
        super().__init__()
        layers = []
        for ci, co in [(1, 16), (16, 32), (32, 64)]:
            layers += [
                nn.Conv3d(ci, co, 3, padding=1),
                nn.BatchNorm3d(co),
                nn.ReLU(True),
                nn.MaxPool3d(2)
            ]

        self.conv = nn.Sequential(*layers)
        self.att = CBAM3D(64) if cbam == "branch" else nn.Identity()
        self.fc = nn.Linear(64, fd)

    def forward(self, x):
        x = self.att(self.conv(x))
        return self.fc(x.mean((2, 3, 4)))

class Net(nn.Module):
    def __init__(
        self,
        scales=(32, 48, 64),
        radiomics=True,
        cbam="branch",
        fd=None
    ):
        super().__init__()
        fd = fd or CFG["FEAT_DIM"]
        self.scales = tuple(scales)
        self.use_rad = radiomics
        self.b = nn.ModuleList([
            Branch(cbam, fd) for _ in self.scales
        ])

        dim = fd * len(self.scales) + (28 if radiomics else 0)
        self.gate = None

        h1, h2 = CFG["FC"]
        d1, d2 = CFG["DROPOUT"]

        self.head = nn.Sequential(
            nn.Linear(dim, h1),
            nn.ReLU(True),
            nn.Dropout(d1),
            nn.Linear(h1, h2),
            nn.ReLU(True),
            nn.Dropout(d2),
            nn.Linear(h2, 2)
        )

    def forward(self, p32, p48, p64, rad):
        inputs = {32: p32, 48: p48, 64: p64}
        pieces = [
            branch(inputs[scale])
            for branch, scale in zip(self.b, self.scales)
        ]

        if self.use_rad:
            pieces.append(rad)

        return self.head(torch.cat(pieces, 1))

# ------------------------------------------------------------
# 4. Load fresh fold models and their own saved scalers
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Inference device:", device)

models = []
hash_rows = []

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for fold in range(3):
    ckpt = MODEL_DIR / f"fold{fold}.pt"
    scaler_path = MODEL_DIR / f"scaler_fold{fold}.pkl"

    if not ckpt.exists() or not scaler_path.exists():
        raise FileNotFoundError(
            f"Missing fold checkpoint or scaler for fold {fold}"
        )

    try:
        state = torch.load(ckpt, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(ckpt, map_location="cpu")

    model = Net(
        scales=(32, 48, 64),
        radiomics=True,
        cbam="branch"
    ).to(device)

    # Strict loading stops if the architecture and weights disagree.
    model.load_state_dict(state, strict=True)
    model.eval()

    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)

    models.append((model, scaler))

    hash_rows.append({
        "file": str(ckpt),
        "sha256": sha256_file(ckpt)
    })
    hash_rows.append({
        "file": str(scaler_path),
        "sha256": sha256_file(scaler_path)
    })

print("Loaded and verified all three fold models.")

# ------------------------------------------------------------
# 5. Predict in batches; no augmentation or training
# ------------------------------------------------------------
fold_probs = np.zeros((3, len(df)), dtype=np.float64)
batch_size = 16

with torch.inference_mode():
    for fold, (model, scaler) in enumerate(models):
        print(f"Predicting with fold {fold}...")

        for start in range(0, len(df), batch_size):
            end = min(start + batch_size, len(df))
            patches = []
            radiomics = []

            for case_id in df["case_id"].iloc[start:end]:
                path = PATCH_DIR / f"{case_id}.npz"

                with np.load(path, allow_pickle=False) as data:
                    p = data["p64"].astype(np.float32)
                    r = data["rad"].astype(np.float32)

                if p.shape != (64, 64, 64):
                    raise ValueError(f"Unexpected patch shape in {path}: {p.shape}")
                if r.shape != (28,):
                    raise ValueError(f"Unexpected radiomics shape in {path}: {r.shape}")
                if not np.isfinite(p).all() or not np.isfinite(r).all():
                    raise ValueError(f"Non-finite values in {path}")

                patches.append(p)
                radiomics.append(r)

            P = np.stack(patches)
            R = np.stack(radiomics)

            # Apply the matching fold's training-fitted scaler.
            R = np.clip(scaler.transform(R), -5, 5).astype(np.float32)

            x = torch.from_numpy(P).to(device)[:, None]
            r = torch.from_numpy(R).to(device)

            logits = model(
                x[:, :, 16:48, 16:48, 16:48],
                x[:, :, 8:56, 8:56, 8:56],
                x,
                r
            )

            fold_probs[fold, start:end] = (
                F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            )

# Match the ensemble method: average the three fold probabilities.
prob = fold_probs.mean(axis=0)
pred = (prob >= threshold).astype(int)

# ------------------------------------------------------------
# 6. Calculate external metrics at the frozen threshold
# ------------------------------------------------------------
tn = int(((y == 0) & (pred == 0)).sum())
fp = int(((y == 0) & (pred == 1)).sum())
fn = int(((y == 1) & (pred == 0)).sum())
tp = int(((y == 1) & (pred == 1)).sum())

sensitivity = tp / (tp + fn) if tp + fn else float("nan")
specificity = tn / (tn + fp) if tn + fp else float("nan")
precision = tp / (tp + fp) if tp + fp else float("nan")
accuracy = (tp + tn) / len(y)
f1 = (
    2 * precision * sensitivity / (precision + sensitivity)
    if np.isfinite(precision) and np.isfinite(sensitivity)
    and precision + sensitivity > 0
    else 0.0
)
mcc_den = np.sqrt(
    (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
)
mcc = ((tp * tn - fp * fn) / mcc_den) if mcc_den else 0.0

metrics = {
    "n_nodules": len(y),
    "n_patients": int(df["patient_id"].nunique()),
    "n_reference_positive": int(y.sum()),
    "n_reference_negative": int((y == 0).sum()),
    "auc": float(roc_auc_score(y, prob)),
    "threshold_from_lidc_development_oof": threshold,
    "accuracy": accuracy,
    "sensitivity": sensitivity,
    "specificity": specificity,
    "precision": precision,
    "f1": f1,
    "mcc": mcc,
    "brier_raw_probabilities": float(brier_score_loss(y, prob)),
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
    "reference_type": "; ".join(sorted(
        df["reference_type"].dropna().astype(str).unique()
    ))
}

predictions = df.copy()
predictions["prob_fold0"] = fold_probs[0]
predictions["prob_fold1"] = fold_probs[1]
predictions["prob_fold2"] = fold_probs[2]
predictions["probability_mean"] = prob
predictions["prediction_at_frozen_threshold"] = pred

# ------------------------------------------------------------
# 7. Save to a new directory; never overwrite earlier reports
# ------------------------------------------------------------
OUT.mkdir(parents=True, exist_ok=False)

predictions.to_csv(OUT / "predictions.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUT / "metrics.csv", index=False)
pd.DataFrame(hash_rows).to_csv(OUT / "checkpoint_hashes.csv", index=False)

statement = f"""
Fresh checkpoint external evaluation: LNDb
Run directory: {RUN}
Patch directory: {PATCH_DIR}
Manifest: {MANIFEST}

Number of nodules: {len(df)}
Number of patients: {df['patient_id'].nunique()}
Reference labels: {metrics['reference_type']}

Model: fresh full model, three saved fold checkpoints, seed 42.
Inference: each fold checkpoint used with its corresponding saved scaler.
Ensemble: mean of the three fold positive-class probabilities.
Threshold: {threshold:.10f}, loaded from the fresh LIDC development OOF prediction artifact.
No retraining, calibration fitting, threshold selection, or tuning was performed on LNDb.
Predictions and metrics are saved separately from the previous LNDb reports.

Interpretation: evaluation against the LNDb manifest's radiologist-assessed
risk-proxy labels; these are not necessarily confirmed cancer diagnoses.
"""
(OUT / "evaluation_statement.txt").write_text(statement.strip() + "\n")

print("\n=== FRESH LNDb EVALUATION ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

print("\nResults saved to:", OUT)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
    confusion_matrix,
)

# --------------------------------------------------
# 1. Load saved LUNGx predictions
# --------------------------------------------------
OUT = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/external_validation_fresh/"
    "LUNGx_20261021_132921/inference"
)

df = pd.read_csv(OUT / "predictions.csv")

required = {
    "patient_id", "label", "probability_mean",
    "prediction_at_frozen_threshold"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if len(df) != 73 or df["patient_id"].nunique() != 60:
    raise ValueError(
        f"Unexpected cohort: {len(df)} nodules, "
        f"{df['patient_id'].nunique()} patients"
    )

y = df["label"].to_numpy(dtype=int)
prob = df["probability_mean"].to_numpy(dtype=float)
patient = df["patient_id"].astype(str).to_numpy()

THRESHOLD = 0.1934976875782013
pred = (prob >= THRESHOLD).astype(int)

# Verify saved classifications agree with the fixed threshold.
saved_pred = df["prediction_at_frozen_threshold"].to_numpy(dtype=int)
if not np.array_equal(pred, saved_pred):
    raise ValueError(
        "Saved predictions do not match the frozen threshold."
    )

print("Nodules:", len(df))
print("Patients:", len(np.unique(patient)))
print("Fixed threshold:", THRESHOLD)
print("Saved predictions verified.")

# --------------------------------------------------
# 2. Define evaluation metrics
# --------------------------------------------------
def calculate_metrics(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= THRESHOLD).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    auc = (
        roc_auc_score(y_true, probabilities)
        if len(np.unique(y_true)) == 2
        else np.nan
    )

    return {
        "ROC-AUC": auc,
        "Accuracy": accuracy_score(y_true, predictions),
        "Sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predictions),
        "Brier score": brier_score_loss(y_true, probabilities),
    }

point = calculate_metrics(y, prob)

# --------------------------------------------------
# 3. Patient-level cluster bootstrap
# --------------------------------------------------
N_BOOT = 2000
SEED = 42

rng = np.random.default_rng(SEED)
unique_patients = np.unique(patient)

# Store all nodules belonging to each patient.
patient_rows = {
    pid: np.flatnonzero(patient == pid)
    for pid in unique_patients
}

boot_results = []

for b in range(N_BOOT):
    # Sample patients with replacement.
    sampled_patients = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True
    )

    # Preserve all nodules from each sampled patient.
    # Repeated patients correctly duplicate their nodules.
    indices = np.concatenate([
        patient_rows[pid] for pid in sampled_patients
    ])

    boot_results.append(
        calculate_metrics(y[indices], prob[indices])
    )

boot_df = pd.DataFrame(boot_results)

# --------------------------------------------------
# 4. Calculate percentile 95% confidence intervals
# --------------------------------------------------
rows = []

for metric, estimate in point.items():
    values = boot_df[metric].dropna().to_numpy()

    lower, upper = np.percentile(values, [2.5, 97.5])

    rows.append({
        "metric": metric,
        "estimate": estimate,
        "ci_95_lower": lower,
        "ci_95_upper": upper,
        "valid_bootstrap_replicates": len(values),
    })

results = pd.DataFrame(rows)

results.to_csv(
    OUT / "metrics_patient_cluster_bootstrap_95CI.csv",
    index=False
)

print("\n=== LUNGx RESULTS: 95% PATIENT-LEVEL BOOTSTRAP CI ===")
print(
    results.to_string(
        index=False,
        formatters={
            "estimate": "{:.4f}".format,
            "ci_95_lower": "{:.4f}".format,
            "ci_95_upper": "{:.4f}".format,
        }
    )
)

print(
    "\nSaved to:",
    OUT / "metrics_patient_cluster_bootstrap_95CI.csv"
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# --------------------------------------------------
# Paths
# --------------------------------------------------
BASE = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/external_validation_fresh/"
    "LUNGx_20261021_132921"
)

OUT = BASE / "inference"
PATCH_DIR = BASE / "patches"

PRED_FILE = OUT / "predictions.csv"
THRESHOLD = 0.1934976875782013

df = pd.read_csv(PRED_FILE)

# --------------------------------------------------
# 1. Verify labels and saved predictions
# --------------------------------------------------
required = [
    "case_id", "patient_id", "label",
    "prob_fold0", "prob_fold1", "prob_fold2",
    "probability_mean",
    "prediction_at_frozen_threshold",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing prediction columns: {missing}")

if df["case_id"].duplicated().any():
    raise ValueError("Duplicate case IDs in predictions.csv")

df["label"] = df["label"].astype(int)
df["prediction"] = (
    df["probability_mean"] >= THRESHOLD
).astype(int)

if not np.array_equal(
    df["prediction"].to_numpy(),
    df["prediction_at_frozen_threshold"].astype(int).to_numpy(),
):
    raise ValueError("Stored predictions disagree with fixed threshold.")

df["error_type"] = np.select(
    [
        (df["label"] == 0) & (df["prediction"] == 1),
        (df["label"] == 1) & (df["prediction"] == 0),
    ],
    ["False positive", "False negative"],
    default="Correct",
)

# --------------------------------------------------
# 2. Compare the three fold probabilities
# --------------------------------------------------
fold_cols = ["prob_fold0", "prob_fold1", "prob_fold2"]

df["fold_probability_std"] = df[fold_cols].std(
    axis=1, ddof=0
)
df["fold_probability_range"] = (
    df[fold_cols].max(axis=1) -
    df[fold_cols].min(axis=1)
)

# --------------------------------------------------
# 3. Check mask size and method for each case
# --------------------------------------------------
mask_rows = []

for case_id in df["case_id"]:
    patch_file = PATCH_DIR / f"{case_id}.npz"

    if not patch_file.exists():
        mask_rows.append({
            "case_id": case_id,
            "mask_voxels": np.nan,
            "mask_method": "PATCH_MISSING",
        })
        continue

    with np.load(patch_file, allow_pickle=False) as data:
        mask = data["mask"].astype(bool)
        method = str(data["mask_method"].item())

    mask_rows.append({
        "case_id": case_id,
        "mask_voxels": int(mask.sum()),
        "mask_method": method,
    })

mask_df = pd.DataFrame(mask_rows)

df = df.merge(
    mask_df,
    on="case_id",
    how="left",
    validate="one_to_one",
)

# --------------------------------------------------
# 4. Save all cases and errors for review
# --------------------------------------------------
audit_cols = [
    "case_id", "patient_id", "label",
    "reference_diagnosis", "prediction", "error_type",
    "probability_mean",
    "prob_fold0", "prob_fold1", "prob_fold2",
    "fold_probability_std", "fold_probability_range",
    "mask_voxels", "mask_method",
]

audit = df[audit_cols].sort_values(
    ["error_type", "probability_mean"],
    ascending=[True, False],
)

audit.to_csv(OUT / "lungx_error_audit.csv", index=False)

errors = audit[audit["error_type"] != "Correct"].copy()
errors.to_csv(OUT / "lungx_misclassified_cases.csv", index=False)

# --------------------------------------------------
# 5. Print a concise diagnostic summary
# --------------------------------------------------
print("=== COHORT CHECK ===")
print("Nodules:", len(df))
print("Patients:", df["patient_id"].nunique())
print("\nClassification counts:")
print(df["error_type"].value_counts().to_string())

print("\n=== MASK METHOD COUNTS ===")
print(df["mask_method"].value_counts(dropna=False).to_string())

print("\n=== MASK VOXEL COUNTS BY ERROR TYPE ===")
print(
    df.groupby("error_type")["mask_voxels"]
      .agg(["count", "median", "min", "max"])
      .round(1)
      .to_string()
)

print("\n=== FOLD DISAGREEMENT ===")
print(
    df.groupby("error_type")[
        ["fold_probability_std", "fold_probability_range"]
    ].mean().round(4).to_string()
)

print("\n=== MISCLASSIFIED CASES ===")
print(
    errors[
        [
            "case_id", "label", "error_type",
            "probability_mean",
            "fold_probability_std",
            "mask_voxels", "mask_method",
        ]
    ].to_string(index=False)
)

print("\n=== SAVED FILES ===")
print(OUT / "lungx_error_audit.csv")
print(OUT / "lungx_misclassified_cases.csv")

In [ ]:
from pathlib import Path

# 1. LUNGx patches generated successfully
patch_dir = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/external_validation_fresh/"
    "LUNGx_20261021_132921/patches"
)

# 2. Fresh LIDC-trained model checkpoints and scalers
model_dir = Path(
    "/content/drive/MyDrive/lidc_work/full_rebuild/"
    "training_20261020_01/models/dev/full/s42"
)

# Check patch directory
print("=" * 55)
print("1. LUNGx PATCHES")
print("=" * 55)

if patch_dir.exists():
    patch_files = sorted(patch_dir.glob("*.npz"))
    print("Patch directory exists:", True)
    print("Number of patch files:", len(patch_files))
    print("Expected count: 73")
    print("PASS:", len(patch_files) == 73)
else:
    print("FAIL: Patch directory does not exist.")

# Check checkpoints and matching scalers
print("\n" + "=" * 55)
print("2. FRESH MODEL CHECKPOINTS AND SCALERS")
print("=" * 55)

if not model_dir.exists():
    print("FAIL: Model directory does not exist:", model_dir)
else:
    print("Model directory exists:", True)

    for fold in range(3):
        checkpoint = model_dir / f"fold{fold}.pt"
        scaler = model_dir / f"scaler_fold{fold}.pkl"

        print(f"\nFold {fold}")
        print("Checkpoint:", checkpoint.exists(), checkpoint.name)
        print("Scaler:    ", scaler.exists(), scaler.name)

# List actual files to help diagnose unexpected filenames
print("\n" + "=" * 55)
print("3. FILES FOUND IN MODEL DIRECTORY")
print("=" * 55)

if model_dir.exists():
    for f in sorted(model_dir.iterdir()):
        if f.is_file():
            print(f"{f.name} ({f.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import ndimage

WORK = Path("/content/drive/MyDrive/lidc_work")
PATCH_DIR = WORK / "patches_full_rebuild"
ELIGIBLE = WORK / "eligible_nodules.csv"
HOLDOUT = WORK / "full_rebuild" / "final_holdout_patients.csv"

RUN = WORK / "full_rebuild" / "training_20261020_01"
OUT = RUN / "external_validation_fresh" / "mask_method_development_sweep.csv"

# Correct development split: exclude the 81 held-out nodules by patient.
eligible = pd.read_csv(ELIGIBLE)
hold = pd.read_csv(HOLDOUT)

eligible["patient"] = eligible["patient"].astype(str).str.strip()
holdout_ids = set(hold["patient"].astype(str).str.strip())
dev = eligible.loc[~eligible["patient"].isin(holdout_ids)].copy()

if len(dev) != 363:
    raise RuntimeError(f"Expected 363 development nodules, found {len(dev)}")

# Fixed, predefined grid; no LUNGx labels are used.
SEARCH_RADII = [6.0, 8.0, 10.0, 12.0, 14.0, 17.4552]
FALLBACK_RADII = [3.5, 4.5, 5.0, 5.9069, 7.0]

# Build the development patch list from the original eligible-nodule table.
patch_paths = []
missing = []

for r in dev.itertuples(index=False):
    path = PATCH_DIR / f"{r.patient}_{str(r.series)[-8:]}_n{int(r.nod):02d}.npz"
    if path.exists():
        patch_paths.append(path)
    else:
        missing.append(str(path))

if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} development patches. Examples: {missing[:3]}"
    )

print("Development nodules:", len(dev))
print("Development patches found:", len(patch_paths))

# Euclidean distance from the center in a 64^3, 1-mm patch.
zz, yy, xx = np.indices((64, 64, 64))
radius_grid = np.sqrt(
    (zz - 32)**2 + (yy - 32)**2 + (xx - 32)**2
)

cases = []

for path in patch_paths:
    with np.load(path, allow_pickle=False) as z:
        p64 = z["p64"].astype(np.float32)
        reference = z["mask"].astype(bool)
        patient = str(np.asarray(z["patient"]).item()).strip()

    if patient in holdout_ids:
        raise RuntimeError(f"Holdout patient found in development patches: {patient}")

    if p64.shape != (64, 64, 64) or reference.shape != (64, 64, 64):
        raise ValueError(f"Unexpected patch or mask shape: {path.name}")

    # Reverse the training normalization to recover HU approximately.
    hu = p64 * 1400.0 - 1000.0

    reference_voxels = int(reference.sum())
    if reference_voxels == 0:
        raise ValueError(f"Empty reference mask: {path.name}")

    # Precompute the threshold component for each search radius.
    for search_radius in SEARCH_RADII:
        labels, n = ndimage.label(
            (hu > -500) & (radius_grid <= search_radius)
        )

        candidate = None
        method = None

        if n > 0:
            central = labels[28:37, 28:37, 28:37]
            ids = central[central > 0]

            if len(ids):
                selected = int(np.bincount(ids).argmax())
                cleaned = ndimage.binary_fill_holes(
                    ndimage.binary_opening(
                        labels == selected, iterations=1
                    )
                )

                if int(cleaned.sum()) >= 20:
                    candidate = cleaned.astype(bool)
                    method = "threshold_component"

        # Compare the same selected component across fallback radii.
        for fallback_radius in FALLBACK_RADII:
            if candidate is None:
                pred_mask = radius_grid <= fallback_radius
                used_fallback = True
                method_used = "fallback"
            else:
                pred_mask = candidate
                used_fallback = False
                method_used = method

            intersection = int(np.logical_and(reference, pred_mask).sum())
            denom = reference_voxels + int(pred_mask.sum())
            dice = 2.0 * intersection / denom if denom else np.nan

            cases.append({
                "case_id": path.stem,
                "search_radius_mm": search_radius,
                "fallback_radius_mm": fallback_radius,
                "dice": dice,
                "fallback_used": used_fallback,
                "mask_method": method_used,
                "reference_voxels": reference_voxels,
                "predicted_voxels": int(pred_mask.sum()),
            })

results = pd.DataFrame(cases)

summary = (
    results.groupby(
        ["search_radius_mm", "fallback_radius_mm"],
        as_index=False
    )
    .agg(
        cases=("case_id", "nunique"),
        mean_dice=("dice", "mean"),
        median_dice=("dice", "median"),
        dice_q25=("dice", lambda s: s.quantile(0.25)),
        dice_q75=("dice", lambda s: s.quantile(0.75)),
        fallback_rate=("fallback_used", "mean"),
        median_predicted_voxels=("predicted_voxels", "median"),
    )
    .sort_values(["median_dice", "mean_dice"], ascending=False)
)

OUT.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT, index=False)

print("\n=== Best configurations by development-mask overlap ===")
print(summary.head(12).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== Current configuration ===")
current = summary[
    np.isclose(summary["search_radius_mm"], 17.4552)
    & np.isclose(summary["fallback_radius_mm"], 5.9069)
]
print(current.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nComparison saved to:", OUT)

In [ ]:
# optional: paper-style 5-fold CV on the FULL cohort (internal; overlaps with the holdout -> NOT independent). Kept for continuity.
if CFG['RUN_FULL_COHORT_CV']:
    for v in ('full', 'img_3s_cbam', 'rad_only', 'size_only'):
        r = run_variant(v, PS, DF, None, tag='fullcv'); print(f"full-cohort CV {v}: AUC {roc_auc_score(DF.label.values, r['oof']):.4f}")
else:
    print('RUN_FULL_COHORT_CV=False (skipped).')

## 6. Statistical inference (reviewer item C)
Patient-cluster bootstrap CIs for **every** model; **paired DeLong** and **patient-bootstrap** tests for all comparisons; absolute AUC differences with CIs (effect sizes); Holm multiplicity correction within pre-declared families; multi-seed mean ± SD.

In [ ]:
# ---------------- statistics library (self-contained) ----------------
import numpy as np, pandas as pd
from scipy.stats import rankdata, norm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve

def fast_auc(y, p):
    y = np.asarray(y); p = np.asarray(p, float); n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return np.nan
    r = rankdata(p); return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1); i = j
    T2 = np.empty(N); T2[J] = T + 1; return T2

def delong_paired(y, p1, p2):
    # Sun & Xu (2014) fast DeLong, paired AUCs on the same cases. Returns auc1, auc2, diff, se, z, two-sided p.
    y = np.asarray(y).astype(int); order = np.argsort(-y); y = y[order]
    P = np.vstack([np.asarray(p1, float), np.asarray(p2, float)])[:, order]; m = int(y.sum()); n = len(y) - m
    tx = np.array([_midrank(P[k, :m]) for k in range(2)]); ty = np.array([_midrank(P[k, m:]) for k in range(2)]); tz = np.array([_midrank(P[k]) for k in range(2)])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]; diff = aucs[0] - aucs[1]
    if var <= 1e-15: return aucs[0], aucs[1], diff, 0.0, np.nan, (1.0 if abs(diff) < 1e-12 else np.nan)
    z = diff / np.sqrt(var); return aucs[0], aucs[1], diff, float(np.sqrt(var)), float(z), float(2 * norm.sf(abs(z)))

class ClusterBoot:
    # patient-level (cluster) bootstrap: resample PATIENTS with replacement, keep all their nodules
    def __init__(self, groups, n_boot=2000, seed=0):
        g = np.asarray(groups).astype(str); up = np.unique(g); by = {u: np.where(g == u)[0] for u in up}; rng = np.random.RandomState(seed)
        self.draws = [np.concatenate([by[u] for u in rng.choice(up, len(up))]) for _ in range(n_boot)]

def boot_vals(cb, fn):
    v = np.array([fn(i) for i in cb.draws], float); return v[np.isfinite(v)]

def auc_ci(y, p, cb):
    y = np.asarray(y); p = np.asarray(p); b = boot_vals(cb, lambda i: fast_auc(y[i], p[i])); lo, hi = np.percentile(b, [2.5, 97.5]); return fast_auc(y, p), lo, hi

def auc_diff_boot(y, pa, pb, cb):
    y = np.asarray(y); pa = np.asarray(pa); pb = np.asarray(pb)
    b = boot_vals(cb, lambda i: fast_auc(y[i], pa[i]) - fast_auc(y[i], pb[i])); lo, hi = np.percentile(b, [2.5, 97.5])
    p = min(1.0, 2 * min((b <= 0).mean(), (b >= 0).mean())); return fast_auc(y, pa) - fast_auc(y, pb), lo, hi, max(p, 1.0 / len(b))

def mean_ci(vals, cb):
    vals = np.asarray(vals, float); b = boot_vals(cb, lambda i: np.nanmean(vals[i])); lo, hi = np.percentile(b, [2.5, 97.5]); return float(np.nanmean(vals)), lo, hi

def conf_metrics(y, p, thr):
    y = np.asarray(y); pr = (np.asarray(p) >= thr).astype(int)
    tp = int(((y == 1) & (pr == 1)).sum()); tn = int(((y == 0) & (pr == 0)).sum()); fp = int(((y == 0) & (pr == 1)).sum()); fn = int(((y == 1) & (pr == 0)).sum())
    d = lambda a, b: a / b if b else np.nan; den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return dict(acc=(tp + tn) / len(y), sens=d(tp, tp + fn), spec=d(tn, tn + fp), prec=d(tp, tp + fp), f1=d(2 * tp, 2 * tp + fp + fn),
                mcc=(tp * tn - fp * fn) / den if den > 0 else np.nan, tp=tp, tn=tn, fp=fp, fn=fn)

def metrics_ci(y, p, thr, cb):
    y = np.asarray(y); p = np.asarray(p); out = {}
    fns = {'auc': lambda i: fast_auc(y[i], p[i])}
    for k in ('acc', 'sens', 'spec', 'prec', 'f1', 'mcc'): fns[k] = (lambda kk: (lambda i: conf_metrics(y[i], p[i], thr)[kk]))(k)
    pt = conf_metrics(y, p, thr); pt['auc'] = fast_auc(y, p)
    for k, fn in fns.items():
        b = boot_vals(cb, fn); out[k] = (pt[k], *np.percentile(b, [2.5, 97.5]))
    return out, {k: pt[k] for k in ('tp', 'tn', 'fp', 'fn')}

def holm(pvals):
    p = np.nan_to_num(np.asarray(pvals, float), nan=1.0); n = len(p); adj = np.empty(n); run = 0.0
    for rank, i in enumerate(np.argsort(p)): run = max(run, (n - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def patient_level(y, p, groups):
    d = pd.DataFrame(dict(y=y, p=p, g=np.asarray(groups).astype(str))).groupby('g').agg(y=('y', 'max'), p=('p', 'max')).reset_index(); return d

def ece_score(y, p, n_bins=10):
    y = np.asarray(y); p = np.asarray(p); idx = np.clip(np.digitize(p, np.linspace(0, 1, n_bins + 1)) - 1, 0, n_bins - 1); e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)

def _logit(p): p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def cal_slope_intercept(y, p):
    lr = LogisticRegression(C=1e6, max_iter=2000).fit(_logit(p)[:, None], y); return float(lr.coef_[0, 0]), float(lr.intercept_[0])
def fit_platt(y, p): return LogisticRegression(C=1e6, max_iter=2000).fit(_logit(p)[:, None], y)
def apply_platt(m, p): return m.predict_proba(_logit(p)[:, None])[:, 1]

def decision_curve(y, p, grid=None):
    y = np.asarray(y); p = np.asarray(p); grid = np.linspace(0.01, 0.99, 99) if grid is None else grid; n = len(y); prev = y.mean(); rows = []
    for t in grid:
        pr = p >= t; tp = ((y == 1) & pr).sum(); fp = ((y == 0) & pr).sum(); w = t / (1 - t)
        rows.append((t, tp / n - fp / n * w, prev - (1 - prev) * w, 0.0))
    return pd.DataFrame(rows, columns=['threshold', 'net_benefit_model', 'net_benefit_treat_all', 'net_benefit_treat_none'])

def op_thresholds(y, p):
    # operating points chosen on DEVELOPMENT predictions only
    fpr, tpr, th = roc_curve(y, p); th = np.clip(th, 0, 1)
    ok_s = np.where(tpr >= 0.95)[0]; ok_t = np.where(fpr <= 0.05)[0]
    return dict(youden=float(th[np.argmax(tpr - fpr)]),
                screening_sens95=float(th[ok_s[0]]) if len(ok_s) else 0.0,      # highest threshold keeping dev sensitivity >= 95 %
                triage_spec95=float(th[ok_t[-1]]) if len(ok_t) else 1.0)        # lowest threshold keeping dev specificity >= 95 %
print('statistics library ready')

In [ ]:
cb_dev, cb_hold = ClusterBoot(DF_DEV.patient, CFG['N_BOOT'], 1), ClusterBoot(DF_HOLD.patient, CFG['N_BOOT'], 2)
DATASETS = {'dev_oof': (y_dev, cb_dev, 0), 'holdout': (y_hold, cb_hold, 1)}         # name -> (labels, bootstrap, index into pred_set tuple)

# ---- 6.1 95 % CI for EVERY model (primary seed) -------------------------------------------------------------
# Let's redefine metrics_ci locally to robustly handle empty or invalid bootstrap values
def safe_percentile(arr, q):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return (np.nan, np.nan)
    return tuple(np.percentile(arr, q))

def safe_metrics_ci(y, p, thr, cb):
    y = np.asarray(y); p = np.asarray(p); out = {}
    fns = {'auc': lambda i: fast_auc(y[i], p[i])}
    for k in ('acc', 'sens', 'spec', 'prec', 'f1', 'mcc'):
        fns[k] = (lambda kk: (lambda i: conf_metrics(y[i], p[i], thr)[kk]))(k)
    pt = conf_metrics(y, p, thr); pt['auc'] = fast_auc(y, p)
    for k, fn in fns.items():
        b = boot_vals(cb, fn)
        lo, hi = safe_percentile(b, [2.5, 97.5])
        out[k] = (pt[k], lo, hi)
    return out, {k: pt[k] for k in ('tp', 'tn', 'fp', 'fn')}

rows = []
for v in AVAILABLE:
    oof, hold = pred_set(v); thr = dev_thr(v)
    a_d = auc_ci(y_dev, oof, cb_dev); a_h = auc_ci(y_hold, hold, cb_hold) if len(hold) else (np.nan,) * 3
    m_h, cm = safe_metrics_ci(y_hold, hold, thr, cb_hold) if len(hold) else ({}, {})
    rows.append(dict(model=v, dev_auc=a_d[0], dev_lo=a_d[1], dev_hi=a_d[2], hold_auc=a_h[0], hold_lo=a_h[1], hold_hi=a_h[2], dev_thr=thr,
                     **{f'hold_{k}': m_h[k][0] for k in m_h if k != 'auc'}, hold_sens_lo=m_h['sens'][1] if m_h else np.nan, hold_sens_hi=m_h['sens'][2] if m_h else np.nan,
                     hold_spec_lo=m_h['spec'][1] if m_h else np.nan, hold_spec_hi=m_h['spec'][2] if m_h else np.nan, **{f'hold_{k}': c for k, c in cm.items()}))
CI_ALL = pd.DataFrame(rows); CI_ALL.to_csv(os.path.join(STAT_DIR, 'ci_every_model.csv'), index=False)
print(CI_ALL[['model', 'dev_auc', 'dev_lo', 'dev_hi', 'hold_auc', 'hold_lo', 'hold_hi', 'dev_thr', 'hold_sens', 'hold_spec']].round(4).to_string(index=False))

In [ ]:
# ---- 6.2 paired comparisons: DeLong + patient bootstrap, effect size, Holm within pre-declared families ----------
PAIRS = [('full', 'rad_only', 'primary', 'Does the CNN add anything beyond 28 radiomics features?'),
         ('full', 'img_3s_cbam', 'primary', 'Does radiomics add anything to the image model?'),
         ('full', 'size_only', 'secondary', 'Fusion vs size shortcut'),
         ('full', 'full_no_cbam', 'secondary', 'CBAM effect (with radiomics)'),
         ('img_3s_cbam', 'img_3s_nocbam', 'secondary', 'CBAM effect (image-only)'),
         ('img_3s_cbam', 'img_32', 'secondary', 'Multi-scale vs 32 only (image-only)'),
         ('img_3s_cbam', 'img_48', 'secondary', 'Multi-scale vs 48 only (image-only)'),
         ('img_3s_cbam', 'img_64', 'secondary', 'Multi-scale vs 64 only (image-only)'),
         ('img_3s_cbam', 'img_32_48', 'secondary', 'Three scales vs 32+48'),
         ('img_3s_cbam', 'img_32_64', 'secondary', 'Three scales vs 32+64'),
         ('img_3s_cbam', 'img_48_64', 'secondary', 'Three scales vs 48+64'),
         ('img_3s_cbam', 'rad_only', 'secondary', 'Image-only vs radiomics-only')]
PAIRS += [('full', v, 'ablation_vs_full', 'Ablation vs full') for v in AVAILABLE if v != 'full']
seen, PAIRS2 = set(), []
for a, b, fam, why in PAIRS:
    if a in AVAILABLE and b in AVAILABLE and (a, b) not in seen: seen.add((a, b)); PAIRS2.append((a, b, fam, why))

CMP = []
for mode in ('primary', 'seedavg'):
    for dname, (yy, cb, k) in DATASETS.items():
        for a, b, fam, why in PAIRS2:
            pa, pb = pred_set(a, mode)[k], pred_set(b, mode)[k]
            if len(pa) == 0 or len(pb) == 0: continue
            auc_a, auc_b, dd, se, z, p_dl = delong_paired(yy, pa, pb); d_b, lo, hi, p_b = auc_diff_boot(yy, pa, pb, cb)
            CMP.append(dict(mode=mode, dataset=dname, family=fam, A=a, B=b, question=why, auc_A=auc_a, auc_B=auc_b, delta_AUC=d_b, ci_lo=lo, ci_hi=hi, p_boot=p_b, p_delong=p_dl))
CMP = pd.DataFrame(CMP)
CMP['p_boot_holm'] = np.nan; CMP['p_delong_holm'] = np.nan
for _, idx in CMP.groupby(['mode', 'dataset', 'family']).groups.items():
    CMP.loc[idx, 'p_boot_holm'] = holm(CMP.loc[idx, 'p_boot'].values); CMP.loc[idx, 'p_delong_holm'] = holm(CMP.loc[idx, 'p_delong'].values)
CMP['significant_boot_holm'] = CMP.p_boot_holm < 0.05
CMP.to_csv(os.path.join(STAT_DIR, 'paired_comparisons.csv'), index=False)
show = CMP[(CMP.mode == 'primary') & (CMP.family.isin(['primary', 'secondary']))]
print(show[['dataset', 'family', 'A', 'B', 'auc_A', 'auc_B', 'delta_AUC', 'ci_lo', 'ci_hi', 'p_boot', 'p_boot_holm', 'p_delong', 'p_delong_holm']].round(4).to_string(index=False))
print('\nNote: DeLong assumes independent cases; nodules cluster within patients, so the patient-cluster bootstrap is the primary inference.')

prim = CMP[(CMP.mode == 'primary') & (CMP.family == 'primary') & (CMP.dataset == 'holdout')]
FUSION_SIG = {r.B: bool(r.p_boot_holm < 0.05 and r.delta_AUC > 0) for r in prim.itertuples()}
print('\nFused significantly better than (holdout, Holm-corrected bootstrap):', FUSION_SIG)
if not all(FUSION_SIG.values()) if FUSION_SIG else True:
    print('=> The fused model is NOT demonstrably better than all simpler comparators. Per the SAP: say so, and position the paper as a rigorous benchmark / evaluation study.')

In [ ]:
# ---- 6.3 multi-seed analysis: SD over TRAINING SEEDS (separate from fold-to-fold SD) ---------------------------------
rows = []
for v in AVAILABLE:
    if VARIANTS[v]['kind'] != 'cnn': continue
    rs = RUNS[v]; hold_auc = [fast_auc(y_hold, r['hold']) for r in rs.values()]; dev_auc = [fast_auc(y_dev, r['oof']) for r in rs.values()]
    fold_sd = np.mean([np.std(r['val_auc'], ddof=1) for r in rs.values()]) if CFG['N_SPLITS'] > 1 else np.nan
    rows.append(dict(model=v, n_seeds=len(rs), dev_auc_mean=np.mean(dev_auc), dev_auc_sd_seeds=np.std(dev_auc, ddof=1) if len(rs) > 1 else np.nan,
                     hold_auc_mean=np.mean(hold_auc), hold_auc_sd_seeds=np.std(hold_auc, ddof=1) if len(rs) > 1 else np.nan, val_auc_sd_folds=fold_sd,
                     hold_auc_seedavg_ensemble=fast_auc(y_hold, pred_set(v, 'seedavg')[1])))
SEEDS_TAB = pd.DataFrame(rows); SEEDS_TAB.to_csv(os.path.join(STAT_DIR, 'multi_seed_summary.csv'), index=False)
print(SEEDS_TAB.round(4).to_string(index=False))

## 7. Ablation table (reviewer item B) and size-shortcut analyses

In [ ]:
# ---- 7.4 the reviewer's ablation table, filled with real numbers --------------------------------------------------
full_rows = CMP[(CMP.mode == 'primary') & (CMP.A == 'full')]
rows = []
for label, v, purpose in ABLATION_TABLE:
    if v not in AVAILABLE: rows.append(dict(model=label, purpose=purpose, note='not run (RUN_ABLATION=False or missing)')); continue
    r = CI_ALL[CI_ALL.model == v].iloc[0]; s = SEEDS_TAB[SEEDS_TAB.model == v]
    ch = full_rows[(full_rows.B == v) & (full_rows.dataset == 'holdout')]; cd = full_rows[(full_rows.B == v) & (full_rows.dataset == 'dev_oof')]
    rows.append(dict(model=label, purpose=purpose, dev_oof_AUC=f"{r.dev_auc:.4f} [{r.dev_lo:.4f}, {r.dev_hi:.4f}]", holdout_AUC=f"{r.hold_auc:.4f} [{r.hold_lo:.4f}, {r.hold_hi:.4f}]",
                     holdout_AUC_seed_mean_sd=(f"{s.hold_auc_mean.iloc[0]:.4f} ± {s.hold_auc_sd_seeds.iloc[0]:.4f}" if len(s) else 'n/a (deterministic)'),
                     delta_vs_full_holdout=(f"{-ch.delta_AUC.iloc[0]:+.4f}" if len(ch) else ''), p_vs_full_holdout_holm=(f"{ch.p_boot_holm.iloc[0]:.3f}" if len(ch) else ''),
                     p_vs_full_dev_holm=(f"{cd.p_boot_holm.iloc[0]:.3f}" if len(cd) else '')))
ABL = pd.DataFrame(rows); ABL.to_csv(os.path.join(STAT_DIR, 'ablation_table.csv'), index=False)
pd.set_option('display.width', 250, 'display.max_colwidth', 60); print(ABL.to_string(index=False))
print('\nDesign note: scale rows are image-only with CBAM on (as in the full model) so that ONLY the scale set varies; delta_vs_full = AUC(variant) - AUC(full).')

In [ ]:
# ---- 7.1 size shortcut: (a) descriptive, (b) diameter-overlap analysis with CIs, (c) incremental value beyond size ---
dsz = lambda df: np.array([r[IM] for r in df.rad.values], float)            # radiomics max_diam (mm), same as the earlier experiment
DF_DEV['d_mm'], DF_HOLD['d_mm'] = dsz(DF_DEV), dsz(DF_HOLD)
print('Median max-diameter (mm) by class, development:'); print(DF_DEV.groupby('label').d_mm.describe()[['count', '50%', 'min', 'max']].round(2))

ben, mal = DF_DEV[DF_DEV.label == 0], DF_DEV[DF_DEV.label == 1]
lo_o, hi_o = max(ben.d_mm.min(), mal.d_mm.min()), min(ben.d_mm.max(), mal.d_mm.max())
print(f"\nOverlap band defined on DEVELOPMENT data only: {lo_o:.2f} - {hi_o:.2f} mm")
oof_full, hold_full = pred_set('full'); oof_img, hold_img = pred_set('img_3s_cbam')
SIZE_ROWS = []
for name, df_, yy, pf, pi, cb in (('dev_oof', DF_DEV, y_dev, oof_full, oof_img, cb_dev), ('holdout', DF_HOLD, y_hold, hold_full, hold_img, cb_hold)):
    m = ((df_.d_mm >= lo_o) & (df_.d_mm <= hi_o)).values
    if m.sum() < 10 or len(set(yy[m])) < 2: print(name, ': too few nodules in overlap band'); continue
    sub = ClusterBoot(df_.patient.values[m], CFG['N_BOOT'], 3)
    row = dict(dataset=name, n=int(m.sum()), n_benign=int((yy[m] == 0).sum()), n_malignant=int((yy[m] == 1).sum()))
    for lab, sc in (('diameter_only', df_.d_mm.values[m]), ('image_only', pi[m]), ('full_fusion', pf[m])):
        a, lo, hi = auc_ci(yy[m], sc, sub); row[lab] = f"{a:.4f} [{lo:.4f}, {hi:.4f}]"
    dd, dlo, dhi, pb = auc_diff_boot(yy[m], pf[m], df_.d_mm.values[m], sub); row['fusion_minus_diameter'] = f"{dd:+.4f} [{dlo:+.4f}, {dhi:+.4f}] p={pb:.4f}"
    SIZE_ROWS.append(row)
SIZE_TAB = pd.DataFrame(SIZE_ROWS); SIZE_TAB.to_csv(os.path.join(STAT_DIR, 'size_overlap_analysis.csv'), index=False); print(SIZE_TAB.T.to_string())

# equal-COUNT size bands inside the overlap region (avoids the earlier tiny / imbalanced equal-width bands)
m = ((DF_DEV.d_mm >= lo_o) & (DF_DEV.d_mm <= hi_o)).values; od = DF_DEV[m].copy(); od['p_full'], od['p_img'] = oof_full[m], oof_img[m]
if len(od) >= 40:
    od['band'] = pd.qcut(od.d_mm, 4, duplicates='drop')
    for b, g in od.groupby('band', observed=True):
        nb, nm = int((g.label == 0).sum()), int((g.label == 1).sum())
        if nb < 5 or nm < 5: print(f"{b}: SKIPPED (benign={nb}, malignant={nm})"); continue
        print(f"{str(b):22s} n={len(g):3d} ben={nb:3d} mal={nm:3d} | diam AUC {fast_auc(g.label, g.d_mm):.3f} | image {fast_auc(g.label, g.p_img):.3f} | full {fast_auc(g.label, g.p_full):.3f}")

# incremental value beyond size: likelihood-ratio test  y ~ log(diam)  vs  y ~ log(diam) + logit(p_full)   (dev OOF)
import statsmodels.api as sm
X0 = sm.add_constant(np.log(DF_DEV.d_mm.values)); X1 = np.column_stack([X0, _logit(oof_full)])
try:
    m0, m1 = sm.Logit(y_dev, X0).fit(disp=0), sm.Logit(y_dev, X1).fit(disp=0, maxiter=200); lr_stat = 2 * (m1.llf - m0.llf)
    from scipy.stats import chi2; print(f"\nIncremental value beyond log-diameter (LR test, dev OOF): chi2={lr_stat:.2f}, p={chi2.sf(lr_stat, 1):.3g}  (nodule-level; indicative only)")
except Exception as e: print('LR test skipped:', repr(e)[:80])
print('\nInterpretation to use in the paper: the overlap analysis shows size does NOT fully explain performance; it does not prove size-independent malignancy detection.')

In [ ]:
# ---- label-shuffle negative control (kept from the earlier notebook) -------------------------------------------------
if CFG['RUN_SHUFFLE']:
    DF_SHUF = DF_DEV.copy(); DF_SHUF['label'] = np.random.RandomState(CFG['SPLIT_SEED']).permutation(DF_SHUF.label.values)
    rs = run_variant('full', PS, DF_SHUF, DF_HOLD, tag='shuffle')          # holdout keeps its REAL labels
    print(f"Shuffled-label control | OOF AUC vs shuffled dev labels {fast_auc(DF_SHUF.label.values, rs['oof']):.4f} | holdout AUC vs real labels {fast_auc(y_hold, rs['hold']):.4f}  (both expected ~0.5)")
else: print('RUN_SHUFFLE=False (skipped).')

## 8. Calibration, clinical utility and threshold stability (reviewer item D)
Calibration curves, Brier score, ECE, calibration slope; Platt recalibration **fitted on development out-of-fold predictions only** and applied to the holdout; decision-curve (net-benefit) analysis; clinically motivated operating points (high-sensitivity *screening*, high-specificity *triage*) whose thresholds are chosen on the development set and then applied unchanged to the holdout; and the fold-to-fold threshold instability.

In [ ]:
CAL_MODELS = [v for v in ('full', 'img_3s_cbam', 'rad_only') if v in AVAILABLE]
rows = []
fig, axs = plt.subplots(2, len(CAL_MODELS), figsize=(5.2 * len(CAL_MODELS), 9), squeeze=False)
for j, v in enumerate(CAL_MODELS):
    oof, hold = pred_set(v); pl_ = fit_platt(y_dev, oof)
    for name, yy, pp in (('dev_oof', y_dev, oof), ('holdout', y_hold, hold)):
        sl, ic = cal_slope_intercept(yy, pp)
        rows.append(dict(model=v, dataset=name, calibration='raw', brier=brier_score_loss(yy, pp), ece=ece_score(yy, pp), cal_slope=sl, cal_intercept=ic))
        if name == 'holdout':
            pr = apply_platt(pl_, pp); sl2, ic2 = cal_slope_intercept(yy, pr)
            rows.append(dict(model=v, dataset=name, calibration='Platt (fit on dev OOF)', brier=brier_score_loss(yy, pr), ece=ece_score(yy, pr), cal_slope=sl2, cal_intercept=ic2))
    for r_, (name, yy, pp) in enumerate((('dev out-of-fold', y_dev, oof), ('holdout', y_hold, hold))):
        a = axs[r_, j]; nb = 8 if len(yy) > 150 else 5
        fr, mp = [], []
        edges = np.quantile(pp, np.linspace(0, 1, nb + 1)); idx = np.clip(np.searchsorted(edges, pp, side='right') - 1, 0, nb - 1)
        for b in range(nb):
            mk_ = idx == b
            if mk_.any(): fr.append(yy[mk_].mean()); mp.append(pp[mk_].mean())
        a.plot([0, 1], [0, 1], 'k--', lw=1); a.plot(mp, fr, 'o-', label='raw'); a.set_title(f"{v} | {name}\nBrier {brier_score_loss(yy, pp):.3f} | ECE {ece_score(yy, pp):.3f}")
        a.set_xlabel('mean predicted probability'); a.set_ylabel('observed malignant fraction'); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'calibration_curves.png'), dpi=200); plt.show()
CAL = pd.DataFrame(rows); CAL.to_csv(os.path.join(STAT_DIR, 'calibration_metrics.csv'), index=False); print(CAL.round(4).to_string(index=False))
print('\nReading: ensembles of overconfident CNNs usually have slope < 1; report calibration honestly and use the recalibrated probabilities (or decision curve) if clinical use is discussed.')

In [ ]:
# ---- decision-curve analysis (holdout) + operating points (thresholds from DEV only) ----------------------------------
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for j, (name, yy, k) in enumerate((('dev out-of-fold', y_dev, 0), ('holdout', y_hold, 1))):
    for v in CAL_MODELS:
        dc = decision_curve(yy, pred_set(v)[k]); axs[j].plot(dc.threshold, dc.net_benefit_model, label=v)
    axs[j].plot(dc.threshold, dc.net_benefit_treat_all, 'k:', label='treat all'); axs[j].axhline(0, color='gray', label='treat none')
    axs[j].set_ylim(-0.1, max(0.5, yy.mean() + 0.05)); axs[j].set_xlabel('threshold probability'); axs[j].set_ylabel('net benefit'); axs[j].set_title(f'Decision curve | {name}'); axs[j].legend(); axs[j].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'decision_curves.png'), dpi=200); plt.show()

rows = []
for v in CAL_MODELS:
    ops = op_thresholds(y_dev, pred_set(v)[0])
    for opn, t in ops.items():
        for name, yy, pp, cb in (('dev_oof', y_dev, pred_set(v)[0], cb_dev), ('holdout', y_hold, pred_set(v)[1], cb_hold)):
            m_, cm = metrics_ci(yy, pp, t, cb)
            rows.append(dict(model=v, operating_point=opn, threshold=t, dataset=name, sens=f"{m_['sens'][0]:.3f} [{m_['sens'][1]:.3f},{m_['sens'][2]:.3f}]",
                             spec=f"{m_['spec'][0]:.3f} [{m_['spec'][1]:.3f},{m_['spec'][2]:.3f}]", prec=f"{m_['prec'][0]:.3f}", **cm))
OPS = pd.DataFrame(rows); OPS.to_csv(os.path.join(STAT_DIR, 'operating_points.csv'), index=False); print(OPS.to_string(index=False))

# threshold stability: fold-wise validation thresholds + bootstrap spread of the development Youden threshold
thr_rows = []
for v in CAL_MODELS:
    ft = RUNS[v][PS if PS in RUNS[v] else sorted(RUNS[v])[0]]['fold_thr']; oof = pred_set(v)[0]
    bt = boot_vals(cb_dev, lambda i: youden_thr(y_dev[i], oof[i]))
    thr_rows.append(dict(model=v, fold_thr_min=ft.min(), fold_thr_max=ft.max(), fold_thr_sd=ft.std(ddof=1) if len(ft) > 1 else np.nan, dev_youden=youden_thr(y_dev, oof), boot_thr_2p5=np.percentile(bt, 2.5), boot_thr_97p5=np.percentile(bt, 97.5)))
THR_TAB = pd.DataFrame(thr_rows); THR_TAB.to_csv(os.path.join(STAT_DIR, 'threshold_stability.csv'), index=False); print('\n', THR_TAB.round(4).to_string(index=False))
print(f"\nDiscussion for the paper: the chosen threshold ({FINAL_THRESHOLD:.4f}) maximises Youden's J on development data; it is a statistical, not a clinical, optimum. A wide fold/bootstrap threshold spread means the operating point is not stable.\n"
      "For screening use the sens>=95% threshold; for triage the spec>=95% threshold; choose by clinical cost of a missed cancer vs. an unnecessary work-up (see decision curves).")

## 9. Reference standard (reviewer item E)
LIDC-IDRI labels are **radiologist malignancy ratings, not pathology**; nodules with intermediate scores are excluded. The task is therefore named *radiologist-assessed malignancy-risk classification* throughout. This section (a) tests robustness to the label definition, (b) evaluates a pathology / follow-up-confirmed subgroup *if* you supply the TCIA diagnosis table, and (c) reports patient-level performance.

In [ ]:
# ---- (a) label-robustness on the holdout (and dev OOF): stricter consensus subsets -------------------------------------
oof_full, hold_full = pred_set('full')
rows = []
for name, df_, yy, pp in (('dev_oof', DF_DEV, y_dev, oof_full), ('holdout', DF_HOLD, y_hold, hold_full)):
    for lab, m in (('all labelled nodules', np.ones(len(df_), bool)),
                   ('strict consensus (score <=1.5 or >=4.5)', ((df_.score <= 1.5) | (df_.score >= 4.5)).values),
                   ('borderline labels only (1.5<score<=2 or 4<=score<4.5)', (((df_.score > 1.5) & (df_.score <= 2)) | ((df_.score >= 4) & (df_.score < 4.5))).values),
                   ('>=4 annotators', (df_.n_ann >= 4).values), ('<=4 annotation cluster (excl. possible merged clusters)', (df_.n_ann <= 4).values)):
        if m.sum() < 8 or len(set(yy[m])) < 2: rows.append(dict(dataset=name, subset=lab, n=int(m.sum()), auc=np.nan)); continue
        rows.append(dict(dataset=name, subset=lab, n=int(m.sum()), n_malignant=int(yy[m].sum()), auc=fast_auc(yy[m], pp[m])))
LABROB = pd.DataFrame(rows); LABROB.to_csv(os.path.join(STAT_DIR, 'label_robustness.csv'), index=False); print(LABROB.round(4).to_string(index=False))

# ---- (b) pathology / follow-up subgroup (needs the TCIA LIDC diagnosis table, patient-level) ---------------------------
DIAG_CSV = os.path.join(CFG['WORK_DIR'], 'lidc_diagnosis_patient_level.csv')
print('\nTo enable the pathology-confirmed subgroup, create', DIAG_CSV, 'with columns:\n  patient (e.g. LIDC-IDRI-0001), diagnosis (benign|malignant), method (biopsy|surgery|followup|radiological_review)\n'
      '(from the TCIA "LIDC-IDRI patient diagnosis" spreadsheet: diagnosis 1 = benign/non-malignant, 2/3 = malignant; method 2 = biopsy, 3 = surgical resection, 4 = progression/response).')
if os.path.exists(DIAG_CSV):
    dg = pd.read_csv(DIAG_CSV); dg['patient'] = dg.patient.astype(str); dg = dg[dg.method.isin(['biopsy', 'surgery', 'followup'])]
    dg['dx'] = (dg.diagnosis.str.lower() == 'malignant').astype(int)
    rows = []
    for name, df_, yy, pp in (('dev_oof', DF_DEV, y_dev, oof_full), ('holdout', DF_HOLD, y_hold, hold_full)):
        cnt = df_.groupby('patient').size(); single = cnt[cnt == 1].index        # patient-level diagnosis maps unambiguously only if the patient has ONE eligible nodule
        sel = df_.patient.isin(single) & df_.patient.isin(dg.patient); sub = df_[sel].merge(dg[['patient', 'dx', 'method']], on='patient')
        if len(sub) < 6: rows.append(dict(dataset=name, n=len(sub), note='too few')); continue
        pp_s = pp[sel.values]; agree = float((sub.dx.values == sub.label.values).mean())
        rows.append(dict(dataset=name, n=len(sub), n_malignant_pathology=int(sub.dx.sum()), radiologist_label_agrees_with_pathology=agree,
                         auc_vs_pathology=fast_auc(sub.dx.values, pp_s) if len(set(sub.dx)) > 1 else np.nan, auc_vs_radiologist_label=fast_auc(sub.label.values, pp_s) if len(set(sub.label)) > 1 else np.nan))
    PATHO = pd.DataFrame(rows); PATHO.to_csv(os.path.join(STAT_DIR, 'pathology_subgroup.csv'), index=False); print(PATHO.round(4).to_string(index=False))
    print('Small, selected subgroup: report as exploratory. Patients with several nodules are excluded because the diagnosis is patient-level.')
else: PATHO = None; print('(pathology subgroup skipped: table not provided -> keep the "radiologist-assessed malignancy-risk" wording)')

# ---- (c) patient-level performance (patient prediction = max nodule probability; label = any malignant) -----------------
rows = []
for name, df_, yy, pp, cb in (('dev_oof', DF_DEV, y_dev, oof_full, cb_dev), ('holdout', DF_HOLD, y_hold, hold_full, cb_hold)):
    pdl = patient_level(yy, pp, df_.patient.values); sub = ClusterBoot(pdl.g.values, CFG['N_BOOT'], 4); a, lo, hi = auc_ci(pdl.y.values, pdl.p.values, sub)
    rows.append(dict(dataset=name, n_patients=len(pdl), n_malignant_patients=int(pdl.y.sum()), patient_level_AUC=f"{a:.4f} [{lo:.4f}, {hi:.4f}]"))
PATLEV = pd.DataFrame(rows); PATLEV.to_csv(os.path.join(STAT_DIR, 'patient_level_auc.csv'), index=False); print('\n', PATLEV.to_string(index=False))

## 9b. Error audit and subgroup analysis (size, solidity, scan protocol)

In [ ]:
ERR = DF_HOLD.copy(); ERR['prob'] = hold_full; ERR['pred'] = (hold_full >= FINAL_THRESHOLD).astype(int); ERR['correct'] = ERR.pred == ERR.label
E = ERR[~ERR.correct][['patient', 'label', 'score', 'diam', 'n_ann', 'prob', 'pred', 'solidity', 'slice_thk']]
print(f"Holdout errors at the development threshold: {len(E)} / {len(ERR)}"); print(E.round(3).to_string(index=False)); E.to_csv(os.path.join(STAT_DIR, 'holdout_errors.csv'), index=False)
print('\nErrors with borderline radiologist scores (1.5 < score < 4.5 side) are label-ambiguity errors, not necessarily model errors:', int(((E.score > 1.5) & (E.score < 4.5)).sum()), 'of', len(E))

def subgroup_table(df_, yy, pp, thr, col, bins=None, labels=None):
    d = df_.copy(); d['p'] = pp; d['y'] = yy
    d['grp'] = pd.cut(d[col], bins=bins, labels=labels) if bins is not None else d[col]
    rows = []
    for g, s in d.groupby('grp', observed=True):
        c = conf_metrics(s.y.values, s.p.values, thr) if len(s) else {}
        rows.append(dict(variable=col, group=str(g), n=len(s), n_malignant=int(s.y.sum()), auc=fast_auc(s.y.values, s.p.values), sens=c.get('sens'), spec=c.get('spec')))
    return rows
SUBR = []
for name, df_, yy, pp in (('dev_oof', DF_DEV, y_dev, oof_full), ('holdout', DF_HOLD, y_hold, hold_full)):
    for r in subgroup_table(df_, yy, pp, FINAL_THRESHOLD, 'd_mm', [0, 6, 10, 20, 1e3], ['<6 mm', '6-10 mm', '10-20 mm', '>20 mm']) \
           + subgroup_table(df_, yy, pp, FINAL_THRESHOLD, 'solidity', [0, 0.6, 0.8, 1.01], ['low', 'mid', 'high']) \
           + subgroup_table(df_, yy, pp, FINAL_THRESHOLD, 'slice_thk', [0, 1.5, 2.5, 100], ['<=1.5 mm', '1.5-2.5 mm', '>2.5 mm']) \
           + subgroup_table(df_, yy, pp, FINAL_THRESHOLD, 'contrast'):
        SUBR.append(dict(dataset=name, **r))
SUBR = pd.DataFrame(SUBR); SUBR.to_csv(os.path.join(STAT_DIR, 'subgroups_lidc.csv'), index=False); print('\n', SUBR.round(3).to_string(index=False))
print('\nSubgroups with one class give AUC=NaN; interpret with the counts. Scanner manufacturer is not in the pylidc DB: slice thickness / contrast are used as protocol proxies.')

## 10. Freeze the final model (before any external data is touched)
The five fold models of the primary-seed `full` ensemble, their radiomics scalers and the **development-chosen thresholds** are copied to `frozen_model/` and SHA-256 hashed. External evaluation (§11) refuses to run if the hashes do not match, which documents that *no retraining, checkpoint selection or threshold tuning* happened on external data.

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

WORK_DIR = Path("/content/drive/MyDrive/lidc_work")
EXPECTED = Path("./lidc_work/test/models/dev/full/s42")

print("Current working directory:", Path.cwd())
print("Drive work directory exists:", WORK_DIR.exists())
print("Expected model directory:", EXPECTED.resolve())
print("Expected directory exists:", EXPECTED.exists())

print("\n=== Saved best-fold checkpoints ===")
for f in sorted(WORK_DIR.glob("best_fold*.pt")):
    print(f, f"({f.stat().st_size / 1e6:.1f} MB)")

print("\n=== Files in expected model directory ===")
if EXPECTED.exists():
    for f in sorted(EXPECTED.iterdir()):
        print(f)
else:
    print("Directory does not exist.")

print("\n=== Frozen-model configuration ===")
print("FROZEN_DIR:", globals().get("FROZEN_DIR"))
print("FROZEN_JSON:", globals().get("FROZEN_JSON"))

In [ ]:
from pathlib import Path

paths = [
    Path("/content/drive/MyDrive/lidc_work"),
    Path("/content/lidc_work/test/models/dev/full/s42"),
    Path("/content/lidc_work/test/frozen_model"),
    Path("/content/drive/MyDrive/external_cohorts/LNDb"),
]

for folder in paths:
    print(f"\nDirectory: {folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        for f in list(folder.iterdir())[:15]:
            print("  ", f.name)

print("\n=== Checkpoint search ===")
for root in [
    Path("/content/drive/MyDrive/lidc_work"),
    Path("/content/lidc_work"),
]:
    if root.exists():
        for f in root.rglob("best_fold*.pt"):
            print("Found:", f)
        for f in root.rglob("fold*.pt"):
            print("Found:", f)

In [ ]:
if not os.path.exists(FROZEN_JSON):
    FROZEN = freeze_model()
    print("Model frozen.")
else:
    FROZEN = json.load(open(FROZEN_JSON))
    print("Existing frozen model found (not overwritten).")

In [ ]:
for name in ["FINAL_THRESHOLD", "y_dev", "pred_set", "op_thresholds"]:
    print(name, "defined:", name in globals())

In [ ]:
# Check whether the existing training predictions are available
if "RUNS" not in globals():
    raise RuntimeError(
        "RUNS is unavailable. Do not rerun the training cell yet; "
        "we need to recover the saved predictions or restore the runtime state."
    )

if "full" not in RUNS or not RUNS["full"]:
    raise RuntimeError(
        "No full-model predictions are available in RUNS."
    )

if "y_dev" not in globals():
    raise RuntimeError("y_dev is missing from the current runtime.")

# Restore the prediction helper without running training
def pred_set(v, mode="primary"):
    rs = RUNS[v]

    if mode == "primary":
        r = rs[PS] if PS in rs else rs[sorted(rs)[0]]
        return r["oof"], r["hold"]

    return (
        np.mean([r["oof"] for r in rs.values()], axis=0),
        np.mean([r["hold"] for r in rs.values()], axis=0),
    )

# Calculate the threshold from development out-of-fold predictions
oof_f, hold_f = pred_set("full")
FINAL_THRESHOLD = youden_thr(y_dev, oof_f)

print("FINAL_THRESHOLD:", FINAL_THRESHOLD)
print("Development OOF AUC:", roc_auc_score(y_dev, oof_f))

In [ ]:
import os
import json
import shutil
import hashlib
import datetime
import numpy as np

# --------------------------------------------------
# 1. Use absolute Google Drive paths
# --------------------------------------------------
MODEL_ROOT = "/content/drive/MyDrive/lidc_work/test/models"

FROZEN_DIR = (
    "/content/drive/MyDrive/lidc_work/test/frozen_model"
)
FROZEN_JSON = os.path.join(FROZEN_DIR, "frozen.json")

# Do not use model_dir as a Path variable.
# It must not overwrite an existing model_dir() function.
SRC_DIR = os.path.join(
    MODEL_ROOT, "dev", "full", f"s{PS}"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def freeze_model():

    # --------------------------------------------------
    # 2. Verify required variables and functions
    # --------------------------------------------------
    required_names = [
        "pred_set",
        "op_thresholds",
        "fit_platt",
        "y_dev",
        "FINAL_THRESHOLD",
        "VARIANTS",
        "CFG",
        "PS",
        "DF_DEV",
        "TASK_NAME",
        "HU_MIN",
        "HU_MAX",
        "PATCH",
    ]

    missing_names = [
        name for name in required_names
        if name not in globals()
    ]

    if missing_names:
        raise NameError(
            "Required training variables/functions are missing: "
            + ", ".join(missing_names)
            + ". Run the relevant definition/prediction cells first."
        )

    if not os.path.isdir(SRC_DIR):
        raise FileNotFoundError(
            f"Model source directory not found: {SRC_DIR}"
        )

    # --------------------------------------------------
    # 3. Check every required checkpoint and scaler
    # --------------------------------------------------
    required_files = [
        os.path.join(SRC_DIR, filename)
        for fold in range(CFG["N_SPLITS"])
        for filename in (
            f"fold{fold}.pt",
            f"scaler_fold{fold}.pkl",
        )
    ]

    missing_files = [
        path for path in required_files
        if not os.path.isfile(path)
    ]

    if missing_files:
        raise FileNotFoundError(
            "Cannot freeze: required model files are missing:\n"
            + "\n".join(missing_files)
        )

    # --------------------------------------------------
    # 4. Validate development out-of-fold predictions
    # --------------------------------------------------
    y = np.asarray(y_dev).reshape(-1)
    probabilities = np.asarray(
        pred_set("full")[0]
    ).reshape(-1)

    if len(y) != len(probabilities):
        raise ValueError(
            f"Prediction/label count mismatch: "
            f"{len(probabilities)} predictions vs {len(y)} labels."
        )

    if len(y) == 0 or not np.isfinite(probabilities).all():
        raise ValueError(
            "Development predictions are empty or contain "
            "non-finite values."
        )

    threshold = float(FINAL_THRESHOLD)

    # Calculate calibration and operating-point information once.
    ops = op_thresholds(y, probabilities)
    calibrator = fit_platt(y, probabilities)

    platt_coef = float(calibrator.coef_[0, 0])
    platt_intercept = float(calibrator.intercept_[0])

    # --------------------------------------------------
    # 5. Copy the validated files and record their hashes
    # --------------------------------------------------
    os.makedirs(FROZEN_DIR, exist_ok=True)

    file_hashes = {}

    for source in required_files:
        filename = os.path.basename(source)
        destination = os.path.join(FROZEN_DIR, filename)

        # Prevent copying a file onto itself.
        if os.path.abspath(source) != os.path.abspath(destination):
            shutil.copy2(source, destination)

        file_hashes[filename] = sha256(destination)

    # --------------------------------------------------
    # 6. Save frozen-model metadata
    # --------------------------------------------------
    meta = {
        "created": datetime.datetime.now().isoformat(),
        "variant": "full",
        "spec": VARIANTS["full"],
        "seed": PS,
        "n_folds": CFG["N_SPLITS"],
        "threshold": threshold,
        "operating_points": ops,
        "platt_coef": platt_coef,
        "platt_intercept": platt_intercept,
        "threshold_source":
            "Youden on development out-of-fold predictions only",
        "hu_window": [HU_MIN, HU_MAX],
        "patch": PATCH,
        "files": file_hashes,
        "dev_patients": int(DF_DEV.patient.nunique()),
        "dev_nodules": int(len(DF_DEV)),
        "task": TASK_NAME,
    }

    # Write to a temporary file first, then replace atomically.
    temporary_json = FROZEN_JSON + ".tmp"

    with open(temporary_json, "w") as f:
        json.dump(meta, f, indent=2)

    os.replace(temporary_json, FROZEN_JSON)

    return meta


# --------------------------------------------------
# 7. Freeze only if metadata does not already exist
# --------------------------------------------------
if os.path.exists(FROZEN_JSON):
    with open(FROZEN_JSON, "r") as f:
        FROZEN = json.load(f)

    print("Existing frozen model found; metadata not overwritten.")

else:
    FROZEN = freeze_model()
    print("Model frozen successfully.")

print({
    key: FROZEN[key]
    for key in (
        "threshold",
        "operating_points",
        "seed",
        "threshold_source",
    )
})

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

FROZEN_JSON = os.path.join(FROZEN_DIR, 'frozen.json')
def freeze_model():
    src = model_dir('dev', 'full', PS); files = {}
    for f in range(CFG['N_SPLITS']):
        for fn in (f'fold{f}.pt', f'scaler_fold{f}.pkl'):
            shutil.copy(os.path.join(src, fn), os.path.join(FROZEN_DIR, fn)); files[fn] = sha256(os.path.join(FROZEN_DIR, fn))
    ops = op_thresholds(y_dev, pred_set('full')[0])
    meta = dict(created=datetime.datetime.now().isoformat(), variant='full', spec=VARIANTS['full'], seed=PS, n_folds=CFG['N_SPLITS'], threshold=FINAL_THRESHOLD, operating_points=ops,
                platt_coef=float(fit_platt(y_dev, pred_set('full')[0]).coef_[0, 0]), platt_intercept=float(fit_platt(y_dev, pred_set('full')[0]).intercept_[0]),
                threshold_source='Youden on development out-of-fold predictions only', hu_window=[HU_MIN, HU_MAX], patch=PATCH, files=files,
                dev_patients=int(DF_DEV.patient.nunique()), dev_nodules=int(len(DF_DEV)), task=TASK_NAME)
    json.dump(meta, open(FROZEN_JSON, 'w'), indent=2); return meta

if not os.path.exists(FROZEN_JSON): FROZEN = freeze_model(); print('Model frozen.')
else: FROZEN = json.load(open(FROZEN_JSON)); print('Existing frozen model found (not overwritten).')
print({k: FROZEN[k] for k in ('threshold', 'operating_points', 'seed', 'threshold_source')})

## 11. Explainability (reviewer item F)
Grad-CAM alone is not enough (and, in the earlier run, was *worse than a centre prior*). This section keeps Grad-CAM as a reported **negative / limited** finding and adds: **occlusion sensitivity**, **integrated gradients**, **counterfactual patch perturbation** (nodule removed vs. matched control patch), **radiomics feature attribution** (permutation importance), the earlier **input-swap sensitivity diagnostic**, and a **blinded radiologist review pack**. All use the frozen 5-model ensemble on *all* holdout nodules (no case picking). Attributions are compared with the radiologist consensus mask and with a centre-prior baseline, using IoU(>0.5), IoU of the top-k voxels (k = mask size, threshold-free), pointing within 4 mm, and attribution energy inside the mask.

In [ ]:
def load_ensemble_from(d, spec):
    ms = []
    for f in range(CFG['N_SPLITS']):
        m = Net(scales=spec['scales'], radiomics=spec['radiomics'], cbam=spec['cbam']).to(DEVICE)
        m.load_state_dict(torch.load(os.path.join(d, f'fold{f}.pt'), map_location=DEVICE)); m.eval()
        ms.append((m, pickle.load(open(os.path.join(d, f'scaler_fold{f}.pkl'), 'rb'))))
    return ms

def verify_frozen():
    meta = json.load(open(FROZEN_JSON))
    for fn, h in meta['files'].items():
        assert sha256(os.path.join(FROZEN_DIR, fn)) == h, f'frozen file changed: {fn}'
    return meta

FROZEN = verify_frozen(); MODELS = load_ensemble_from(FROZEN_DIR, FROZEN['spec'])
norm01 = lambda a: (a - a.min()) / (a.max() - a.min() + 1e-8)

@torch.no_grad()
def ens_predict(models, P, R_raw=None, R_scaled=None, bs=32):
    # P: (B,64,64,64) in [0,1]; R_raw: (28,) or (B,28) raw radiomics (scaled per model); or R_scaled for already-scaled input
    P = np.asarray(P, np.float32); B = len(P); out = np.zeros((len(models), B))
    for mi, (m, sc) in enumerate(models):
        m.eval()
        Rm = np.broadcast_to(R_scaled, (B, 28)) if R_scaled is not None else np.clip(sc.transform(np.array(np.broadcast_to(R_raw, (B, 28)))), -5, 5)
        Rm = np.ascontiguousarray(Rm, dtype=np.float32)
        for i in range(0, B, bs):
            x = torch.from_numpy(P[i:i + bs]).to(DEVICE)[:, None]; r = torch.from_numpy(Rm[i:i + bs]).to(DEVICE)
            out[mi, i:i + bs] = F.softmax(m(x[:, :, 16:48, 16:48, 16:48], x[:, :, 8:56, 8:56, 8:56], x, r), 1)[:, 1].cpu().numpy()
    return out.mean(0)

IMGS = np.stack([np.load(f)['p64'].astype(np.float32) for f in DF_HOLD.file.values]); MASKS = np.stack([np.load(f)['mask'].astype(bool) for f in DF_HOLD.file.values])
RADS = np.stack(DF_HOLD.rad.values).astype(np.float32)
P_CHECK = ens_predict(MODELS, IMGS, RADS)
print('consistency: max |frozen-ensemble P - stored holdout P| = %.5f (should be ~0)' % np.abs(P_CHECK - pred_set('full')[1]).max())
NX = len(DF_HOLD) if CFG['EXPL_MAX_NODULES'] is None else min(CFG['EXPL_MAX_NODULES'], len(DF_HOLD))

In [ ]:
# ---------------- attribution methods ----------------
def gradcam_map(models, p, R_raw, b, layer):
    cams = []
    for m, sc in models:
        m.eval(); m.zero_grad(set_to_none=True)
        rad = torch.from_numpy(np.clip(sc.transform(R_raw[None]), -5, 5).astype(np.float32)).to(DEVICE); x = torch.from_numpy(p)[None, None].to(DEVICE); store = []
        h = m.b[b].conv[layer].register_forward_hook(lambda mod, i, o: store.append(o))
        out = m(x[:, :, 16:48, 16:48, 16:48], x[:, :, 8:56, 8:56, 8:56], x, rad); h.remove()
        fm = store[0]; fm.retain_grad(); out[0, 1].backward()                       # malignant-class score
        cam = F.relu((fm.grad.mean((2, 3, 4), keepdim=True) * fm).sum(1, keepdim=True)); sz = (32, 48, 64)[b]
        cam = F.interpolate(cam, size=(sz,) * 3, mode='trilinear', align_corners=False)[0, 0].detach().cpu().numpy(); o = (sz - 32) // 2
        cams.append(norm01(cam[o:o + 32, o:o + 32, o:o + 32]))
    return norm01(np.mean(cams, 0))

def occlusion_map(models, p, R_raw, win=8, stride=4, chunk=64):
    base = ens_predict(models, p[None], R_raw)[0]; fill = float(np.median(p)); org = list(itertools.product(range(16, 48 - win + 1, stride), repeat=3))
    drop, cnt = np.zeros((32, 32, 32)), np.zeros((32, 32, 32))
    for i in range(0, len(org), chunk):
        sel = org[i:i + chunk]; batch = np.repeat(p[None], len(sel), 0)
        for k, (a, b, c) in enumerate(sel): batch[k, a:a + win, b:b + win, c:c + win] = fill
        pr = ens_predict(models, batch, R_raw)
        for (a, b, c), q in zip(sel, pr): drop[a - 16:a - 16 + win, b - 16:b - 16 + win, c - 16:c - 16 + win] += base - q; cnt[a - 16:a - 16 + win, b - 16:b - 16 + win, c - 16:c - 16 + win] += 1
    return norm01(np.clip(drop / np.maximum(cnt, 1), 0, None))

def ig_map(models, p, R_raw, steps=24, chunk=8):
    x = torch.from_numpy(p)[None, None].to(DEVICE); maps, deltas = [], []
    for m, sc in models:
        m.eval(); rad = torch.from_numpy(np.clip(sc.transform(R_raw[None]), -5, 5).astype(np.float32)).to(DEVICE); total = torch.zeros_like(x)
        al = torch.linspace(1.0 / steps, 1.0, steps, device=DEVICE)
        for i in range(0, steps, chunk):
            a = al[i:i + chunk].view(-1, 1, 1, 1, 1); xs = (a * x).clone().requires_grad_(True)       # baseline = 0 = air
            lo = m(xs[:, :, 16:48, 16:48, 16:48], xs[:, :, 8:56, 8:56, 8:56], xs, rad.expand(len(a), -1)); m.zero_grad(set_to_none=True); lo[:, 1].sum().backward(); total += xs.grad.sum(0, keepdim=True)
        ig = (x * total / steps)[0, 0].detach().cpu().numpy()
        with torch.no_grad():
            f1 = m(x[:, :, 16:48, 16:48, 16:48], x[:, :, 8:56, 8:56, 8:56], x, rad)[0, 1].item(); z0 = torch.zeros_like(x); f0 = m(z0[:, :, 16:48, 16:48, 16:48], z0[:, :, 8:56, 8:56, 8:56], z0, rad)[0, 1].item()
        deltas.append(abs(ig.sum() - (f1 - f0)) / (abs(f1 - f0) + 1e-8)); maps.append(ig)
    a = ndimage.gaussian_filter(np.clip(np.mean(maps, 0)[16:48, 16:48, 16:48], 0, None), 1.0)
    return norm01(a), float(np.mean(deltas))

def attr_metrics(a, mask):
    k = int(mask.sum()); pk = np.unravel_index(a.argmax(), a.shape); near = ndimage.binary_dilation(mask, iterations=4); b = a > 0.5
    bk = np.zeros(a.size, bool); bk[np.argpartition(a.ravel(), -k)[-k:]] = True; bk = bk.reshape(a.shape)
    return dict(iou=(b & mask).sum() / max((b | mask).sum(), 1), iou_topk=(bk & mask).sum() / max((bk | mask).sum(), 1), point=float(mask[pk]), point_tol=float(near[pk]), energy=float((a * mask).sum() / (a.sum() + 1e-8)))

G_CENTRE = norm01(np.exp(-((np.indices((32, 32, 32)) - 15.5) ** 2).sum(0) / (2 * 5.0 ** 2))); G_UNIFORM = np.ones((32, 32, 32))
VAR_CAM = {'Grad-CAM 32-branch 4^3': (0, 11), 'Grad-CAM 64-branch 8^3': (2, 11), 'Grad-CAM 64-branch 16^3': (2, 7)}

def fill_region(p, reg):
    ring = ndimage.binary_dilation(reg, iterations=5) & ~reg; q = p.copy(); q[reg] = np.median(p[ring]); return q
def control_region(reg, rng):
    for _ in range(50):
        d = rng.randn(3); d = d / np.linalg.norm(d) * rng.uniform(14, 22); r2 = ndimage.shift(reg.astype(float), d, order=0) > 0.5
        if r2.sum() >= 0.8 * reg.sum(): return r2
    return None
print('attribution methods ready')

In [ ]:
# ---------------- run on all (or the first NX) holdout nodules ----------------
rng = np.random.RandomState(1); rows = []; REVIEW_MAPS = {}
review_idx = set()
if CFG['N_REVIEW'] > 0:
    err_idx = list(np.where((hold_full >= FINAL_THRESHOLD).astype(int) != y_hold)[0][:max(1, CFG['N_REVIEW'] // 5)])
    rest = [i for i in np.random.RandomState(7).permutation(NX) if i not in err_idx]
    review_idx = set((err_idx + rest)[:CFG['N_REVIEW']])
t0 = time.time()
for i in range(NX):
    p, mask, R_ = IMGS[i], MASKS[i][16:48, 16:48, 16:48], RADS[i]
    maps = {n: gradcam_map(MODELS, p, R_, b, li) for n, (b, li) in VAR_CAM.items()}
    maps['Occlusion (8^3, stride 4)'] = occlusion_map(MODELS, p, R_)
    maps['Integrated gradients'], conv = ig_map(MODELS, p, R_)
    rec = dict(i=i, patient=DF_HOLD.patient.values[i], true=int(y_hold[i]), prob=float(hold_full[i]), ig_completeness_err=conv)
    for n, a in maps.items(): rec.update({f'{k}|{n}': v for k, v in attr_metrics(a, mask).items()})
    for n, a in (('centre prior', G_CENTRE), ('uniform', G_UNIFORM)): rec.update({f'{k}|{n}': v for k, v in attr_metrics(a, mask).items()})
    reg = ndimage.binary_dilation(MASKS[i], iterations=2); ctl = control_region(reg, rng)       # counterfactual patch perturbation
    batch = [p, fill_region(p, reg)] + ([fill_region(p, ctl)] if ctl is not None else [])
    pr = ens_predict(MODELS, np.stack(batch), R_); rec.update(p0=pr[0], p_nodule_removed=pr[1], p_control_removed=pr[2] if ctl is not None else np.nan)
    rows.append(rec)
    if i in review_idx: REVIEW_MAPS[i] = maps
    if (i + 1) % 10 == 0: print(f"{i+1}/{NX} nodules | {(time.time()-t0)/60:.1f} min")
EX = pd.DataFrame(rows); EX.to_csv(os.path.join(EXPL_DIR, 'explainability_per_nodule.csv'), index=False)

cbx = ClusterBoot(EX.patient.values, CFG['N_BOOT'], 5); methods = list(VAR_CAM) + ['Occlusion (8^3, stride 4)', 'Integrated gradients', 'centre prior', 'uniform']
tab = []
for n in methods:
    row = dict(method=n)
    for k in ('iou', 'iou_topk', 'point_tol', 'energy'):
        mu, lo, hi = mean_ci(EX[f'{k}|{n}'].values, cbx); row[k] = f"{mu:.3f} [{lo:.3f},{hi:.3f}]"
    tab.append(row)
EXPL_TAB = pd.DataFrame(tab); EXPL_TAB.to_csv(os.path.join(EXPL_DIR, 'attribution_vs_mask.csv'), index=False)
print(f"=== Attribution vs radiologist mask, mean over {len(EX)} holdout nodules (patient-bootstrap 95% CI) ==="); print(EXPL_TAB.to_string(index=False))
print(f"\nIG completeness error (mean relative): {EX.ig_completeness_err.mean():.3f}  (small = attributions sum to the output change)")
better = {n: float((EX[f'iou_topk|{n}'] - EX['iou_topk|centre prior']).mean()) for n in methods if n not in ('centre prior', 'uniform')}
print('Mean top-k IoU minus centre-prior IoU (positive = better than assuming "the nodule is in the centre"):', {k: round(v, 3) for k, v in better.items()})

In [ ]:
# ---- counterfactual patch perturbation ----------------------------------------------------------------------------------
thr = FINAL_THRESHOLD; EX['d_nodule'] = EX.p_nodule_removed - EX.p0; EX['d_control'] = EX.p_control_removed - EX.p0
EX['flip_nodule'] = (EX.p_nodule_removed >= thr) != (EX.p0 >= thr); EX['flip_control'] = ((EX.p_control_removed >= thr) != (EX.p0 >= thr)) & EX.p_control_removed.notna()
print(EX.groupby('true')[['p0', 'p_nodule_removed', 'p_control_removed', 'd_nodule', 'd_control', 'flip_nodule', 'flip_control']].mean().round(3).T.to_string())
mu_n, lo_n, hi_n = mean_ci(np.abs(EX.d_nodule.values), cbx); mu_c, lo_c, hi_c = mean_ci(np.abs(EX.d_control.fillna(0).values), cbx)
print(f"\nmean |dP| nodule removed {mu_n:.3f} [{lo_n:.3f},{hi_n:.3f}] vs control patch {mu_c:.3f} [{lo_c:.3f},{hi_c:.3f}]")
print('Reading: removing the nodule moving P much more than an equal-sized control patch => prediction depends on the nodule itself, even if saliency maps are coarse.\n'
      'NOTE: radiomics (computed on the real mask) are held fixed here, so this probes the IMAGE pathway only.')

# ---- input-swap sensitivity diagnostic (not causal feature importance) ----------------------------------------------------
rg = np.random.RandomState(0); N = len(y_hold); shift = lambda: (np.arange(N) + rg.randint(1, N)) % N
sc_ = {'intact': ens_predict(MODELS, IMGS, RADS)}
s = shift(); sc_['CT swapped between nodules'] = ens_predict(MODELS, IMGS[s], RADS)
s = shift(); sc_['radiomics swapped between nodules'] = ens_predict(MODELS, IMGS, RADS[s])
sc_['radiomics = training mean'] = ens_predict(MODELS, IMGS, R_scaled=np.zeros(28, np.float32))
SWAP = pd.DataFrame([dict(scenario=k, AUC=fast_auc(y_hold, v), mean_abs_prob_change=np.abs(v - sc_['intact']).mean()) for k, v in sc_.items()])
SWAP.to_csv(os.path.join(EXPL_DIR, 'input_swap.csv'), index=False); print('\n', SWAP.round(3).to_string(index=False))
print('Report as a sensitivity diagnostic. AUC stays ~intact when CT is swapped => CT pathway nearly ignored; collapses when radiomics swapped => radiomics dominates.')

# ---- radiomics feature attribution: permutation importance (holdout, frozen ensemble, image fixed) -----------------------
base_auc = fast_auc(y_hold, ens_predict(MODELS, IMGS, RADS)); prm = np.random.RandomState(3); rows = []
for j, name in enumerate(RAD_NAMES):
    drops = []
    for _ in range(10 if CFG['MODE'] == 'full' else 3):
        R2 = RADS.copy(); R2[:, j] = R2[prm.permutation(N), j]; drops.append(base_auc - fast_auc(y_hold, ens_predict(MODELS, IMGS, R2)))
    rows.append(dict(feature=name, auc_drop_mean=np.mean(drops), auc_drop_sd=np.std(drops)))
for g, cols in RAD_GROUPS.items():
    drops = []
    for _ in range(10 if CFG['MODE'] == 'full' else 3):
        R2 = RADS.copy(); R2[:, cols] = R2[prm.permutation(N)][:, cols]; drops.append(base_auc - fast_auc(y_hold, ens_predict(MODELS, IMGS, R2)))
    rows.append(dict(feature=f'[GROUP] {g}', auc_drop_mean=np.mean(drops), auc_drop_sd=np.std(drops)))
PERM = pd.DataFrame(rows).sort_values('auc_drop_mean', ascending=False); PERM.to_csv(os.path.join(EXPL_DIR, 'radiomics_permutation_importance.csv'), index=False)
print(f'\nHoldout AUC {base_auc:.4f}; top radiomics attributions (AUC drop when permuted):'); print(PERM.head(12).round(4).to_string(index=False))

In [ ]:
# ---- blinded radiologist review pack: panels (CT+mask / Grad-CAM / occlusion / IG) with random IDs, label & prediction hidden ----
REV_DIR = os.path.join(EXPL_DIR, 'radiologist_review'); os.makedirs(REV_DIR, exist_ok=True)
ids = np.random.RandomState(11).permutation(len(REVIEW_MAPS)); keyrows, formrows = [], []
for rid, (i, maps) in zip(ids, REVIEW_MAPS.items()):
    cid = f"R{rid:03d}"; p = IMGS[i][16:48, 16:48, 16:48]; m = MASKS[i][16:48, 16:48, 16:48]
    show = [('CT + radiologist mask', None), ('Grad-CAM 64-branch 8^3', maps['Grad-CAM 64-branch 8^3']), ('Occlusion (8^3, stride 4)', maps['Occlusion (8^3, stride 4)']), ('Integrated gradients', maps['Integrated gradients'])]
    fig, ax = plt.subplots(3, 4, figsize=(12, 9))
    for c, (t, a) in enumerate(show):
        for r_, (sl, ms, am) in enumerate(((p[16], m[16], None if a is None else a[16]), (p[:, 16], m[:, 16], None if a is None else a[:, 16]), (p[:, :, 16], m[:, :, 16], None if a is None else a[:, :, 16]))):
            ax[r_, c].imshow(sl, cmap='gray', vmin=0, vmax=1)
            if am is not None: ax[r_, c].imshow(am, cmap='jet', alpha=0.4, vmin=0, vmax=1)
            ax[r_, c].contour(ms.astype(float), [0.5], colors='w', linewidths=1); ax[r_, c].axis('off')
        ax[0, c].set_title(t, fontsize=9)
    plt.suptitle(cid); plt.tight_layout(); plt.savefig(os.path.join(REV_DIR, f'{cid}.png'), dpi=110); plt.close()
    keyrows.append(dict(case_id=cid, holdout_index=int(i), patient=DF_HOLD.patient.values[i], true=int(y_hold[i]), prob=float(hold_full[i])))
    formrows.append(dict(case_id=cid, map_boundary_agreement_0none_1partial_2good='', highlights_spiculation_0_1='', highlights_heterogeneity_0_1='', highlights_other_relevant_feature_0_1='', attention_on_non_nodule_structure_0_1='', comment=''))
pd.DataFrame(keyrows).to_csv(os.path.join(REV_DIR, 'KEY_do_not_share_with_reviewers.csv'), index=False)
pd.DataFrame(formrows).to_csv(os.path.join(REV_DIR, 'review_form.csv'), index=False)
print(f'Review pack: {len(formrows)} blinded panels in {REV_DIR}. Give PNGs + review_form.csv to >=2 radiologists, then run analyze_reviews().')

def analyze_reviews(form_csv, key_csv=os.path.join(REV_DIR, 'KEY_do_not_share_with_reviewers.csv')):
    f = pd.read_csv(form_csv); cols = [c for c in f.columns if c not in ('case_id', 'comment')]; f[cols] = f[cols].apply(pd.to_numeric, errors='coerce')
    print(f[cols].describe().loc[['count', 'mean']].round(3).to_string())
    if 'reviewer' in f.columns:      # optional column if you concatenate several reviewers' sheets
        agg = f.groupby('case_id')[cols[0]].agg(['count', 'std']); print('multi-reviewer spread for boundary agreement:', agg['std'].mean())
    return f

## 12. External validation (reviewer item A — the most important addition)
**Rules enforced by the code:** the frozen ensemble and the frozen development threshold are used *unchanged* (hash-verified); no external data enter training, checkpoint selection, scaling or threshold tuning; the radiomics scaler is the training-only scaler stored with each fold model.

**You must supply the data** (it cannot be streamed automatically): put one folder per cohort in `CFG['EXTERNAL_DIR']`, each containing `manifest.csv` with columns

`case_id, patient_id, image_path, world_x, world_y, world_z, label, diameter_mm` and optionally `scanner, slice_thickness_mm`

* `image_path`: a DICOM series folder or a `.mhd/.nii.gz` volume (HU); `world_*`: nodule centre in mm (DICOM patient coordinates / the cohort's finding file); `label`: 0 benign, 1 malignant; `diameter_mm`: nominal diameter (used for the automatic nodule mask).
* **LNDb (primary):** verify which reference standard it provides for your task. If it has no malignancy label, state the proxy you derive and its limits — do not call it pathology.
* **LUNGx (secondary):** reported as having biopsy / surgery / follow-up-confirmed benign vs malignant labels (please verify on the TCIA page); this is the pathology-grade test.
* No radiologist masks exist externally, so radiomics are computed from an **automatic mask** (threshold + connected component around the centre). §12.1 first quantifies how much this mask source changes results on LIDC-IDRI.
* Do **not** claim external validation from another LIDC split.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from urllib.parse import quote
import requests
import subprocess
import hashlib
import json
import csv
import time

# --------------------------------------------------
# LNDb Version 4: persistent Google Drive storage
# --------------------------------------------------
RECORD_ID = '8348419'

BASE = Path('/content/drive/MyDrive/external_cohorts/LNDb')
ARCHIVE_DIR = BASE / 'raw_archives'
METADATA_DIR = BASE / 'metadata'
LOG_DIR = BASE / 'logs'

for folder in (ARCHIVE_DIR, METADATA_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

STATUS_FILE = LOG_DIR / 'download_status.csv'
RECORD_FILE = METADATA_DIR / 'zenodo_record.json'

# Get the official record and file list.
api_url = f'https://zenodo.org/api/records/{RECORD_ID}'
response = requests.get(api_url, timeout=90)
response.raise_for_status()
record = response.json()

RECORD_FILE.write_text(json.dumps(record, indent=2))

files = record.get('files', [])
if not files:
    raise RuntimeError('No files listed in the official Zenodo record.')

def get_name(info):
    return info.get('key') or info.get('filename')

def destination(name):
    if name.lower().endswith(('.rar', '.zip')):
        return ARCHIVE_DIR / name
    return METADATA_DIR / name

def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024), b''
        ):
            h.update(block)
    return h.hexdigest()

# Keep a durable status log, updated after each file.
if STATUS_FILE.exists():
    with open(STATUS_FILE, newline='') as f:
        old_rows = list(csv.DictReader(f))
else:
    old_rows = []

status = {row['filename']: row for row in old_rows}

def save_status(name, size, checksum, path, state, note=''):
    status[name] = {
        'filename': name,
        'expected_bytes': str(size),
        'expected_checksum': checksum or '',
        'path': str(path),
        'status': state,
        'note': note,
        'updated_unix': str(int(time.time()))
    }

    fields = [
        'filename', 'expected_bytes', 'expected_checksum',
        'path', 'status', 'note', 'updated_unix'
    ]

    temp = STATUS_FILE.with_suffix('.tmp')
    with open(temp, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(status.values())

    temp.replace(STATUS_FILE)

# Small metadata first, then archives in ascending file size.
files.sort(key=lambda info: (
    get_name(info).lower().endswith(('.rar', '.zip')),
    int(info.get('size', 0))
))

total_bytes = sum(int(f.get('size', 0)) for f in files)

print('Dataset:', record['metadata']['title'])
print('Version:', record['metadata'].get('version', 'see record'))
print('Number of files:', len(files))
print(f'Total size: {total_bytes / (1024**3):.2f} GiB')
print('Destination:', BASE)
print('Progress log:', STATUS_FILE)

def download_one(info):
    name = get_name(info)
    expected = int(info['size'])
    checksum_field = info.get('checksum', '')
    expected_md5 = (
        checksum_field.split(':', 1)[1].lower()
        if checksum_field.startswith('md5:')
        else None
    )

    dest = destination(name)
    marker = dest.with_name(dest.name + '.verified')
    dest.parent.mkdir(parents=True, exist_ok=True)

    # Skip only files whose official checksum was verified,
    # or whose size was verified when no checksum is published.
    if dest.exists() and dest.stat().st_size == expected:
        if expected_md5:
            if marker.exists() and marker.read_text().strip() == expected_md5:
                print(f'SKIP verified: {name}')
                save_status(name, expected, expected_md5, dest, 'verified')
                return

            print(f'Checking existing file: {name}')
            if md5_file(dest).lower() == expected_md5:
                marker.write_text(expected_md5)
                print(f'Already complete: {name}')
                save_status(name, expected, expected_md5, dest, 'verified')
                return

            print('Checksum mismatch; removing invalid file.')
            dest.unlink()
            marker.unlink(missing_ok=True)
        else:
            marker.write_text('SIZE_ONLY')
            save_status(name, expected, '', dest, 'size_verified')
            print(f'Complete by size (no checksum supplied): {name}')
            return

    # Never resume from a file that is already too large.
    if dest.exists() and dest.stat().st_size > expected:
        dest.unlink()
        marker.unlink(missing_ok=True)

    url = (
        f'https://zenodo.org/records/{RECORD_ID}/files/'
        f'{quote(name, safe="")}?download=1'
    )

    print(f'\nDownloading/resuming: {name}')
    print(f'Size: {expected / (1024**3):.3f} GiB')

    save_status(
        name, expected, expected_md5 or '', dest, 'downloading'
    )

    try:
        subprocess.run([
            'curl',
            '--fail',
            '--location',
            '--retry', '5',
            '--retry-all-errors',
            '--retry-delay', '5',
            '--connect-timeout', '60',
            '--continue-at', '-',
            '--output', str(dest),
            '--progress-bar',
            url
        ], check=True)

        if not dest.exists() or dest.stat().st_size != expected:
            actual = dest.stat().st_size if dest.exists() else 0
            raise RuntimeError(
                f'{name}: expected {expected} bytes; '
                f'found {actual}. Rerun this cell to resume.'
            )

        if expected_md5:
            print('Verifying MD5 checksum...')
            actual_md5 = md5_file(dest).lower()

            if actual_md5 != expected_md5:
                dest.unlink(missing_ok=True)
                marker.unlink(missing_ok=True)
                raise RuntimeError(
                    f'Checksum mismatch for {name}. '
                    'Invalid file removed; rerun the cell.'
                )

            marker.write_text(expected_md5)
        else:
            marker.write_text('SIZE_ONLY')

        save_status(
            name, expected, expected_md5 or '', dest, 'verified'
        )
        print(f'COMPLETE & VERIFIED: {name}')

    except Exception as exc:
        save_status(
            name, expected, expected_md5 or '', dest,
            'incomplete', str(exc)
        )
        raise

for info in files:
    download_one(info)

print('\nAll listed LNDb files have been processed.')
print('Verify every row is marked verified or size_verified:')
print(STATUS_FILE)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import shutil
import os

ZIP_FILE = Path(
    '/content/drive/MyDrive/external_cohorts/LUNGx/tools/'
    'TCIA_Data_Retriever_linux_x86_64.zip'
)

EXTRACT_DIR = Path('/content/tcia_data_retriever')

assert ZIP_FILE.exists(), (
    f'ZIP not found: {ZIP_FILE}\n'
    'Check that you uploaded it to the correct Drive folder.'
)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE) as z:
    print('Files included in archive:')
    for name in z.namelist()[:25]:
        print(' ', name)

    z.extractall(EXTRACT_DIR)

# Find executable files without assuming the ZIP's exact layout.
candidates = [
    p for p in EXTRACT_DIR.rglob('*')
    if p.is_file()
    and 'tcia_data_retriever' in p.name.lower()
    and not p.name.lower().endswith(('.zip', '.txt', '.md'))
]

for p in candidates:
    p.chmod(p.stat().st_mode | 0o111)

print('\nExtracted to:', EXTRACT_DIR)
print('\nPotential executables:')
for p in candidates:
    print(p)

print('\nExtraction complete.')


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

ROOT = Path('/content/drive/MyDrive/external_cohorts/LUNGx')
METADATA = ROOT / 'metadata'
IMAGES = ROOT / 'images'

APP = Path(
    '/content/tcia_data_retriever/'
    'TCIA_Data_Retriever-x86_64.AppImage'
)

# Find the manifest
manifests = list(METADATA.glob('*.tcia'))

if len(manifests) != 1:
    raise RuntimeError(
        f'Expected exactly one .tcia manifest in {METADATA}, '
        f'but found {len(manifests)}: {manifests}'
    )

MANIFEST = manifests[0]

if not APP.exists():
    raise FileNotFoundError(f'Retriever not found: {APP}')

APP.chmod(0o755)
IMAGES.mkdir(parents=True, exist_ok=True)

print('Manifest:', MANIFEST)
print('Image destination:', IMAGES)

# Confirm that you have reviewed and accepted TCIA's data-use policy.
accepted = input(
    'After reviewing the TCIA data-use policy, '
    'type I AGREE to begin the download: '
).strip()

if accepted != 'I AGREE':
    raise RuntimeError('Download cancelled; policy not confirmed.')

command = [
    str(APP),
    '--appimage-extract-and-run',
    '--cli',
    '--input', str(MANIFEST),
    '--output', str(IMAGES),
    '--skip-existing',
    '--save-log',
    '--processes', '4',
    '--max-retries', '5',
    '--accept-data-policy'
]

print('\nStarting LUNGx download...')
subprocess.run(command, check=True)

print('\nDownload command completed.')
print('Check the output above for download errors.')
print('Files are stored at:', IMAGES)


In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path('/content/drive/MyDrive/external_cohorts/LUNGx')
IMAGES = ROOT / 'images'
METADATA = ROOT / 'metadata'

# Check the image download
if IMAGES.exists():
    files = [p for p in IMAGES.rglob('*') if p.is_file()]
    total_bytes = sum(p.stat().st_size for p in files)

    print('=== LUNGx image verification ===')
    print('Image folder exists:', True)
    print('Total files:', len(files))
    print(f'Total size: {total_bytes / (1024**3):.2f} GiB')

    print('\nFile extensions:')
    print(Counter(
        p.suffix.lower() or '[no extension]'
        for p in files
    ).most_common(10))

    print('\nExample files:')
    for p in files[:10]:
        print(p.relative_to(ROOT))
else:
    print('Images folder NOT FOUND:', IMAGES)

# Check supporting metadata
print('\n=== Metadata verification ===')
if METADATA.exists():
    for p in sorted(METADATA.iterdir()):
        print(p.name, '-', 'file' if p.is_file() else 'folder')
else:
    print('Metadata folder NOT FOUND:', METADATA)

In [ ]:
from pathlib import Path
import os
import pandas as pd

BASE = Path('/content/drive/MyDrive/external_cohorts')
LNDB = BASE / 'LNDb'
LUNGX = BASE / 'LUNGx'

def inspect_folder(root, max_examples=12):
    print(f'\n{"=" * 65}')
    print(root)
    print('=' * 65)

    if not root.exists():
        print('FOLDER NOT FOUND')
        return

    files = [p for p in root.rglob('*') if p.is_file()]
    total = sum(p.stat().st_size for p in files)

    print('Files:', len(files))
    print(f'Total stored size: {total / (1024**3):.2f} GiB')
    print('\nExample files:')

    for p in files[:max_examples]:
        print(f'  {p.relative_to(root)}')

    manifests = list(root.rglob('manifest.csv'))
    print('\nmanifest.csv files:', manifests or 'NONE')

print('LNDb download status')
status_file = LNDB / 'logs' / 'download_status.csv'

if status_file.exists():
    status = pd.read_csv(status_file)
    print(status[['filename', 'status']].to_string(index=False))
else:
    print('No LNDb download-status file found.')

inspect_folder(LNDB)
inspect_folder(LUNGX)

In [ ]:
# Inspect LNDb CSV files
lndb_metadata = LNDB / 'metadata'

print('\n========== LNDb metadata ==========')

for path in sorted(lndb_metadata.glob('*.csv')):
    print(f'\n--- {path.name} ---')
    try:
        df = pd.read_csv(path)
        print('Shape:', df.shape)
        print('Columns:', df.columns.tolist())
        print(df.head(3).to_string(index=False))
    except Exception as e:
        print('Could not read:', e)

# Inspect LUNGx Excel files
lungx_metadata = LUNGX / 'metadata'

print('\n========== LUNGx metadata ==========')

for path in sorted(lungx_metadata.glob('*.xlsx')):
    print(f'\n--- {path.name} ---')
    try:
        xls = pd.ExcelFile(path)
        print('Sheets:', xls.sheet_names)

        for sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            print(f'\nSheet: {sheet} | Shape: {df.shape}')
            print('Columns:', df.columns.tolist())
            print(df.head(3).to_string(index=False))
    except Exception as e:
        print('Could not read:', e)

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

ROOT = Path(
    '/content/drive/MyDrive/external_cohorts/LNDb'
)

ZIP_FILE = ROOT / 'raw_archives' / 'trainset_csv.zip'
OUT = ROOT / 'metadata' / 'trainset_csv'

assert ZIP_FILE.exists(), f'Missing: {ZIP_FILE}'
OUT.mkdir(parents=True, exist_ok=True)

# Extract the annotation tables.
with ZipFile(ZIP_FILE) as z:
    print('Files inside trainset_csv.zip:')
    for name in z.namelist():
        print(' ', name)

    z.extractall(OUT)

# Inspect the key annotation tables.
for name in [
    'trainCTs.csv',
    'trainNodules.csv',
    'trainNodules_gt.csv',
    'trainFleischner.csv',
]:
    matches = list(OUT.rglob(name))

    if not matches:
        print(f'\n{name}: not found')
        continue

    path = matches[0]
    df = pd.read_csv(path)

    print(f'\n--- {name} ---')
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    print(df.head(3).to_string(index=False))

# Summarize the risk scores already available in allNods.csv.
allnods_path = ROOT / 'metadata' / 'allNods.csv'
df = pd.read_csv(allnods_path)

print('\n--- allNods.csv malignancy ratings ---')
print(df['Malignancy'].describe())
print('\nScore counts:')
print(df['Malignancy'].value_counts(dropna=False).sort_index())

if 'Where' in df.columns:
    print('\nAnnotation sources:')
    print(df['Where'].value_counts(dropna=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(
    '/content/drive/MyDrive/external_cohorts/LNDb'
)
META = ROOT / 'metadata'
CSV_DIR = META / 'trainset_csv'

alln = pd.read_csv(META / 'allNods.csv')
gt = pd.read_csv(CSV_DIR / 'trainNodules_gt.csv')

alln['Malignancy'] = pd.to_numeric(
    alln['Malignancy'], errors='coerce'
)

# True when the finding has a radiologist annotation source.
alln['has_radiologist_annotation'] = (
    alln['Where'].fillna('')
    .str.contains('RadAnnotation', case=False)
)

# Match each allNods finding to its consensus finding.
keys = ['LNDbID', 'FindingID']
gt_cols = keys + ['Nodule', 'x', 'y', 'z', 'AgrLevel', 'Volume']

print('Duplicate consensus keys:', gt.duplicated(keys).sum())

joined = alln.merge(
    gt[gt_cols],
    on=keys,
    how='left',
    suffixes=('_all', '_gt'),
    indicator=True
)

joined['Nodule_gt'] = pd.to_numeric(
    joined['Nodule_gt'], errors='coerce'
)

print('\n=== Matching audit ===')
print('allNods rows:', len(alln))
print('Matched consensus findings:',
      int((joined['_merge'] == 'both').sum()))
print('Unmatched findings:',
      int((joined['_merge'] != 'both').sum()))
print('Matched, confirmed nodule findings:',
      int(((joined['_merge'] == 'both') &
           (joined['Nodule_gt'] == 1)).sum()))

print('\n=== Source types ===')
print(alln['Where'].value_counts(dropna=False).to_string())

print('\n=== Score distribution by annotation source ===')
print(pd.crosstab(
    alln['Where'].fillna('Missing source'),
    alln['Malignancy'].notna(),
    margins=True
).to_string())

# Exclude missing/invalid scores and non-nodules.
valid = joined[
    (joined['_merge'] == 'both') &
    (joined['Nodule_gt'] == 1) &
    joined['has_radiologist_annotation'] &
    joined['Malignancy'].between(1, 5)
].copy()

valid['proxy_label'] = np.where(
    valid['Malignancy'] <= 2, 0,
    np.where(valid['Malignancy'] >= 4, 1, -1)
)

print('\n=== Candidate risk-label counts ===')
print(valid['proxy_label'].map({
    0: 'Benign-risk proxy (score 1–2)',
    1: 'Malignant-risk proxy (score 4–5)',
    -1: 'Intermediate score (exclude)'
}).value_counts().to_string())

print('\nCandidate nodules with scores 0 or 5:')
print(
    joined.loc[
        joined['Malignancy'].isin([0, 5]),
        ['LNDbID', 'FindingID', 'Malignancy', 'Where']
    ].head(15).to_string(index=False)
)

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path(
    '/content/drive/MyDrive/external_cohorts/LNDb'
)
ARCHIVES = ROOT / 'raw_archives'

# Install a RAR listing/extraction utility in temporary Colab storage.
subprocess.run(
    ['apt-get', '-qq', 'update'],
    check=True
)
subprocess.run(
    ['apt-get', '-qq', 'install', '-y', 'unar'],
    check=True
)

print('=== Downloaded training archives ===')
for p in sorted(ARCHIVES.glob('data*.rar')):
    print(f'{p.name}: {p.stat().st_size / (1024**3):.2f} GiB')

# Inspect the first archive without extracting it.
archive = ARCHIVES / 'data0.rar'
assert archive.exists(), f'Archive not found: {archive}'

result = subprocess.run(
    ['lsar', str(archive)],
    capture_output=True,
    text=True,
    timeout=180
)

print('\n=== data0.rar archive listing ===')
print(result.stdout[:12000])

if result.returncode != 0:
    print('Listing error:')
    print(result.stderr[:3000])

In [ ]:
from pathlib import Path
import subprocess
import re
import pandas as pd

ROOT = Path('/content/drive/MyDrive/external_cohorts/LNDb')
ARCHIVES = ROOT / 'raw_archives'
IMAGE_ROOT = ROOT / 'images' / 'train'
LOG_DIR = ROOT / 'logs'

IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert subprocess.run(
    ['which', 'unar'],
    capture_output=True,
    text=True
).returncode == 0, 'unar is not installed.'

pattern = re.compile(r'(LNDb-\d+\.(?:mhd|raw))\s*$', re.I)
report = []

for i in range(6):
    name = f'data{i}.rar'
    archive = ARCHIVES / name
    out = IMAGE_ROOT / f'data{i}'
    out.mkdir(parents=True, exist_ok=True)

    if not archive.exists():
        raise FileNotFoundError(f'Missing archive: {archive}')

    print(f'\n=== {name} ===')

    listing = subprocess.run(
        ['lsar', str(archive)],
        capture_output=True,
        text=True,
        check=True,
        timeout=300
    ).stdout

    expected = sorted({
        match.group(1)
        for line in listing.splitlines()
        if (match := pattern.search(line.strip()))
    })

    if not expected:
        raise RuntimeError(f'No CT files listed in {name}')

    # Check recursively: unar may create a nested folder.
    def missing_files():
        found = {
            p.name
            for p in out.rglob('*')
            if p.is_file() and p.stat().st_size > 0
        }
        return [fname for fname in expected if fname not in found]

    missing = missing_files()
    print('Files expected:', len(expected))
    print('Files still missing:', len(missing))

    if missing:
        print('Extracting archive...')
        subprocess.run(
            ['unar', '-f', '-o', str(out), str(archive)],
            check=True,
            timeout=7200
        )

    missing = missing_files()

    mhd_count = sum(
        f.lower().endswith('.mhd') for f in expected
    )
    raw_count = sum(
        f.lower().endswith('.raw') for f in expected
    )

    status = 'OK' if not missing else 'INCOMPLETE'

    report.append({
        'archive': name,
        'expected_files': len(expected),
        'mhd_files': mhd_count,
        'raw_files': raw_count,
        'missing_files': len(missing),
        'status': status
    })

    print(f'MHD files: {mhd_count}; RAW files: {raw_count}')
    print('Status:', status)

    if missing:
        print('Missing examples:', missing[:10])
        pd.DataFrame(report).to_csv(
            LOG_DIR / 'training_extraction_status.csv',
            index=False
        )
        raise RuntimeError(
            f'{name} remains incomplete. See the log above.'
        )

    pd.DataFrame(report).to_csv(
        LOG_DIR / 'training_extraction_status.csv',
        index=False
    )

print('\n=== Extraction summary ===')
print(pd.DataFrame(report).to_string(index=False))
print('\nStatus log saved to:')
print(LOG_DIR / 'training_extraction_status.csv')

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path('/content/drive/MyDrive/external_cohorts/LNDb')
IMAGE_ROOT = ROOT / 'images' / 'train'
ARCHIVE = ROOT / 'raw_archives' / 'data0.rar'
OUT = IMAGE_ROOT / 'data0'

print('=== Check existing extraction ===')
print('Output folder exists:', OUT.exists())

# Search recursively in case files were extracted into a nested folder.
for name in ['LNDb-0001.mhd', 'LNDb-0001.raw', 'LNDb-0002.mhd']:
    matches = list(IMAGE_ROOT.rglob(name))
    print(f'\n{name}:')
    if matches:
        for p in matches[:5]:
            print(' FOUND:', p, f'({p.stat().st_size:,} bytes)')
    else:
        print(' NOT FOUND')

print('\n=== Contents of expected output folder ===')
if OUT.exists():
    for p in list(OUT.iterdir())[:15]:
        print(p.name, '/', 'folder' if p.is_dir() else 'file')

print('\n=== Test archive extraction ===')
result = subprocess.run(
    ['unar', '-f', '-o', str(OUT), str(ARCHIVE)],
    capture_output=True,
    text=True,
    timeout=1800
)

print('Exit code:', result.returncode)
print('STDOUT (last 5000 characters):')
print(result.stdout[-5000:])
print('STDERR (last 5000 characters):')
print(result.stderr[-5000:])

print('\n=== Verify after extraction attempt ===')
for name in ['LNDb-0001.mhd', 'LNDb-0001.raw']:
    matches = list(IMAGE_ROOT.rglob(name))
    print(name, 'FOUND' if matches else 'NOT FOUND')
    for p in matches[:3]:
        print(' ', p, f'({p.stat().st_size:,} bytes)')

In [ ]:
file_path = "/content/drive/MyDrive/external_cohorts/LNDb/raw_archives/data0.rar"
output_path = "/content/drive/MyDrive/external_cohorts/LNDb/extracted"

!mkdir -p "{output_path}"
! /tmp/7zip26/7zz x "{file_path}" -o"{output_path}" -y

In [ ]:
file_path = "/content/drive/MyDrive/external_cohorts/LNDb/raw_archives/data0.rar"
output_path = "/content/drive/MyDrive/external_cohorts/LNDb/extracted"

!mkdir -p "{output_path}"
! /tmp/7zip26/7zz x "{file_path}" -o"{output_path}" -y

In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'data0.rar' in files:
        print(os.path.join(root, 'data0.rar'))

In [ ]:
!wget -q https://github.com/ip7z/7zip/releases/download/26.04/7z2604-linux-x64.tar.xz \
    -O /tmp/7zip.tar.xz

!mkdir -p /tmp/7zip26
!tar -xJf /tmp/7zip.tar.xz -C /tmp/7zip26
!chmod +x /tmp/7zip26/7zz

! /tmp/7zip26/7zz

In [ ]:
file_path = "/content/drive/MyDrive/external_cohorts/LNDb/raw_archives/data0.rar"

! /tmp/7zip26/7zz t "{file_path}"

In [ ]:
!apt-get install -y p7zip-full

In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk

ROOT = Path('/content/drive/MyDrive/external_cohorts/LNDb')
META = ROOT / 'metadata'
CSV_DIR = META / 'trainset_csv'
IMAGE_ROOT = ROOT / 'images' / 'train'
LOG_DIR = ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load the annotated training-cohort tables.
alln = pd.read_csv(META / 'allNods.csv')
gt = pd.read_csv(CSV_DIR / 'trainNodules_gt.csv')
train_ct = pd.read_csv(CSV_DIR / 'trainCTs.csv')
acq = pd.read_csv(META / 'LNDbAcqParams.csv')

# Normalize IDs and numeric fields.
for df in (alln, gt, train_ct, acq):
    df['LNDbID'] = pd.to_numeric(
        df['LNDbID'], errors='coerce'
    )

alln['FindingID'] = pd.to_numeric(
    alln['FindingID'], errors='coerce'
)
gt['FindingID'] = pd.to_numeric(
    gt['FindingID'], errors='coerce'
)
gt['Nodule'] = pd.to_numeric(gt['Nodule'], errors='coerce')
alln['Malignancy'] = pd.to_numeric(
    alln['Malignancy'], errors='coerce'
)
alln['DiamEq_Rad'] = pd.to_numeric(
    alln['DiamEq_Rad'], errors='coerce'
)
gt['Volume'] = pd.to_numeric(gt['Volume'], errors='coerce')

train_ids = set(train_ct['LNDbID'].dropna().astype(int))

print('Annotated training CT IDs:', len(train_ids))

# 2. Find the extracted CT volumes recursively.
mhd_by_id = {}
duplicates = []

for p in IMAGE_ROOT.rglob('LNDb-*.mhd'):
    match = re.fullmatch(r'LNDb-(\d+)', p.stem, re.I)
    if not match:
        continue

    lid = int(match.group(1))
    if lid in mhd_by_id:
        duplicates.append((lid, str(mhd_by_id[lid]), str(p)))
    else:
        mhd_by_id[lid] = p

if duplicates:
    raise RuntimeError(
        'Duplicate CT IDs found; inspect before continuing: '
        + repr(duplicates[:5])
    )

missing_ct = sorted(train_ids - set(mhd_by_id))
extra_ct = sorted(set(mhd_by_id) - train_ids)

print('Extracted MHD volumes:', len(mhd_by_id))
print('Training CT IDs without images:', len(missing_ct))
print('Images not listed in trainCTs.csv:', len(extra_ct))

if missing_ct:
    raise RuntimeError(
        f'Missing training CT volumes: {missing_ct[:20]}'
    )

# 3. Match radiologist ratings to consensus finding coordinates.
keys = ['LNDbID', 'FindingID']

if alln.duplicated(keys).any():
    raise RuntimeError(
        'allNods.csv has duplicated LNDbID/FindingID keys. '
        'Resolve these before creating a manifest.'
    )

if gt.duplicated(keys).any():
    raise RuntimeError(
        'trainNodules_gt.csv has duplicated finding keys.'
    )

gt_cols = keys + [
    'x', 'y', 'z', 'Nodule', 'Volume', 'AgrLevel'
]

joined = alln.merge(
    gt[gt_cols],
    on=keys,
    how='inner',
    suffixes=('_all', '_gt'),
    validate='one_to_one'
)

# Use only annotated training cases, actual nodules, and
# findings with radiologist-assessed malignancy ratings.
source = joined['Where'].fillna('').str.contains(
    'RadAnnotation', case=False, regex=False
)

candidate = joined[
    joined['LNDbID'].isin(train_ids)
    & source
    & (joined['Nodule_gt'] == 1)
    & joined['Malignancy'].between(1, 5)
].copy()

# Fixed proxy-label rule:
# 1-2 = lower-risk/benign proxy
# 4-5 = higher-risk/malignant proxy
# scores between 2 and 4 are excluded.
candidate['label'] = np.where(
    candidate['Malignancy'] <= 2, 0,
    np.where(candidate['Malignancy'] >= 4, 1, -1)
)
candidate = candidate[candidate['label'].isin([0, 1])].copy()

# 4. Obtain a nominal diameter.
# Prefer the radiologist-equivalent diameter. Where unavailable,
# derive an equivalent-sphere diameter from consensus volume.
diam = candidate['DiamEq_Rad'].to_numpy(dtype=float)
volume = candidate['Volume'].to_numpy(dtype=float)

diam_from_volume = np.full(len(candidate), np.nan)
ok_vol = np.isfinite(volume) & (volume > 0)
diam_from_volume[ok_vol] = (
    2 * (3 * volume[ok_vol] / (4 * np.pi)) ** (1 / 3)
)

bad_diam = ~np.isfinite(diam) | (diam <= 0)
diam[bad_diam] = diam_from_volume[bad_diam]
candidate['diameter_mm'] = diam

candidate = candidate[
    np.isfinite(candidate['diameter_mm'])
    & (candidate['diameter_mm'] > 0)
    & candidate[['x_gt', 'y_gt', 'z_gt']].notna().all(axis=1)
].copy()

# 5. Create manifest records.
records = []

acq_cols = [
    'LNDbID', 'Manufacturer', 'ManufacturerModelName',
    'SliceThickness'
]
acq_small = acq[acq_cols].drop_duplicates('LNDbID')
acq_map = acq_small.set_index('LNDbID').to_dict('index')

for r in candidate.itertuples(index=False):
    lid = int(r.LNDbID)
    finding = int(r.FindingID)
    a = acq_map.get(lid, {})

    manufacturer = str(a.get('Manufacturer', '') or '')
    model = str(a.get('ManufacturerModelName', '') or '')
    scanner = ' '.join(
        s for s in (manufacturer, model)
        if s and s.lower() != 'nan'
    )

    records.append({
        'case_id': f'LNDb-{lid:04d}_F{finding:04d}',
        'patient_id': f'LNDb-{lid:04d}',
        'image_path': str(mhd_by_id[lid]),
        'world_x': float(r.x_gt),
        'world_y': float(r.y_gt),
        'world_z': float(r.z_gt),
        'label': int(r.label),
        'diameter_mm': float(r.diameter_mm),
        'scanner': scanner,
        'slice_thickness_mm': a.get('SliceThickness', np.nan),
        'reference_score': float(r.Malignancy),
        'reference_type': 'radiologist_assessed_risk_proxy'
    })

manifest = pd.DataFrame(records).sort_values(
    ['patient_id', 'case_id']
).reset_index(drop=True)

if manifest['case_id'].duplicated().any():
    raise RuntimeError('Duplicate case IDs in candidate manifest.')

# 6. Validate the physical coordinates using MHD headers.
# This reads image geometry without loading all CT voxel arrays.
geometry = {}
bad_coords = []

for row in manifest.itertuples(index=False):
    image_path = row.image_path

    if image_path not in geometry:
        reader = sitk.ImageFileReader()
        reader.SetFileName(image_path)
        reader.ReadImageInformation()

        geometry[image_path] = {
            'origin': np.asarray(reader.GetOrigin(), dtype=float),
            'spacing': np.asarray(reader.GetSpacing(), dtype=float),
            'direction': np.asarray(
                reader.GetDirection(), dtype=float
            ).reshape(3, 3),
            'size': np.asarray(reader.GetSize(), dtype=float)
        }

    g = geometry[image_path]
    world = np.array(
        [row.world_x, row.world_y, row.world_z], dtype=float
    )

    # Convert physical coordinates to continuous voxel index (x,y,z).
    idx = np.linalg.solve(
        g['direction'], world - g['origin']
    ) / g['spacing']

    # Allow coordinates within the physical extent of edge voxels.
    valid = np.all(idx >= -0.5) and np.all(
        idx <= g['size'] - 0.5
    )

    if not valid:
        bad_coords.append({
            'case_id': row.case_id,
            'image_path': image_path,
            'world_x': row.world_x,
            'world_y': row.world_y,
            'world_z': row.world_z,
            'index_x': idx[0],
            'index_y': idx[1],
            'index_z': idx[2],
            'size_x': g['size'][0],
            'size_y': g['size'][1],
            'size_z': g['size'][2]
        })

bad_path = LOG_DIR / 'coordinate_audit_failures.csv'

print('\n=== Manifest audit ===')
print('Candidate nodules:', len(manifest))
print('Patients:', manifest['patient_id'].nunique())
print('Benign-risk proxy:', int((manifest['label'] == 0).sum()))
print('Malignant-risk proxy:', int((manifest['label'] == 1).sum()))
print('Unique CT volumes:', manifest['image_path'].nunique())
print('Invalid physical coordinates:', len(bad_coords))

if bad_coords:
    pd.DataFrame(bad_coords).to_csv(bad_path, index=False)
    print('Coordinate problems saved to:', bad_path)
    raise RuntimeError(
        'Manifest NOT saved because some coordinates fall outside '
        'their CT volumes. Inspect the coordinate audit first.'
    )

# Save only after the coordinate audit passes.
out = ROOT / 'manifest.csv'
manifest.to_csv(out, index=False)

print('\nManifest saved successfully:', out)
print('\nLabel counts:')
print(manifest['label'].value_counts().sort_index())
print('\nPreview:')
print(manifest.head().to_string(index=False))


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/external_cohorts/LNDb')
manifest = pd.read_csv(ROOT / 'manifest.csv')

sample = manifest.sample(
    n=min(6, len(manifest)),
    random_state=42
).reset_index(drop=True)

fig, axes = plt.subplots(6, 3, figsize=(10, 18))

for i, row in sample.iterrows():
    img = sitk.ReadImage(row.image_path)
    vol = sitk.GetArrayFromImage(img).astype(np.float32)

    spacing_zyx = np.asarray(img.GetSpacing())[::-1]
    centre_zyx = np.asarray(
        img.TransformPhysicalPointToContinuousIndex((
            float(row.world_x),
            float(row.world_y),
            float(row.world_z)
        ))
    )[::-1]

    hu, _ = make_patch(
        vol,
        np.zeros(vol.shape, dtype=bool),
        centre_zyx,
        spacing_zyx
    )

    mask = auto_mask(hu, row.diameter_mm)

    # Three orthogonal centre slices.
    views = [
        (hu[32, :, :], mask[32, :, :]),
        (hu[:, 32, :], mask[:, 32, :]),
        (hu[:, :, 32], mask[:, :, 32])
    ]

    for j, (image_slice, mask_slice) in enumerate(views):
        ax = axes[i, j]
        ax.imshow(
            image_slice, cmap='gray',
            vmin=-1000, vmax=400
        )
        if mask_slice.any():
            ax.contour(
                mask_slice.astype(float),
                levels=[0.5], colors='red'
            )
        ax.axis('off')

    axes[i, 0].set_title(
        f'{row.case_id} | label={row.label} '
        f'| score={row.reference_score}'
    )

plt.suptitle(
    'LNDb coordinate and automatic-mask sanity check',
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MASK_DIR = ROOT / "masks_sample" / "masks"

manifest = pd.read_csv(ROOT / "manifest.csv")

# case_id: (radiologist ID, finding label in that radiologist mask)
targets = {
    "LNDb-0304_F0006": (2, 4),
    "LNDb-0035_F0008": (2, 3),
    "LNDb-0002_F0006": (2, 2),
    "LNDb-0111_F0003": (2, 1),
    "LNDb-0035_F0004": (1, 4),
    "LNDb-0183_F0008": (2, 5),
}


def make_auto_mask(hu, diameter_mm, thr=-500):
    """Reproduce auto_mask and report whether the fallback was used."""
    zz, yy, xx = np.indices(hu.shape)
    r = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    fallback = r <= max(float(diameter_mm) / 2, 2.5)
    radius = max(float(diameter_mm) * 0.75 + 1, 4)

    labels, n = ndimage.label(
        (hu > thr) & (r <= radius)
    )

    if n == 0:
        return fallback, "fallback_no_components"

    central = labels[28:37, 28:37, 28:37]
    ids = central[central > 0]

    if len(ids) == 0:
        return fallback, "fallback_no_central_component"

    selected = np.bincount(ids).argmax()

    candidate = ndimage.binary_fill_holes(
        ndimage.binary_opening(
            labels == selected,
            iterations=1
        )
    )

    if candidate.sum() < 20:
        return fallback, "fallback_component_too_small"

    return candidate, "threshold_component"


results = []
plot_data = []

for case_id, (rad_id, finding_id) in targets.items():
    rows = manifest[manifest["case_id"] == case_id]

    if len(rows) != 1:
        print(f"Skipping {case_id}: found {len(rows)} manifest rows")
        continue

    row = rows.iloc[0]
    ct_path = Path(row["image_path"])
    ref_path = (
        MASK_DIR /
        f"{row['patient_id']}_rad{rad_id}.mhd"
    )

    if not ct_path.exists() or not ref_path.exists():
        print(f"Skipping {case_id}: CT or reference mask not found")
        continue

    # Load image and radiologist mask.
    ct_img = sitk.ReadImage(str(ct_path))
    ref_img = sitk.ReadImage(str(ref_path))

    geometry_matches = (
        ct_img.GetSize() == ref_img.GetSize()
        and np.allclose(ct_img.GetSpacing(), ref_img.GetSpacing())
        and np.allclose(ct_img.GetOrigin(), ref_img.GetOrigin())
        and np.allclose(ct_img.GetDirection(), ref_img.GetDirection())
    )

    # Align mask to CT grid if needed.
    if not geometry_matches:
        ref_img = sitk.Resample(
            ref_img,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            ref_img.GetPixelID()
        )

    ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)
    ref_full = (
        sitk.GetArrayFromImage(ref_img) == finding_id
    )

    world_xyz = (
        float(row["world_x"]),
        float(row["world_y"]),
        float(row["world_z"])
    )

    x, y, z = ct_img.TransformPhysicalPointToIndex(world_xyz)

    if not (
        0 <= x < ct.shape[2]
        and 0 <= y < ct.shape[1]
        and 0 <= z < ct.shape[0]
    ):
        print(f"Skipping {case_id}: coordinate outside CT")
        continue

    centre = (z, y, x)
    spacing_zyx = np.asarray(
        ct_img.GetSpacing(), dtype=float
    )[::-1]

    # Use exactly the existing patch preprocessing.
    hu_patch, ref_patch = make_patch(
        ct, ref_full, centre, spacing_zyx
    )
    ref_patch = ref_patch.astype(bool)

    auto_patch, method = make_auto_mask(
        hu_patch, float(row["diameter_mm"])
    )

    ref_count = int(ref_patch.sum())
    auto_count = int(auto_patch.sum())
    intersection = int(np.logical_and(ref_patch, auto_patch).sum())

    denom = ref_count + auto_count
    dice = 2 * intersection / denom if denom else np.nan

    if ref_count < 20:
        status = "Sparse reference: Dice not reliable"
    elif ref_count <= 25:
        status = "Very small reference: interpret cautiously"
    else:
        status = "Exploratory overlap result"

    results.append({
        "case_id": case_id,
        "radiologist": rad_id,
        "finding_label": finding_id,
        "geometry_matched_before_resampling": geometry_matches,
        "reference_voxels_in_patch": ref_count,
        "automatic_voxels": auto_count,
        "intersection_voxels": intersection,
        "dice": dice,
        "automatic_mask_method": method,
        "interpretation": status,
    })

    # Keep display arrays only for this small six-case diagnostic.
    plot_data.append((
        case_id, hu_patch.copy(), ref_patch.copy(),
        auto_patch.copy(), dice, method, ref_count
    ))

    print(
        f"{case_id}: Dice={dice:.4f}, "
        f"reference={ref_count}, automatic={auto_count}, "
        f"method={method}"
    )

    del ct, ref_full, ref_patch, auto_patch, hu_patch
    del ct_img, ref_img

# Save the results.
results_df = pd.DataFrame(results)
out_path = ROOT / "mask_reference_comparison_6.csv"
results_df.to_csv(out_path, index=False)

print("\n=== SUMMARY ===")
display(results_df)
print("Saved:", out_path)

# Plot all six cases: one row per case, three orthogonal views.
fig, axes = plt.subplots(
    len(plot_data), 3,
    figsize=(13, 4 * len(plot_data)),
    squeeze=False
)

for i, (case_id, hu, ref, auto, dice, method, ref_count) in enumerate(plot_data):
    # Display slices near the reference-mask centroid if possible.
    if ref.any():
        cz, cy, cx = np.rint(
            np.argwhere(ref).mean(axis=0)
        ).astype(int)
    else:
        cz, cy, cx = np.array(hu.shape) // 2

    views = [
        ("Axial", hu[cz], ref[cz], auto[cz]),
        ("Coronal", hu[:, cy, :], ref[:, cy, :], auto[:, cy, :]),
        ("Sagittal", hu[:, :, cx], ref[:, :, cx], auto[:, :, cx]),
    ]

    for j, (name, image, green, red) in enumerate(views):
        ax = axes[i, j]
        ax.imshow(
            image, cmap="gray",
            vmin=HU_MIN, vmax=HU_MAX,
            origin="lower"
        )

        if green.any() and not green.all():
            ax.contour(
                green.astype(float), levels=[0.5],
                colors="lime", linewidths=1.3
            )

        if red.any() and not red.all():
            ax.contour(
                red.astype(float), levels=[0.5],
                colors="red", linewidths=1.3
            )

        ax.axis("off")
        ax.set_title(
            f"{name}" if j else
            f"{case_id}\nDice={dice:.3f} | Ref voxels={ref_count}\n{method}"
        )

plt.tight_layout()
plt.show()

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")

manifest = pd.read_csv(ROOT / "manifest.csv")
nodules = pd.read_csv(
    ROOT / "metadata" / "trainset_csv" / "trainNodules.csv"
)

MIN_VOLUME_MM3 = 100.0
MAX_DISTANCE_MM = 5.0
MAX_PATIENTS = 15

candidates = []

for _, row in manifest.iterrows():
    match = re.search(r"LNDb-(\d+)", str(row["patient_id"]))
    if not match:
        continue

    patient_num = int(match.group(1))

    xyz = np.array([
        row["world_x"],
        row["world_y"],
        row["world_z"]
    ], dtype=float)

    ann = nodules[
        (nodules["LNDbID"] == patient_num) &
        (nodules["Nodule"] == 1)
    ].copy()

    if ann.empty:
        continue

    ann[["x", "y", "z", "Volume"]] = ann[
        ["x", "y", "z", "Volume"]
    ].apply(pd.to_numeric, errors="coerce")

    ann = ann.dropna(subset=["x", "y", "z", "Volume"])
    ann = ann[ann["Volume"] >= MIN_VOLUME_MM3]

    if ann.empty:
        continue

    ann["distance_mm"] = np.linalg.norm(
        ann[["x", "y", "z"]].to_numpy() - xyz,
        axis=1
    )

    ann = ann[ann["distance_mm"] <= MAX_DISTANCE_MM]

    if ann.empty:
        continue

    # Prefer the closest eligible annotation.
    best = ann.sort_values(
        ["distance_mm", "Volume"],
        ascending=[True, False]
    ).iloc[0]

    candidates.append({
        "case_id": row["case_id"],
        "patient_id": row["patient_id"],
        "image_path": row["image_path"],
        "RadID": int(best["RadID"]),
        "FindingID": int(best["FindingID"]),
        "annotation_volume_mm3": float(best["Volume"]),
        "coordinate_distance_mm": float(best["distance_mm"]),
        "reference_score": row["reference_score"],
        "label": row["label"],
    })

candidate_df = pd.DataFrame(candidates)

if candidate_df.empty:
    print(
        "No candidates met these criteria. "
        "We can relax the volume or distance threshold."
    )
else:
    candidate_df = candidate_df.sort_values(
        ["annotation_volume_mm3", "coordinate_distance_mm"],
        ascending=[False, True]
    )

    # One candidate per patient to avoid repeatedly extracting the same scan.
    candidate_df = (
        candidate_df
        .drop_duplicates("patient_id")
        .head(MAX_PATIENTS)
        .reset_index(drop=True)
    )

    output = ROOT / "mask_candidates.csv"
    candidate_df.to_csv(output, index=False)

    print("Candidate nodules:", len(candidate_df))
    print("Saved to:", output)
    display(candidate_df)

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
ARCHIVE = ROOT / "raw_archives" / "masks.rar"
CANDIDATES_FILE = ROOT / "mask_candidates.csv"
MASK_DIR = ROOT / "mask_candidates_ref"

candidates = pd.read_csv(CANDIDATES_FILE)

# Find the previously installed RAR utility.
if "UNRAR" not in globals() or not Path(UNRAR).exists():
    options = list(Path("/content/rar_tools").rglob("unrar"))
    if not options:
        raise FileNotFoundError(
            "UnRAR is not available. Re-run the earlier extractor-install cell."
        )
    UNRAR = str(options[0])

if not ARCHIVE.exists():
    raise FileNotFoundError(f"Archive not found: {ARCHIVE}")

MASK_DIR.mkdir(parents=True, exist_ok=True)

# Read archive filenames; this does not extract the archive.
listing = subprocess.run(
    [UNRAR, "lb", str(ARCHIVE)],
    capture_output=True,
    text=True,
    check=True
)
available = set(listing.stdout.splitlines())

targets = []
for _, row in candidates.iterrows():
    patient = str(row["patient_id"])
    rad_id = int(row["RadID"])

    for extension in (".mhd", ".raw"):
        member = f"masks/{patient}_rad{rad_id}{extension}"
        if member in available:
            targets.append(member)
        else:
            print("Not present in archive:", member)

targets = sorted(set(targets))
print("Files to extract:", len(targets))

if not targets:
    raise RuntimeError("No matching reference-mask files were found.")

# Extract only the required files.
subprocess.run(
    [UNRAR, "x", "-o+", str(ARCHIVE), *targets, str(MASK_DIR) + "/"],
    check=True
)

headers = list(MASK_DIR.rglob("*.mhd"))
print("\nReference mask headers extracted:", len(headers))
for path in headers:
    print(path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MASK_DIR = ROOT / "mask_candidates_ref" / "masks"
CANDIDATES_FILE = ROOT / "mask_candidates.csv"

OUT_DIR = ROOT / "mask_candidate_qc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidates = pd.read_csv(CANDIDATES_FILE)


def auto_mask_with_status(hu, diameter_mm, thr=-500):
    """Same automatic-mask logic, returning the selected pathway."""
    zz, yy, xx = np.indices(hu.shape)
    r = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    fallback = r <= max(float(diameter_mm) / 2, 2.5)
    radius = max(float(diameter_mm) * 0.75 + 1, 4)

    labels, n = ndimage.label(
        (hu > thr) & (r <= radius)
    )

    if n == 0:
        return fallback, "fallback_no_components"

    central = labels[28:37, 28:37, 28:37]
    ids = central[central > 0]

    if len(ids) == 0:
        return fallback, "fallback_no_central_component"

    selected = np.bincount(ids).argmax()

    candidate = ndimage.binary_fill_holes(
        ndimage.binary_opening(
            labels == selected,
            iterations=1
        )
    )

    if candidate.sum() < 20:
        return fallback, "fallback_component_too_small"

    return candidate, "threshold_component"


results = []

for _, row in candidates.iterrows():
    case_id = str(row["case_id"])
    patient_id = str(row["patient_id"])
    rad_id = int(row["RadID"])
    finding_id = int(row["FindingID"])

    ct_path = Path(row["image_path"])
    ref_path = MASK_DIR / f"{patient_id}_rad{rad_id}.mhd"

    if not ct_path.exists() or not ref_path.exists():
        print(f"SKIPPED {case_id}: CT or mask is missing")
        continue

    ct_img = sitk.ReadImage(str(ct_path))
    ref_img = sitk.ReadImage(str(ref_path))

    geometry_matches = (
        ct_img.GetSize() == ref_img.GetSize()
        and np.allclose(ct_img.GetSpacing(), ref_img.GetSpacing())
        and np.allclose(ct_img.GetOrigin(), ref_img.GetOrigin())
        and np.allclose(ct_img.GetDirection(), ref_img.GetDirection())
    )

    if not geometry_matches:
        ref_img = sitk.Resample(
            ref_img,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            ref_img.GetPixelID()
        )

    ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)
    ref_full = (
        sitk.GetArrayFromImage(ref_img) == finding_id
    )

    # Physical coordinates -> CT array coordinates (z, y, x).
    world_xyz = (
        float(row["world_x"]) if "world_x" in row else np.nan,
        float(row["world_y"]) if "world_y" in row else np.nan,
        float(row["world_z"]) if "world_z" in row else np.nan
    )

    # Candidate CSV normally does not contain world coordinates.
    # Retrieve them from the manifest if needed.
    if not np.all(np.isfinite(world_xyz)):
        manifest_row = pd.read_csv(ROOT / "manifest.csv")
        match = manifest_row[
            manifest_row["case_id"] == case_id
        ]
        if len(match) != 1:
            print(f"SKIPPED {case_id}: manifest row is ambiguous")
            continue

        mr = match.iloc[0]
        world_xyz = (
            float(mr["world_x"]),
            float(mr["world_y"]),
            float(mr["world_z"])
        )
        diameter_mm = float(mr["diameter_mm"])
    else:
        manifest_row = pd.read_csv(ROOT / "manifest.csv")
        match = manifest_row[
            manifest_row["case_id"] == case_id
        ]
        if len(match) != 1:
            print(f"SKIPPED {case_id}: manifest row is ambiguous")
            continue
        diameter_mm = float(match.iloc[0]["diameter_mm"])

    try:
        x, y, z = ct_img.TransformPhysicalPointToIndex(world_xyz)
    except RuntimeError:
        print(f"SKIPPED {case_id}: coordinate conversion failed")
        continue

    if not (
        0 <= x < ct.shape[2]
        and 0 <= y < ct.shape[1]
        and 0 <= z < ct.shape[0]
    ):
        print(f"SKIPPED {case_id}: coordinate outside CT")
        continue

    centre_zyx = (z, y, x)
    spacing_zyx = np.asarray(
        ct_img.GetSpacing(), dtype=float
    )[::-1]

    # Use your existing preprocessing on the CT and reference mask.
    hu_patch, ref_patch = make_patch(
        ct, ref_full, centre_zyx, spacing_zyx
    )
    ref_patch = ref_patch.astype(bool)

    auto_patch, method = auto_mask_with_status(
        hu_patch, diameter_mm
    )

    ref_n = int(ref_patch.sum())
    auto_n = int(auto_patch.sum())
    intersection = int(np.logical_and(ref_patch, auto_patch).sum())

    denom = ref_n + auto_n
    dice = (2.0 * intersection / denom) if denom else np.nan

    native_voxel_volume = float(np.prod(ct_img.GetSpacing()))
    native_reference_voxels = int(ref_full.sum())
    native_reference_volume_mm3 = (
        native_reference_voxels * native_voxel_volume
    )

    # A sparse reference is not an adequate segmentation benchmark.
    if ref_n < 20:
        interpretation = "Sparse reference; Dice not reliable"
    else:
        interpretation = "Usable for exploratory overlap assessment"

    results.append({
        "case_id": case_id,
        "RadID": rad_id,
        "FindingID": finding_id,
        "reported_volume_mm3": float(row["annotation_volume_mm3"]),
        "native_reference_voxels": native_reference_voxels,
        "native_mask_volume_mm3": native_reference_volume_mm3,
        "reference_voxels_in_patch": ref_n,
        "automatic_voxels": auto_n,
        "intersection_voxels": intersection,
        "dice": dice,
        "automatic_mask_method": method,
        "fallback_used": method.startswith("fallback"),
        "geometry_matched_before_resampling": geometry_matches,
        "interpretation": interpretation
    })

    # Create three-view overlay and save one PNG per case.
    if ref_patch.any():
        cz, cy, cx = np.rint(
            np.argwhere(ref_patch).mean(axis=0)
        ).astype(int)
    else:
        cz, cy, cx = np.array(hu_patch.shape) // 2

    views = [
        ("Axial", hu_patch[cz], ref_patch[cz], auto_patch[cz]),
        ("Coronal", hu_patch[:, cy, :],
         ref_patch[:, cy, :], auto_patch[:, cy, :]),
        ("Sagittal", hu_patch[:, :, cx],
         ref_patch[:, :, cx], auto_patch[:, :, cx])
    ]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, (view_name, image, green, red) in zip(axes, views):
        ax.imshow(
            image, cmap="gray",
            vmin=HU_MIN, vmax=HU_MAX,
            origin="lower"
        )

        if green.any() and not green.all():
            ax.contour(
                green.astype(float), levels=[0.5],
                colors="lime", linewidths=1.3
            )

        if red.any() and not red.all():
            ax.contour(
                red.astype(float), levels=[0.5],
                colors="red", linewidths=1.3
            )

        ax.set_title(view_name)
        ax.axis("off")

    fig.suptitle(
        f"{case_id} | Finding {finding_id} | "
        f"Dice={dice:.3f} | {method}"
    )
    plt.tight_layout()
    fig.savefig(
        OUT_DIR / f"{case_id}_overlay.png",
        dpi=130,
        bbox_inches="tight"
    )
    plt.close(fig)

    print(
        f"{case_id}: Dice={dice:.3f}, ref={ref_n}, "
        f"auto={auto_n}, method={method}"
    )

    del ct, ref_full, ref_patch, auto_patch, hu_patch
    del ct_img, ref_img


results_df = pd.DataFrame(results)
report_path = ROOT / "mask_candidate_comparison.csv"
results_df.to_csv(report_path, index=False)

print("\n=== COMPARISON SUMMARY ===")
if len(results_df):
    display(
        results_df[
            [
                "case_id",
                "reported_volume_mm3",
                "reference_voxels_in_patch",
                "automatic_voxels",
                "dice",
                "automatic_mask_method",
                "interpretation"
            ]
        ]
    )

    evaluable = results_df[
        results_df["reference_voxels_in_patch"] >= 20
    ]

    print("\nTotal cases processed:", len(results_df))
    print("Cases with at least 20 reference voxels:", len(evaluable))
    print(
        "Fallback masks:",
        int(results_df["fallback_used"].sum()),
        "/",
        len(results_df)
    )

    if len(evaluable):
        print(
            "Median Dice on exploratory evaluable cases:",
            round(evaluable["dice"].median(), 4)
        )
        print(
            "Dice IQR:",
            round(evaluable["dice"].quantile(0.25), 4),
            "to",
            round(evaluable["dice"].quantile(0.75), 4)
        )

print("\nCSV:", report_path)
print("Overlay images:", OUT_DIR)

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

OUT_DIR = Path(
    "/content/drive/MyDrive/external_cohorts/LNDb/mask_candidate_qc"
)

cases = [
    "LNDb-0066_F0001",
    "LNDb-0060_F0005",
    "LNDb-0119_F0002",
    "LNDb-0151_F0001",
]

for case_id in cases:
    path = OUT_DIR / f"{case_id}_overlay.png"

    print(f"\n{case_id}")

    if not path.exists():
        print("Overlay not found:", path)
        continue

    display(Image.open(path))

In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MASK_DIR = ROOT / "mask_candidates_ref" / "masks"

case_id = "LNDb-0066_F0001"
rad_id = 1
finding_id = 1

manifest = pd.read_csv(ROOT / "manifest.csv")
row = manifest.loc[manifest["case_id"] == case_id].iloc[0]

ct_img = sitk.ReadImage(str(row["image_path"]))
ref_img = sitk.ReadImage(
    str(MASK_DIR / f"{row['patient_id']}_rad{rad_id}.mhd")
)

same_geometry = (
    ct_img.GetSize() == ref_img.GetSize()
    and np.allclose(ct_img.GetSpacing(), ref_img.GetSpacing())
    and np.allclose(ct_img.GetOrigin(), ref_img.GetOrigin())
    and np.allclose(ct_img.GetDirection(), ref_img.GetDirection())
)

if not same_geometry:
    ref_img = sitk.Resample(
        ref_img, ct_img, sitk.Transform(),
        sitk.sitkNearestNeighbor, 0, ref_img.GetPixelID()
    )

ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)
ref_full = sitk.GetArrayFromImage(ref_img) == finding_id

world_xyz = (
    float(row["world_x"]),
    float(row["world_y"]),
    float(row["world_z"])
)
x, y, z = ct_img.TransformPhysicalPointToIndex(world_xyz)

centre = (z, y, x)
spacing_zyx = np.asarray(ct_img.GetSpacing())[::-1]

hu, ref = make_patch(ct, ref_full, centre, spacing_zyx)
ref = ref.astype(bool)

# Reproduce the automatic segmentation stages.
zz, yy, xx = np.indices(hu.shape)
r = np.sqrt((zz - 32)**2 + (yy - 32)**2 + (xx - 32)**2)

diameter = float(row["diameter_mm"])
radius = max(diameter * 0.75 + 1, 4)

eligible = (hu > -500) & (r <= radius)
labels, n = ndimage.label(eligible)

central = labels[28:37, 28:37, 28:37]
ids = central[central > 0]

print("Case:", case_id)
print("Diameter (mm):", round(diameter, 2))
print("Reference voxels:", int(ref.sum()))
print("Reference centroid (z,y,x):",
      np.argwhere(ref).mean(axis=0).round(2))
print("Patch centre:", np.array(hu.shape) / 2 - 0.5)
print("Distance from reference centroid to patch centre:",
      round(float(np.linalg.norm(
          np.argwhere(ref).mean(axis=0) -
          (np.array(hu.shape) / 2 - 0.5)
      )), 2), "voxels")

print("\nReference-mask HU statistics:")
ref_hu = hu[ref]
print("Min / median / max:",
      np.min(ref_hu), np.median(ref_hu), np.max(ref_hu))
print("Fraction above -500 HU:",
      round(float(np.mean(ref_hu > -500)), 3))
print("Fraction within automatic search radius:",
      round(float(np.mean(r[ref] <= radius)), 3))
print("Connected components inside search region:", n)

if len(ids):
    selected = int(np.bincount(ids).argmax())
    raw_component = labels == selected

    cleaned = ndimage.binary_fill_holes(
        ndimage.binary_opening(raw_component, iterations=1)
    )

    print("\nSelected component voxels:", int(raw_component.sum()))
    print("After morphological cleaning:", int(cleaned.sum()))
    print("Reference ∩ raw component:",
          int(np.logical_and(ref, raw_component).sum()))
    print("Reference ∩ cleaned component:",
          int(np.logical_and(ref, cleaned).sum()))

    for name, candidate in [
        ("raw component", raw_component),
        ("cleaned component", cleaned)
    ]:
        denom = int(ref.sum()) + int(candidate.sum())
        dice = (
            2 * np.logical_and(ref, candidate).sum() / denom
            if denom else np.nan
        )
        print(f"Dice against {name}: {dice:.4f}")
else:
    print("No component intersects the central selection region.")

del ct, ref_full, ct_img, ref_img, hu, ref

In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MASK_DIR = ROOT / "mask_candidates_ref" / "masks"

candidates = pd.read_csv(ROOT / "mask_candidates.csv")
manifest = pd.read_csv(ROOT / "manifest.csv")

manifest_by_case = manifest.set_index("case_id")
results = []


def dice_score(a, b):
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)
    denom = int(a.sum()) + int(b.sum())

    if denom == 0:
        return np.nan

    return 2.0 * np.logical_and(a, b).sum() / denom


for _, candidate in candidates.iterrows():
    case_id = str(candidate["case_id"])

    if case_id not in manifest_by_case.index:
        print("Skipping: no manifest row for", case_id)
        continue

    row = manifest_by_case.loc[case_id]

    if isinstance(row, pd.DataFrame):
        raise ValueError(f"Duplicate manifest entries for {case_id}")

    rad_id = int(candidate["RadID"])
    finding_id = int(candidate["FindingID"])

    ct_path = Path(row["image_path"])
    ref_path = MASK_DIR / (
        f"{candidate['patient_id']}_rad{rad_id}.mhd"
    )

    if not ct_path.exists() or not ref_path.exists():
        print("Skipping missing CT/reference:", case_id)
        continue

    ct_img = sitk.ReadImage(str(ct_path))
    ref_img = sitk.ReadImage(str(ref_path))

    geometry_matches = (
        ct_img.GetSize() == ref_img.GetSize()
        and np.allclose(ct_img.GetSpacing(), ref_img.GetSpacing())
        and np.allclose(ct_img.GetOrigin(), ref_img.GetOrigin())
        and np.allclose(ct_img.GetDirection(), ref_img.GetDirection())
    )

    if not geometry_matches:
        ref_img = sitk.Resample(
            ref_img,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            ref_img.GetPixelID()
        )

    ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)
    reference_full = (
        sitk.GetArrayFromImage(ref_img) == finding_id
    )

    world_xyz = (
        float(row["world_x"]),
        float(row["world_y"]),
        float(row["world_z"])
    )

    x, y, z = ct_img.TransformPhysicalPointToIndex(world_xyz)

    if not (
        0 <= x < ct.shape[2]
        and 0 <= y < ct.shape[1]
        and 0 <= z < ct.shape[0]
    ):
        print("Skipping coordinate outside CT:", case_id)
        continue

    spacing_zyx = np.asarray(
        ct_img.GetSpacing(), dtype=float
    )[::-1]

    hu, reference = make_patch(
        ct,
        reference_full,
        (z, y, x),
        spacing_zyx
    )
    reference = reference.astype(bool)

    # Same threshold, search radius and connected-component logic
    # as the current auto_mask implementation.
    zz, yy, xx = np.indices(hu.shape)
    radius_from_center = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    diameter = float(row["diameter_mm"])
    fallback = radius_from_center <= max(diameter / 2, 2.5)
    search_radius = max(diameter * 0.75 + 1, 4)

    eligible = (
        (hu > -500) &
        (radius_from_center <= search_radius)
    )

    labels, number_components = ndimage.label(eligible)

    raw_component = np.zeros_like(reference, dtype=bool)
    cleaned_component = np.zeros_like(reference, dtype=bool)
    fallback_reason = ""

    if number_components == 0:
        fallback_reason = "no_components"

    else:
        central = labels[28:37, 28:37, 28:37]
        ids = central[central > 0]

        if len(ids) == 0:
            fallback_reason = "no_central_component"
        else:
            selected = int(np.bincount(ids).argmax())
            raw_component = labels == selected

            cleaned_component = ndimage.binary_fill_holes(
                ndimage.binary_opening(
                    raw_component,
                    iterations=1
                )
            )

            if cleaned_component.sum() < 20:
                fallback_reason = "component_too_small"

    # Reproduce the exact final-mask decision.
    if fallback_reason:
        final_mask = fallback.copy()
        final_method = "fallback_" + fallback_reason
    else:
        final_mask = cleaned_component.copy()
        final_method = "threshold_component"

    raw_dice = (
        dice_score(reference, raw_component)
        if raw_component.any() else np.nan
    )

    cleaned_dice = (
        dice_score(reference, cleaned_component)
        if cleaned_component.any() else np.nan
    )

    final_dice = dice_score(reference, final_mask)

    # How much of the raw component's true overlap survived cleaning?
    raw_overlap = int(
        np.logical_and(reference, raw_component).sum()
    )
    cleaned_overlap = int(
        np.logical_and(reference, cleaned_component).sum()
    )

    results.append({
        "case_id": case_id,
        "reference_voxels": int(reference.sum()),
        "raw_component_voxels": int(raw_component.sum()),
        "cleaned_component_voxels": int(cleaned_component.sum()),
        "raw_overlap_voxels": raw_overlap,
        "cleaned_overlap_voxels": cleaned_overlap,
        "raw_dice": raw_dice,
        "cleaned_dice": cleaned_dice,
        "final_dice": final_dice,
        "cleaning_dice_change": (
            cleaned_dice - raw_dice
            if np.isfinite(cleaned_dice)
            and np.isfinite(raw_dice) else np.nan
        ),
        "final_method": final_method,
        "geometry_matched_before_resampling": geometry_matches
    })

    print(
        f"{case_id}: raw={raw_dice:.3f}, "
        f"cleaned={cleaned_dice:.3f}, "
        f"final={final_dice:.3f}, "
        f"method={final_method}"
    )

    del ct, reference_full, reference, hu
    del ct_img, ref_img


report = pd.DataFrame(results)
out_path = ROOT / "mask_cleaning_diagnostic_15.csv"
report.to_csv(out_path, index=False)

print("\n=== PER-CASE RESULTS ===")
display(
    report.sort_values("cleaning_dice_change")[
        [
            "case_id",
            "reference_voxels",
            "raw_component_voxels",
            "cleaned_component_voxels",
            "raw_dice",
            "cleaned_dice",
            "final_dice",
            "cleaning_dice_change",
            "final_method"
        ]
    ]
)

print("\n=== OVERALL COMPARISON ===")

valid = report.dropna(subset=["raw_dice", "cleaned_dice"])

print("Cases analysed:", len(report))
print("Cases with comparable raw and cleaned masks:", len(valid))

if len(valid):
    print("Median raw Dice:", round(valid["raw_dice"].median(), 4))
    print("Median cleaned Dice:", round(valid["cleaned_dice"].median(), 4))
    print(
        "Cases where cleaning reduced Dice:",
        int((valid["cleaning_dice_change"] < 0).sum()),
        "of",
        len(valid)
    )
    print(
        "Cases where cleaning increased Dice:",
        int((valid["cleaning_dice_change"] > 0).sum()),
        "of",
        len(valid)
    )

print("\nSaved report:", out_path)

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
manifest = pd.read_csv(ROOT / "manifest.csv")
nodules = pd.read_csv(
    ROOT / "metadata" / "trainset_csv" / "trainNodules.csv"
)

# Convert annotation fields to numeric values.
for col in ["LNDbID", "RadID", "FindingID", "Nodule",
            "x", "y", "z", "Volume"]:
    nodules[col] = pd.to_numeric(nodules[col], errors="coerce")

pool = []

for _, row in manifest.iterrows():
    patient_match = re.search(r"LNDb-(\d+)", str(row["patient_id"]))
    if not patient_match:
        continue

    patient_num = int(patient_match.group(1))
    xyz = np.array(
        [row["world_x"], row["world_y"], row["world_z"]],
        dtype=float
    )

    ann = nodules[
        (nodules["LNDbID"] == patient_num) &
        (nodules["Nodule"] == 1) &
        (nodules["Volume"] >= 100)
    ].dropna(subset=["x", "y", "z", "Volume"]).copy()

    if ann.empty:
        continue

    ann["distance_mm"] = np.linalg.norm(
        ann[["x", "y", "z"]].to_numpy() - xyz,
        axis=1
    )

    ann = ann[ann["distance_mm"] <= 5]
    if ann.empty:
        continue

    # Use the closest matching eligible annotation.
    best = ann.sort_values(
        ["distance_mm", "Volume"],
        ascending=[True, False]
    ).iloc[0]

    pool.append({
        "case_id": row["case_id"],
        "patient_id": row["patient_id"],
        "image_path": row["image_path"],
        "world_x": row["world_x"],
        "world_y": row["world_y"],
        "world_z": row["world_z"],
        "diameter_mm": row["diameter_mm"],
        "reference_score": row["reference_score"],
        "label": row["label"],
        "RadID": int(best["RadID"]),
        "FindingID": int(best["FindingID"]),
        "annotation_volume_mm3": float(best["Volume"]),
        "coordinate_distance_mm": float(best["distance_mm"]),
    })

pool_df = pd.DataFrame(pool)

if pool_df.empty:
    raise RuntimeError("No eligible annotations matched the manifest.")

# Define size groups in mm³.
pool_df["size_group"] = pd.cut(
    pool_df["annotation_volume_mm3"],
    bins=[100, 500, 2000, 5000, float("inf")],
    labels=["Small", "Medium", "Large", "Very large"],
    right=False
)

# Retain eligible cases and choose one case per patient.
pool_df = pool_df.dropna(subset=["size_group"])
pool_df = pool_df.sort_values("coordinate_distance_mm")
pool_df = pool_df.drop_duplicates("patient_id")

# Fixed seed makes sampling reproducible.
rng = np.random.default_rng(42)
samples = []

for group in ["Small", "Medium", "Large", "Very large"]:
    group_df = pool_df[pool_df["size_group"] == group]
    n = min(5, len(group_df))

    if n:
        selected_indices = rng.choice(
            group_df.index.to_numpy(),
            size=n,
            replace=False
        )
        samples.append(pool_df.loc[selected_indices])

sample_df = pd.concat(samples, ignore_index=True)
sample_df = sample_df.sort_values(
    ["size_group", "annotation_volume_mm3"]
).reset_index(drop=True)

out_path = ROOT / "mask_candidates_stratified.csv"
sample_df.to_csv(out_path, index=False)

print("Eligible pool:", len(pool_df))
print("Selected cases:", len(sample_df))
print("\nCases per volume group:")
print(sample_df["size_group"].value_counts().sort_index())

display(sample_df[
    [
        "case_id", "size_group", "annotation_volume_mm3",
        "diameter_mm", "RadID", "FindingID",
        "coordinate_distance_mm"
    ]
])

print("\nSaved:", out_path)

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
ARCHIVE = ROOT / "raw_archives" / "masks.rar"
CANDIDATES = ROOT / "mask_candidates_stratified.csv"
MASK_DIR = ROOT / "mask_candidates_stratified_ref"

df = pd.read_csv(CANDIDATES)
MASK_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the RAR utility installed earlier.
if "UNRAR" not in globals() or not Path(UNRAR).exists():
    matches = list(Path("/content/rar_tools").rglob("unrar"))
    if not matches:
        raise FileNotFoundError(
            "UnRAR not found. Re-run the earlier extractor installation."
        )
    UNRAR = str(matches[0])

listing = subprocess.run(
    [UNRAR, "lb", str(ARCHIVE)],
    capture_output=True,
    text=True,
    check=True
)

available = set(listing.stdout.splitlines())
targets = []
missing = []

for _, row in df.iterrows():
    patient = str(row["patient_id"])
    rad = int(row["RadID"])

    for ext in [".mhd", ".raw"]:
        member = f"masks/{patient}_rad{rad}{ext}"
        if member in available:
            targets.append(member)
        else:
            missing.append(member)

print("Cases requested:", len(df))
print("Archive files available:", len(set(targets)))

if missing:
    print("\nMissing archive files:")
    for name in missing:
        print(name)

if not targets:
    raise RuntimeError("No matching masks found in the archive.")

# Extract only the available matching mask files.
subprocess.run(
    [
        UNRAR, "x", "-o+",
        str(ARCHIVE),
        *sorted(set(targets)),
        str(MASK_DIR) + "/"
    ],
    check=True
)

headers = sorted(MASK_DIR.rglob("*.mhd"))
print("\nExtracted reference mask headers:", len(headers))
for p in headers:
    print(p)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MANIFEST_FILE = ROOT / "manifest.csv"
CANDIDATES_FILE = ROOT / "mask_candidates_stratified.csv"
MASK_DIR = ROOT / "mask_candidates_stratified_ref" / "masks"

OUT_DIR = ROOT / "mask_stratified_qc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_FILE)
candidates = pd.read_csv(CANDIDATES_FILE)

# Preserve the existing segmentation method; do not tune it on LNDb.
def auto_mask_with_status(hu, diam_mm, thr=-500):
    zz, yy, xx = np.indices(hu.shape)
    r = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    fallback = r <= max(float(diam_mm) / 2, 2.5)
    radius = max(float(diam_mm) * 0.75 + 1, 4)

    labels, n = ndimage.label(
        (hu > thr) & (r <= radius)
    )

    if n == 0:
        return fallback, "fallback_no_components"

    central = labels[28:37, 28:37, 28:37]
    ids = central[central > 0]

    if len(ids) == 0:
        return fallback, "fallback_no_central_component"

    selected = int(np.bincount(ids).argmax())

    cleaned = ndimage.binary_fill_holes(
        ndimage.binary_opening(
            labels == selected,
            iterations=1
        )
    )

    if cleaned.sum() < 20:
        return fallback, "fallback_component_too_small"

    return cleaned, "threshold_component"


def calculate_dice(reference, automatic):
    reference = np.asarray(reference, dtype=bool)
    automatic = np.asarray(automatic, dtype=bool)

    denominator = int(reference.sum()) + int(automatic.sum())

    if denominator == 0:
        return np.nan

    return (
        2.0 * np.logical_and(reference, automatic).sum()
        / denominator
    )


results = []

# One manifest entry per case is expected.
if manifest["case_id"].duplicated().any():
    raise ValueError("Duplicate case_id values found in manifest.csv.")

manifest_lookup = manifest.set_index("case_id")

for _, candidate in candidates.iterrows():
    case_id = str(candidate["case_id"])
    patient_id = str(candidate["patient_id"])
    rad_id = int(candidate["RadID"])
    finding_id = int(candidate["FindingID"])
    size_group = str(candidate["size_group"])

    if case_id not in manifest_lookup.index:
        print(f"SKIP {case_id}: not found in manifest")
        continue

    row = manifest_lookup.loc[case_id]

    ct_path = Path(row["image_path"])
    ref_path = MASK_DIR / f"{patient_id}_rad{rad_id}.mhd"

    if not ct_path.exists() or not ref_path.exists():
        print(f"SKIP {case_id}: CT or reference mask is missing")
        continue

    ct_img = sitk.ReadImage(str(ct_path))
    ref_img = sitk.ReadImage(str(ref_path))

    geometry_matches = (
        ct_img.GetSize() == ref_img.GetSize()
        and np.allclose(ct_img.GetSpacing(), ref_img.GetSpacing())
        and np.allclose(ct_img.GetOrigin(), ref_img.GetOrigin())
        and np.allclose(ct_img.GetDirection(), ref_img.GetDirection())
    )

    # Resample the radiologist mask onto the CT grid if necessary.
    if not geometry_matches:
        ref_img = sitk.Resample(
            ref_img,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            ref_img.GetPixelID()
        )

    ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)
    reference_full = (
        sitk.GetArrayFromImage(ref_img) == finding_id
    )

    world_xyz = (
        float(row["world_x"]),
        float(row["world_y"]),
        float(row["world_z"])
    )

    x, y, z = ct_img.TransformPhysicalPointToIndex(world_xyz)

    if not (
        0 <= x < ct.shape[2]
        and 0 <= y < ct.shape[1]
        and 0 <= z < ct.shape[0]
    ):
        print(f"SKIP {case_id}: coordinate outside the CT")
        continue

    spacing_zyx = np.asarray(
        ct_img.GetSpacing(), dtype=float
    )[::-1]

    # Existing make_patch function: same crop, resampling and padding.
    hu_patch, reference_patch = make_patch(
        ct,
        reference_full,
        (z, y, x),
        spacing_zyx
    )

    reference_patch = reference_patch.astype(bool)

    automatic_patch, method = auto_mask_with_status(
        hu_patch,
        float(row["diameter_mm"])
    )

    reference_voxels = int(reference_patch.sum())
    automatic_voxels = int(automatic_patch.sum())
    intersection_voxels = int(
        np.logical_and(reference_patch, automatic_patch).sum()
    )

    dice = calculate_dice(reference_patch, automatic_patch)

    if reference_voxels < 20:
        interpretation = "Sparse reference; Dice not reliable"
    else:
        interpretation = "Exploratory overlap result"

    results.append({
        "case_id": case_id,
        "size_group": size_group,
        "patient_id": patient_id,
        "RadID": rad_id,
        "FindingID": finding_id,
        "annotation_volume_mm3": float(
            candidate["annotation_volume_mm3"]
        ),
        "coordinate_distance_mm": float(
            candidate["coordinate_distance_mm"]
        ),
        "reference_voxels": reference_voxels,
        "automatic_voxels": automatic_voxels,
        "intersection_voxels": intersection_voxels,
        "dice": dice,
        "automatic_mask_method": method,
        "fallback_used": method.startswith("fallback"),
        "geometry_matched_before_resampling": geometry_matches,
        "interpretation": interpretation
    })

    print(
        f"{case_id} [{size_group}]: Dice={dice:.3f}, "
        f"reference={reference_voxels}, "
        f"automatic={automatic_voxels}, method={method}"
    )

    # Save diagnostic overlays for later review.
    if reference_patch.any():
        cz, cy, cx = np.rint(
            np.argwhere(reference_patch).mean(axis=0)
        ).astype(int)
    else:
        cz, cy, cx = np.array(hu_patch.shape) // 2

    views = [
        ("Axial", hu_patch[cz],
         reference_patch[cz], automatic_patch[cz]),
        ("Coronal", hu_patch[:, cy, :],
         reference_patch[:, cy, :], automatic_patch[:, cy, :]),
        ("Sagittal", hu_patch[:, :, cx],
         reference_patch[:, :, cx], automatic_patch[:, :, cx])
    ]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, (name, image, green, red) in zip(axes, views):
        ax.imshow(
            image,
            cmap="gray",
            vmin=HU_MIN,
            vmax=HU_MAX,
            origin="lower"
        )

        if green.any() and not green.all():
            ax.contour(
                green.astype(float),
                levels=[0.5],
                colors="lime",
                linewidths=1.2
            )

        if red.any() and not red.all():
            ax.contour(
                red.astype(float),
                levels=[0.5],
                colors="red",
                linewidths=1.2
            )

        ax.set_title(name)
        ax.axis("off")

    fig.suptitle(
        f"{case_id} | {size_group} | "
        f"Dice={dice:.3f} | {method}"
    )
    plt.tight_layout()
    fig.savefig(
        OUT_DIR / f"{case_id}_overlay.png",
        dpi=120,
        bbox_inches="tight"
    )
    plt.close(fig)

    del ct, reference_full, reference_patch, automatic_patch, hu_patch
    del ct_img, ref_img


report = pd.DataFrame(results)

if report.empty:
    raise RuntimeError("No cases were evaluated. Check the paths and manifest.")

# Save the per-case metrics.
per_case_path = ROOT / "mask_stratified_results.csv"
report.to_csv(per_case_path, index=False)

# Summarize only cases with sufficient reference voxels.
report["dice_evaluable"] = report["reference_voxels"] >= 20

summary = (
    report.groupby("size_group", observed=True)
    .agg(
        cases_processed=("case_id", "count"),
        evaluable_cases=("dice_evaluable", "sum"),
        median_dice=("dice", "median"),
        fallback_cases=("fallback_used", "sum")
    )
    .reset_index()
)

# IQR calculated only among cases with usable reference masks.
evaluable = report[report["dice_evaluable"]].copy()

iqr = (
    evaluable.groupby("size_group", observed=True)["dice"]
    .agg(
        dice_q1=lambda s: s.quantile(0.25),
        dice_q3=lambda s: s.quantile(0.75)
    )
    .reset_index()
)

summary = summary.merge(iqr, on="size_group", how="left")
summary_path = ROOT / "mask_stratified_summary.csv"
summary.to_csv(summary_path, index=False)

print("\n=== PER-CASE RESULTS ===")
display(
    report[
        [
            "case_id", "size_group", "reference_voxels",
            "automatic_voxels", "dice",
            "automatic_mask_method", "interpretation"
        ]
    ]
)

print("\n=== SUMMARY BY SIZE GROUP ===")
display(summary)

print("\nCases processed:", len(report))
print("Fallback cases:", int(report["fallback_used"].sum()))
print("Cases with usable reference masks:", len(evaluable))
print("Per-case CSV:", per_case_path)
print("Summary CSV:", summary_path)
print("Overlay folder:", OUT_DIR)

In [ ]:
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
ARCHIVE = ROOT / "raw_archives" / "masks.rar"
manifest = pd.read_csv(ROOT / "manifest.csv")
nodules = pd.read_csv(
    ROOT / "metadata" / "trainset_csv" / "trainNodules.csv"
)

for col in ["LNDbID", "RadID", "FindingID", "Nodule",
            "x", "y", "z", "Volume"]:
    nodules[col] = pd.to_numeric(nodules[col], errors="coerce")

pool = []

for _, row in manifest.iterrows():
    m = re.search(r"LNDb-(\d+)", str(row["patient_id"]))
    if not m:
        continue

    patient_num = int(m.group(1))
    xyz = np.array([
        row["world_x"], row["world_y"], row["world_z"]
    ], dtype=float)

    ann = nodules[
        (nodules["LNDbID"] == patient_num) &
        (nodules["Nodule"] == 1) &
        (nodules["Volume"] >= 100)
    ].dropna(subset=["x", "y", "z", "Volume"]).copy()

    if ann.empty:
        continue

    ann["distance_mm"] = np.linalg.norm(
        ann[["x", "y", "z"]].to_numpy() - xyz,
        axis=1
    )

    ann = ann[ann["distance_mm"] <= 5]
    if ann.empty:
        continue

    best = ann.sort_values(
        ["distance_mm", "Volume"],
        ascending=[True, False]
    ).iloc[0]

    pool.append({
        "case_id": row["case_id"],
        "patient_id": row["patient_id"],
        "image_path": row["image_path"],
        "world_x": row["world_x"],
        "world_y": row["world_y"],
        "world_z": row["world_z"],
        "diameter_mm": row["diameter_mm"],
        "RadID": int(best["RadID"]),
        "FindingID": int(best["FindingID"]),
        "annotation_volume_mm3": float(best["Volume"]),
        "coordinate_distance_mm": float(best["distance_mm"]),
    })

pool_df = pd.DataFrame(pool)

# One candidate per patient, matching the earlier selection procedure.
pool_df = (
    pool_df.sort_values("coordinate_distance_mm")
    .drop_duplicates("patient_id")
    .reset_index(drop=True)
)

pool_file = ROOT / "mask_candidates_all_eligible.csv"
pool_df.to_csv(pool_file, index=False)

print("Eligible candidates:", len(pool_df))
print("Saved:", pool_file)

# Reuse the RAR utility installed earlier.
if "UNRAR" not in globals() or not Path(UNRAR).exists():
    paths = list(Path("/content/rar_tools").rglob("unrar"))
    if not paths:
        raise FileNotFoundError(
            "UnRAR not found; rerun the earlier extractor-installation cell."
        )
    UNRAR = str(paths[0])

listing = subprocess.run(
    [UNRAR, "lb", str(ARCHIVE)],
    capture_output=True,
    text=True,
    check=True
)
available = set(listing.stdout.splitlines())

targets = []
missing = []

for _, row in pool_df.iterrows():
    stem = f"masks/{row['patient_id']}_rad{int(row['RadID'])}"
    for ext in [".mhd", ".raw"]:
        member = stem + ext
        if member in available:
            targets.append(member)
        else:
            missing.append(member)

targets = sorted(set(targets))
mask_dir = ROOT / "mask_all_eligible_ref"
mask_dir.mkdir(parents=True, exist_ok=True)

print("Reference files available:", len(targets))
print("Missing archive files:", len(missing))

if missing:
    print("First missing paths:", missing[:10])

if targets:
    subprocess.run(
        [UNRAR, "x", "-o+", str(ARCHIVE),
         *targets, str(mask_dir) + "/"],
        check=True
    )

print("Extracted reference headers:", len(list(mask_dir.rglob("*.mhd"))))

In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
CANDIDATES_FILE = ROOT / "mask_candidates_all_eligible.csv"
MASK_DIR = ROOT / "mask_all_eligible_ref" / "masks"

OUT_FILE = ROOT / "mask_all_eligible_results.csv"
SUMMARY_FILE = ROOT / "mask_all_eligible_summary.csv"

candidates = pd.read_csv(CANDIDATES_FILE)
manifest = pd.read_csv(ROOT / "manifest.csv")

if manifest["case_id"].duplicated().any():
    raise ValueError("Duplicate case_id values found in manifest.csv")

manifest_lookup = manifest.set_index("case_id")

# Keep the existing threshold, search radius, and cleaning method.
def automatic_mask(hu, diameter_mm, threshold=-500):
    zz, yy, xx = np.indices(hu.shape)
    r = np.sqrt(
        (zz - 32)**2 +
        (yy - 32)**2 +
        (xx - 32)**2
    )

    fallback = r <= max(float(diameter_mm) / 2, 2.5)
    radius = max(float(diameter_mm) * 0.75 + 1, 4)

    labels, n = ndimage.label(
        (hu > threshold) & (r <= radius)
    )

    if n == 0:
        return fallback, "fallback_no_components"

    central = labels[28:37, 28:37, 28:37]
    ids = central[central > 0]

    if len(ids) == 0:
        return fallback, "fallback_no_central_component"

    selected = int(np.bincount(ids).argmax())

    cleaned = ndimage.binary_fill_holes(
        ndimage.binary_opening(
            labels == selected,
            iterations=1
        )
    )

    if cleaned.sum() < 20:
        return fallback, "fallback_component_too_small"

    return cleaned, "threshold_component"


def dice_score(reference, predicted):
    reference = np.asarray(reference, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)

    denominator = int(reference.sum()) + int(predicted.sum())

    if denominator == 0:
        return np.nan

    return (
        2 * np.logical_and(reference, predicted).sum()
        / denominator
    )


# Size groups are based on reported annotation volume.
volume_bins = [100, 500, 2000, 5000, np.inf]
volume_labels = ["Small", "Medium", "Large", "Very large"]

results = []
errors = []

for i, candidate in candidates.iterrows():
    case_id = str(candidate["case_id"])
    patient_id = str(candidate["patient_id"])
    rad_id = int(candidate["RadID"])
    finding_id = int(candidate["FindingID"])

    if case_id not in manifest_lookup.index:
        errors.append((case_id, "Missing manifest row"))
        continue

    row = manifest_lookup.loc[case_id]

    ct_path = Path(row["image_path"])
    ref_path = MASK_DIR / f"{patient_id}_rad{rad_id}.mhd"

    if not ct_path.exists():
        errors.append((case_id, "Missing CT"))
        continue

    if not ref_path.exists():
        errors.append((case_id, "Missing reference mask"))
        continue

    try:
        ct_img = sitk.ReadImage(str(ct_path))
        ref_img = sitk.ReadImage(str(ref_path))

        # Align reference mask to CT geometry if needed.
        geometry_matches = (
            ct_img.GetSize() == ref_img.GetSize()
            and np.allclose(
                ct_img.GetSpacing(), ref_img.GetSpacing()
            )
            and np.allclose(
                ct_img.GetOrigin(), ref_img.GetOrigin()
            )
            and np.allclose(
                ct_img.GetDirection(), ref_img.GetDirection()
            )
        )

        if not geometry_matches:
            ref_img = sitk.Resample(
                ref_img,
                ct_img,
                sitk.Transform(),
                sitk.sitkNearestNeighbor,
                0,
                ref_img.GetPixelID()
            )

        ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)

        reference_full = (
            sitk.GetArrayFromImage(ref_img) == finding_id
        )

        if not reference_full.any():
            errors.append(
                (case_id, f"Finding label {finding_id} absent from mask")
            )
            continue

        world_xyz = (
            float(row["world_x"]),
            float(row["world_y"]),
            float(row["world_z"])
        )

        x, y, z = ct_img.TransformPhysicalPointToIndex(world_xyz)

        if not (
            0 <= x < ct.shape[2]
            and 0 <= y < ct.shape[1]
            and 0 <= z < ct.shape[0]
        ):
            errors.append((case_id, "Coordinate outside CT"))
            continue

        spacing_zyx = np.asarray(
            ct_img.GetSpacing(), dtype=float
        )[::-1]

        # Use the same existing crop/resampling/padding pipeline.
        hu_patch, reference_patch = make_patch(
            ct,
            reference_full,
            (z, y, x),
            spacing_zyx
        )

        reference_patch = reference_patch.astype(bool)

        predicted_patch, method = automatic_mask(
            hu_patch,
            float(row["diameter_mm"])
        )

        ref_voxels = int(reference_patch.sum())
        pred_voxels = int(predicted_patch.sum())
        intersection = int(
            np.logical_and(reference_patch, predicted_patch).sum()
        )

        dice = dice_score(reference_patch, predicted_patch)

        results.append({
            "case_id": case_id,
            "patient_id": patient_id,
            "RadID": rad_id,
            "FindingID": finding_id,
            "annotation_volume_mm3": float(
                candidate["annotation_volume_mm3"]
            ),
            "size_group": pd.cut(
                [float(candidate["annotation_volume_mm3"])],
                bins=volume_bins,
                labels=volume_labels,
                right=False
            )[0],
            "coordinate_distance_mm": float(
                candidate["coordinate_distance_mm"]
            ),
            "reference_voxels": ref_voxels,
            "automatic_voxels": pred_voxels,
            "intersection_voxels": intersection,
            "dice": dice,
            "automatic_mask_method": method,
            "fallback_used": method.startswith("fallback"),
            "geometry_matched_before_resampling": geometry_matches,
            "interpretation": (
                "Sparse reference; interpret cautiously"
                if ref_voxels < 20
                else "Exploratory overlap result"
            )
        })

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(candidates)} candidates")

        del ct, reference_full, reference_patch, predicted_patch
        del hu_patch, ct_img, ref_img

    except Exception as exc:
        errors.append((case_id, repr(exc)))
        print("ERROR:", case_id, repr(exc))


report = pd.DataFrame(results)

if report.empty:
    raise RuntimeError(
        "No cases evaluated successfully. Inspect paths and errors."
    )

report.to_csv(OUT_FILE, index=False)

# Exclude very sparse masks from the primary Dice summary.
report["dice_evaluable"] = (
    (report["reference_voxels"] >= 20)
    & report["dice"].notna()
)

evaluable = report[report["dice_evaluable"]]

summary_rows = []

for group in volume_labels:
    group_all = report[report["size_group"] == group]
    group_eval = evaluable[evaluable["size_group"] == group]
    scores = group_eval["dice"].dropna()

    summary_rows.append({
        "size_group": group,
        "cases_processed": len(group_all),
        "evaluable_cases": len(group_eval),
        "median_dice": scores.median() if len(scores) else np.nan,
        "dice_q1": scores.quantile(0.25) if len(scores) else np.nan,
        "dice_q3": scores.quantile(0.75) if len(scores) else np.nan,
        "fallback_cases": int(group_all["fallback_used"].sum()),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_FILE, index=False)

print("\n=== OVERALL SUMMARY ===")
print("Candidates:", len(candidates))
print("Successfully processed:", len(report))
print("Processing errors:", len(errors))
print("Cases with usable reference masks:", len(evaluable))
print("Fallback masks:", int(report["fallback_used"].sum()))

scores = evaluable["dice"].dropna()

if len(scores):
    print("Overall median Dice:", round(scores.median(), 4))
    print("Overall Dice IQR:",
          round(scores.quantile(0.25), 4),
          "to",
          round(scores.quantile(0.75), 4))

print("\n=== SUMMARY BY SIZE GROUP ===")
display(summary)

print("\n=== TEN LOWEST DICE SCORES ===")
display(
    report.sort_values("dice")[
        [
            "case_id",
            "size_group",
            "reference_voxels",
            "automatic_voxels",
            "dice",
            "automatic_mask_method"
        ]
    ].head(10)
)

print("\nResults saved to:", OUT_FILE)
print("Summary saved to:", SUMMARY_FILE)

if errors:
    print("\nFirst processing errors:")
    for err in errors[:15]:
        print(err)

In [ ]:
# Run the full LNDb external-cohort evaluation.
# Uses the existing manifest, frozen model ensemble,
# preprocessing, and fixed operating threshold.

evaluate_external("LNDb")

In [ ]:
import os
from glob import glob

LNDB_ROOT = "/content/drive/MyDrive/external_cohorts/LNDb"

for path in glob(os.path.join(LNDB_ROOT, "**", "*.mhd"), recursive=True):
    name = os.path.basename(path)
    if "_rad" in name:
        print(path)

In [ ]:
from pathlib import Path
import subprocess
import hashlib

LNDB_ROOT = Path(
    "/content/drive/MyDrive/external_cohorts/LNDb"
)

ARCHIVE_DIR = LNDB_ROOT / "raw_archives"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

MASK_ARCHIVE = ARCHIVE_DIR / "masks.rar"

URL = "https://zenodo.org/records/7153205/files/masks.rar?download=1"

if not MASK_ARCHIVE.exists():
    subprocess.run(
        ["wget", "-c", "-O", str(MASK_ARCHIVE), URL],
        check=True
    )

md5 = hashlib.md5(MASK_ARCHIVE.read_bytes()).hexdigest()

print("Archive:", MASK_ARCHIVE)
print("Size (MB):", round(MASK_ARCHIVE.stat().st_size / 1e6, 2))
print("MD5:", md5)

assert md5 == "858fe9380cc5c665f610a21fd04cf34e", (
    "Archive checksum mismatch. Do not extract this file yet."
)

print("Archive verified successfully.")

In [ ]:
from pathlib import Path
import subprocess
import hashlib

LNDB_ROOT = Path("/content/drive/MyDrive/external_cohorts/LNDb")
MASK_ARCHIVE = LNDB_ROOT / "raw_archives" / "masks.rar"

if not MASK_ARCHIVE.exists():
    print("ERROR: masks.rar was not found.")
else:
    print("File size:", round(MASK_ARCHIVE.stat().st_size / 1e6, 2), "MB")

    md5 = hashlib.md5(MASK_ARCHIVE.read_bytes()).hexdigest()
    print("MD5:", md5)

    expected = "858fe9380cc5c665f610a21fd04cf34e"
    print("Checksum correct:", md5 == expected)

    for command in [
        ["7z", "l", str(MASK_ARCHIVE)],
        ["7z", "t", str(MASK_ARCHIVE)],
    ]:
        print("\nRunning:", " ".join(command))
        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        print("Exit code:", result.returncode)
        print(result.stdout[-5000:])

        if result.stderr:
            print("STDERR:", result.stderr[-2000:])

In [ ]:
import subprocess
import tarfile
from pathlib import Path

# Download official RAR for Linux x64
URL = "https://www.rarlab.com/rar/rarlinux-x64-723.tar.gz"
PACKAGE = "/content/rarlinux-x64-723.tar.gz"
TOOL_DIR = Path("/content/rar_tools")
TOOL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["wget", "-q", "-O", PACKAGE, URL],
    check=True
)

with tarfile.open(PACKAGE, "r:gz") as archive:
    archive.extractall(TOOL_DIR)

candidates = list(TOOL_DIR.rglob("unrar"))

if not candidates:
    raise FileNotFoundError(
        "The unrar executable was not found in the downloaded package."
    )

UNRAR = str(candidates[0])
UNRAR_PATH = Path(UNRAR)
UNRAR_PATH.chmod(UNRAR_PATH.stat().st_mode | 0o111)

result = subprocess.run(
    [UNRAR, "-?"],
    capture_output=True,
    text=True
)

print("UnRAR executable:", UNRAR)
print((result.stdout or result.stderr)[:1000])

In [ ]:
import subprocess
from pathlib import Path

LNDB_ROOT = Path(
    "/content/drive/MyDrive/external_cohorts/LNDb"
)

MASK_ARCHIVE = LNDB_ROOT / "raw_archives" / "masks.rar"
MASK_DIR = LNDB_ROOT / "masks_sample"
MASK_DIR.mkdir(parents=True, exist_ok=True)

patient_ids = [
    "LNDb-0304",
    "LNDb-0035",
    "LNDb-0002",
    "LNDb-0111",
    "LNDb-0183",
]

# Read the archive's file listing using the compatible extractor.
listing = subprocess.run(
    [UNRAR, "lb", str(MASK_ARCHIVE)],
    capture_output=True,
    text=True,
    check=True
)

available = set(listing.stdout.splitlines())

# Select each available radiologist mask header and its associated raw volume.
targets = []

for patient in patient_ids:
    for rad in (1, 2):
        stem = f"masks/{patient}_rad{rad}"

        for extension in (".mhd", ".raw"):
            name = stem + extension
            if name in available:
                targets.append(name)

print("Matching archive files:", len(targets))
for name in targets:
    print(name)

if not targets:
    raise RuntimeError(
        "No matching masks found; inspect the archive listing."
    )

# Extract only the selected files, preserving the masks/ folder structure.
subprocess.run(
    [
        UNRAR, "x", "-o+",
        str(MASK_ARCHIVE),
        *targets,
        str(MASK_DIR) + "/"
    ],
    check=True
)

extracted = list(MASK_DIR.rglob("*.mhd"))
print("\nExtracted mask headers:", len(extracted))

for path in extracted:
    print(path)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import zipfile

ROOT = Path(
    "/content/drive/MyDrive/external_cohorts/LNDb"
)

MASK_DIR = ROOT / "masks_sample" / "masks"

# 1. Locate trainNodules.csv
csv_files = list(ROOT.rglob("trainNodules.csv"))
zip_files = list(ROOT.rglob("trainset_csv.zip"))

if csv_files:
    nodules = pd.read_csv(csv_files[0])
    print("Loaded CSV:", csv_files[0])

elif zip_files:
    with zipfile.ZipFile(zip_files[0]) as zf:
        matches = [
            name for name in zf.namelist()
            if name.endswith("trainNodules.csv")
        ]

        if not matches:
            raise FileNotFoundError(
                "trainNodules.csv not found inside trainset_csv.zip"
            )

        with zf.open(matches[0]) as f:
            nodules = pd.read_csv(f)

        print("Loaded from ZIP:", matches[0])

else:
    raise FileNotFoundError(
        "Could not locate trainNodules.csv or trainset_csv.zip"
    )

print("\n=== NODULE ANNOTATION TABLE ===")
print("Columns:", nodules.columns.tolist())
print(nodules.head(5).to_string(index=False))


# 2. Inspect each extracted reference mask
print("\n=== REFERENCE MASK DETAILS ===")

for path in sorted(MASK_DIR.glob("*.mhd")):
    image = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(image)

    values, counts = np.unique(arr, return_counts=True)
    nonzero = values != 0

    print(f"\nFile: {path.name}")
    print("Image size (x, y, z):", image.GetSize())
    print("Spacing:", image.GetSpacing())
    print("Origin:", image.GetOrigin())
    print("Direction:", image.GetDirection())
    print("Unique voxel values:", values[:30].tolist())
    print("Nonzero values / voxel counts:")

    for value, count in zip(values[nonzero], counts[nonzero]):
        print(f"  Finding ID {value}: {count:,} voxels")

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import zipfile

ROOT = Path(
    "/content/drive/MyDrive/external_cohorts/LNDb"
)

# Load your external-validation manifest
manifest = pd.read_csv(ROOT / "manifest.csv")

# Load the individual radiologists' annotations
csv_path = ROOT / "metadata" / "trainset_csv" / "trainNodules.csv"
nodules = pd.read_csv(csv_path)

target_cases = [
    "LNDb-0304_F0006",
    "LNDb-0035_F0008",
    "LNDb-0002_F0006",
    "LNDb-0111_F0003",
    "LNDb-0035_F0004",
    "LNDb-0183_F0008",
]

print("Manifest columns:", manifest.columns.tolist())

# Find the coordinate columns in the manifest
coordinate_options = [
    ("x_gt", "y_gt", "z_gt"),
    ("world_x", "world_y", "world_z"),
    ("x", "y", "z"),
]

coord_cols = next(
    (
        cols for cols in coordinate_options
        if all(c in manifest.columns for c in cols)
    ),
    None
)

if coord_cols is None:
    raise ValueError(
        "Cannot identify manifest coordinate columns. "
        "Check the printed manifest columns."
    )

print("Using manifest coordinates:", coord_cols)

for case_id in target_cases:
    selected = manifest[manifest["case_id"] == case_id]

    if selected.empty:
        print(f"\nNOT FOUND IN MANIFEST: {case_id}")
        continue

    row = selected.iloc[0]

    # Extract integer patient number: LNDb-0304 -> 304
    match = re.search(r"LNDb-(\d+)", case_id)
    if not match:
        print(f"\nCannot parse patient ID: {case_id}")
        continue

    patient_num = int(match.group(1))

    xyz = row[list(coord_cols)].to_numpy(dtype=float)

    candidates = nodules[
        (nodules["LNDbID"] == patient_num) &
        (nodules["Nodule"] == 1)
    ].copy()

    candidates[["x", "y", "z"]] = candidates[
        ["x", "y", "z"]
    ].apply(pd.to_numeric, errors="coerce")

    candidates = candidates.dropna(subset=["x", "y", "z"])

    candidates["distance_mm"] = np.linalg.norm(
        candidates[["x", "y", "z"]].to_numpy() - xyz,
        axis=1
    )

    print(f"\n{'=' * 65}")
    print("Manifest case:", case_id)
    print("Manifest coordinates:", xyz)

    if candidates.empty:
        print("No matching radiologist findings found.")
        continue

    # Show nearest annotations separately for each radiologist
    for rad_id, group in candidates.groupby("RadID"):
        nearest = group.nsmallest(3, "distance_mm")

        print(f"\nRadiologist {rad_id}:")
        print(
            nearest[
                [
                    "FindingID", "x", "y", "z",
                    "Nodule", "Volume", "Text", "distance_mm"
                ]
            ].to_string(index=False)
        )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/external_cohorts/LNDb"
)

MASK_DIR = ROOT / "masks_sample" / "masks"
manifest = pd.read_csv(ROOT / "manifest.csv")

targets = {
    "LNDb-0304_F0006": (2, 4),
    "LNDb-0035_F0008": (2, 3),
    "LNDb-0002_F0006": (2, 2),
    "LNDb-0111_F0003": (2, 1),
    "LNDb-0035_F0004": (1, 4),
    "LNDb-0183_F0008": (2, 5),
}

for case_id, (rad_id, finding_id) in targets.items():
    row_match = manifest[manifest["case_id"] == case_id]

    if row_match.empty:
        print(f"Missing manifest row: {case_id}")
        continue

    row = row_match.iloc[0]
    patient_num = int(case_id.split("_")[0].split("-")[1])

    ct_path = Path(row["image_path"])
    mask_path = MASK_DIR / (
        f"LNDb-{patient_num:04d}_rad{rad_id}.mhd"
    )

    if not ct_path.exists() or not mask_path.exists():
        print(f"Missing CT or mask for {case_id}")
        continue

    ct_img = sitk.ReadImage(str(ct_path))
    mask_img = sitk.ReadImage(str(mask_path))

    # Align the mask to the CT grid if their geometry differs.
    same_geometry = (
        ct_img.GetSize() == mask_img.GetSize()
        and np.allclose(ct_img.GetSpacing(), mask_img.GetSpacing())
        and np.allclose(ct_img.GetOrigin(), mask_img.GetOrigin())
        and np.allclose(ct_img.GetDirection(), mask_img.GetDirection())
    )

    if not same_geometry:
        mask_img = sitk.Resample(
            mask_img,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            mask_img.GetPixelID()
        )

    ct = sitk.GetArrayFromImage(ct_img)
    mask = sitk.GetArrayFromImage(mask_img) == finding_id

    xyz_mm = (
        float(row["world_x"]),
        float(row["world_y"]),
        float(row["world_z"])
    )

    # SimpleITK returns index as (x, y, z); arrays are [z, y, x].
    x, y, z = ct_img.TransformPhysicalPointToIndex(xyz_mm)

    if not (
        0 <= x < ct.shape[2]
        and 0 <= y < ct.shape[1]
        and 0 <= z < ct.shape[0]
    ):
        print(f"Coordinate outside CT: {case_id}")
        continue

    voxel_count = int(mask.sum())

    print(
        f"{case_id}: finding={finding_id}, "
        f"reference-mask voxels={voxel_count}, "
        f"CT index (x,y,z)=({x},{y},{z}), "
        f"geometry initially matched={same_geometry}"
    )

    # Show a local region around the supplied coordinates.
    half_xy = 40
    half_z = 30

    x0, x1 = max(0, x-half_xy), min(ct.shape[2], x+half_xy+1)
    y0, y1 = max(0, y-half_xy), min(ct.shape[1], y+half_xy+1)
    z0, z1 = max(0, z-half_z), min(ct.shape[0], z+half_z+1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Axial: horizontal x, vertical y.
    axes[0].imshow(
        ct[z, y0:y1, x0:x1],
        cmap="gray", vmin=-1000, vmax=400, origin="lower"
    )
    axes[0].contour(
        mask[z, y0:y1, x0:x1].astype(float),
        levels=[0.5], colors="lime", linewidths=1.5
    )
    axes[0].scatter(x-x0, y-y0, c="yellow", marker="+", s=100)
    axes[0].set_title("Axial")

    # Coronal: horizontal x, vertical z.
    axes[1].imshow(
        ct[z0:z1, y, x0:x1],
        cmap="gray", vmin=-1000, vmax=400, origin="lower"
    )
    axes[1].contour(
        mask[z0:z1, y, x0:x1].astype(float),
        levels=[0.5], colors="lime", linewidths=1.5
    )
    axes[1].scatter(x-x0, z-z0, c="yellow", marker="+", s=100)
    axes[1].set_title("Coronal")

    # Sagittal: horizontal y, vertical z.
    axes[2].imshow(
        ct[z0:z1, y0:y1, x],
        cmap="gray", vmin=-1000, vmax=400, origin="lower"
    )
    axes[2].contour(
        mask[z0:z1, y0:y1, x].astype(float),
        levels=[0.5], colors="lime", linewidths=1.5
    )
    axes[2].scatter(y-y0, z-z0, c="yellow", marker="+", s=100)
    axes[2].set_title("Sagittal")

    fig.suptitle(
        f"{case_id} | reference label {finding_id} | "
        f"{voxel_count} voxels"
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    del ct, mask, ct_img, mask_img

In [ ]:
import subprocess
from pathlib import Path

MASK_DIR = LNDB_ROOT / "masks"

# Install an extractor that supports RAR archives.
subprocess.run(
    ["bash", "-lc", "apt-get -qq update && apt-get -qq install -y p7zip-full"],
    check=True
)

MASK_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["7z", "x", "-y", str(MASK_ARCHIVE), f"-o{MASK_DIR}"],
    check=True
)

print("Extraction completed.")

external  validation start here above is manifest ready code


In [ ]:
def auto_mask(hu, diam_mm, thr=-500):
    # automatic nodule mask on a 64^3 / 1 mm HU patch centred on the nodule
    zz, yy, xx = np.indices(hu.shape); r = np.sqrt((zz - 32) ** 2 + (yy - 32) ** 2 + (xx - 32) ** 2)
    fallback = r <= max(float(diam_mm) / 2, 2.5); rad_mm = max(float(diam_mm) * 0.75 + 1.0, 4.0)
    lab, n = ndimage.label((hu > thr) & (r <= rad_mm))
    if n == 0: return fallback
    ids = lab[28:37, 28:37, 28:37]; ids = ids[ids > 0]
    if len(ids) == 0: return fallback
    m = ndimage.binary_fill_holes(ndimage.binary_opening(lab == np.bincount(ids).argmax(), iterations=1))
    return m if m.sum() >= 20 else fallback

dice = lambda a, b: 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

# ---- 12.1 mask-source sensitivity on LIDC holdout: radiologist mask vs automatic mask for the radiomics vector -----------------
rad_auto, dices = [], []
for i in range(len(DF_HOLD)):
    hu = IMGS[i] * (HU_MAX - HU_MIN) + HU_MIN; m_auto = auto_mask(hu, DF_HOLD.diam.values[i]); dices.append(dice(m_auto, MASKS[i])); rad_auto.append(radiomics28(hu, m_auto))
rad_auto = np.stack(rad_auto).astype(np.float32); p_auto = ens_predict(MODELS, IMGS, rad_auto); p_man = ens_predict(MODELS, IMGS, RADS)
MASKSRC = pd.DataFrame([dict(radiomics_mask='radiologist consensus (training condition)', holdout_AUC=fast_auc(y_hold, p_man)), dict(radiomics_mask='automatic (external-style)', holdout_AUC=fast_auc(y_hold, p_auto))])
print(MASKSRC.round(4).to_string(index=False)); print(f'Dice(auto, radiologist) median {np.median(dices):.3f} [IQR {np.percentile(dices,25):.3f}-{np.percentile(dices,75):.3f}]')
MASKSRC.to_csv(os.path.join(EXT_DIR, 'mask_source_sensitivity_lidc.csv'), index=False)
print('If AUC drops with automatic masks, an external drop is partly a MASK-SOURCE shift, not only a population shift: report both.')

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
import pandas as pd

# Actual external datasets directory in Google Drive
EXTERNAL_ROOT = Path(
    "/content/drive/MyDrive/external_cohorts"
)

LNDB_MANIFEST = EXTERNAL_ROOT / "LNDb" / "manifest.csv"

# Verify that the manifest really exists
print("External root exists:", EXTERNAL_ROOT.exists())
print("LNDb manifest exists:", LNDB_MANIFEST.exists())

if not LNDB_MANIFEST.exists():
    raise FileNotFoundError(
        f"Manifest not found: {LNDB_MANIFEST}"
    )

man = pd.read_csv(LNDB_MANIFEST)

print("Manifest rows:", len(man))
print("Manifest columns:", man.columns.tolist())

# Fix the path used by evaluate_external()
CFG["EXTERNAL_DIR"] = str(EXTERNAL_ROOT)

# Save evaluation outputs and cached patches to Google Drive
EXT_DIR = "/content/drive/MyDrive/lidc_work/external_validation"
os.makedirs(EXT_DIR, exist_ok=True)

print("\nUpdated CFG['EXTERNAL_DIR']:", CFG["EXTERNAL_DIR"])
print("Evaluation output directory:", EXT_DIR)

# Confirm which cohorts will now be discovered
available = [
    d.name
    for d in EXTERNAL_ROOT.iterdir()
    if d.is_dir() and (d / "manifest.csv").exists()
]

print("Cohorts with manifests:", available)

In [ ]:
import SimpleITK as sitk
def read_ct(path):
    if os.path.isdir(path):
        rd = sitk.ImageSeriesReader(); rd.SetFileNames(rd.GetGDCMSeriesFileNames(path)); return rd.Execute()
    return sitk.ReadImage(path)

def ext_patch(row, cache_dir):
    fp = os.path.join(cache_dir, f"{row.case_id}.npz")
    if os.path.exists(fp): z = np.load(fp); return z['p64'].astype(np.float32), z['rad'].astype(np.float32), z['mask']
    img = read_ct(row.image_path); vol = sitk.GetArrayFromImage(img).astype(np.float32); sp = np.array(img.GetSpacing())[::-1]
    centre = np.array(img.TransformPhysicalPointToContinuousIndex((float(row.world_x), float(row.world_y), float(row.world_z))))[::-1]
    hu, _ = make_patch(vol, np.zeros(vol.shape, bool), centre, sp)
    m = auto_mask(hu, row.diameter_mm); rad = radiomics28(hu, m); p = ((hu - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32)
    np.savez_compressed(fp, p64=p.astype(np.float16), rad=rad, mask=m); return p, rad, m

def evaluate_external(cohort):
    cdir = os.path.join(CFG['EXTERNAL_DIR'], cohort); man = pd.read_csv(os.path.join(cdir, 'manifest.csv'))
    meta = verify_frozen(); thr = meta['threshold']; models = load_ensemble_from(FROZEN_DIR, meta['spec'])               # hash-verified frozen ensemble
    assert not man.patient_id.astype(str).str.contains('LIDC', case=False).any(), 'LIDC cases found in an "external" cohort'
    cache = os.path.join(EXT_DIR, cohort, 'patches'); os.makedirs(cache, exist_ok=True); P_, R_, M_ = [], [], []
    for r in man.itertuples():
        p, rad, m = ext_patch(r, cache); P_.append(p); R_.append(rad); M_.append(m)
    P_, R_ = np.stack(P_), np.stack(R_); y = man.label.values.astype(int); pr = ens_predict(models, P_, R_); grp = man.patient_id.astype(str).values
    cb = ClusterBoot(grp, CFG['N_BOOT'], 6); mci, cm = metrics_ci(y, pr, thr, cb); out = os.path.join(EXT_DIR, cohort); os.makedirs(out, exist_ok=True)
    tab = pd.DataFrame([dict(metric=k, estimate=v[0], ci_lo=v[1], ci_hi=v[2]) for k, v in mci.items()]); tab.to_csv(os.path.join(out, 'metrics_nodule_level.csv'), index=False)
    pdl = patient_level(y, pr, grp); sub = ClusterBoot(pdl.g.values, CFG['N_BOOT'], 7); a, lo, hi = auc_ci(pdl.y.values, pdl.p.values, sub)
    print(f"\n===== EXTERNAL: {cohort} | {len(man)} nodules / {man.patient_id.nunique()} patients | malignant {y.sum()} | frozen threshold {thr:.4f} =====")
    print(tab.round(4).to_string(index=False)); print('confusion matrix (tn fp fn tp):', cm['tn'], cm['fp'], cm['fn'], cm['tp']); print(f'patient-level AUC {a:.4f} [{lo:.4f}, {hi:.4f}]')
    # calibration (raw frozen probabilities + Platt params learned on LIDC dev only), Brier, ECE
    pcal = 1 / (1 + np.exp(-(meta['platt_coef'] * _logit(pr) + meta['platt_intercept']))); sl, ic = cal_slope_intercept(y, pr)
    calt = pd.DataFrame([dict(version='raw', brier=brier_score_loss(y, pr), ece=ece_score(y, pr), slope=sl, intercept=ic), dict(version='Platt (LIDC dev params, frozen)', brier=brier_score_loss(y, pcal), ece=ece_score(y, pcal), slope=cal_slope_intercept(y, pcal)[0], intercept=cal_slope_intercept(y, pcal)[1])])
    print(calt.round(4).to_string(index=False)); calt.to_csv(os.path.join(out, 'calibration.csv'), index=False)
       # -----------------------------------------------
    # Corrected external-validation plots
    # -----------------------------------------------
    def calibration_points(y_true, probs, n_bins=8):
        y_true = np.asarray(y_true)
        probs = np.asarray(probs)

        # Remove duplicate boundaries when predictions are clustered
        edges = np.unique(
            np.quantile(probs, np.linspace(0, 1, n_bins + 1))
        )

        if len(edges) < 2:
            return [float(np.mean(probs))], [
                float(np.mean(y_true))
            ]

        bin_ids = np.digitize(
            probs, edges[1:-1], right=True
        )

        mean_pred, mean_obs = [], []

        for b in range(len(edges) - 1):
            mask = bin_ids == b

            if mask.any():
                mean_pred.append(float(probs[mask].mean()))
                mean_obs.append(float(y_true[mask].mean()))

        return mean_pred, mean_obs

    fig, axs = plt.subplots(1, 3, figsize=(17, 5))

    # 1. Calibration: raw and frozen Platt-calibrated
    axs[0].plot(
        [0, 1], [0, 1], "k--", label="Perfect calibration"
    )

    x_raw, y_raw = calibration_points(y, pr)
    x_cal, y_cal = calibration_points(y, pcal)

    axs[0].plot(
        x_raw, y_raw, "o-",
        label="Raw probabilities"
    )
    axs[0].plot(
        x_cal, y_cal, "s-",
        label="Platt calibrated"
    )

    axs[0].set(
        title=f"{cohort}: calibration",
        xlabel="Mean predicted probability",
        ylabel="Observed positive fraction",
        xlim=(0, 1),
        ylim=(0, 1)
    )
    axs[0].legend(fontsize=8)

    # 2. ROC curve
    fpr_, tpr_, _ = roc_curve(y, pr)
    auc_value = roc_auc_score(y, pr)

    axs[1].plot(fpr_, tpr_, label=f"AUC = {auc_value:.3f}")
    axs[1].plot([0, 1], [0, 1], "k--", label="Random ranking")
    axs[1].set(
        title=f"{cohort}: ROC curve",
        xlabel="False positive rate",
        ylabel="True positive rate",
        xlim=(0, 1),
        ylim=(0, 1)
    )
    axs[1].legend()

    # 3. Decision curve
    dc = decision_curve(y, pr)

    # Focus on thresholds up to 0.60, including the frozen threshold
    visible = dc.loc[
        (dc["threshold"] >= 0) &
        (dc["threshold"] <= 0.60)
    ]

    axs[2].plot(
        visible["threshold"],
        visible["net_benefit_model"],
        label="Model"
    )
    axs[2].plot(
        visible["threshold"],
        visible["net_benefit_treat_all"],
        "k:",
        label="Treat all"
    )
    axs[2].axhline(
        0, color="gray", linestyle="--", label="Treat none"
    )

    # Set the y-axis to the observed range in the displayed region
    values = np.concatenate([
        visible["net_benefit_model"].to_numpy(),
        visible["net_benefit_treat_all"].to_numpy(),
        np.array([0.0])
    ])
    values = values[np.isfinite(values)]

    ymin, ymax = values.min(), values.max()
    pad = max((ymax - ymin) * 0.10, 0.01)

    axs[2].set_ylim(ymin - pad, ymax + pad)
    axs[2].set_xlim(0, 0.60)
    axs[2].set(
        title=f"{cohort}: decision curve",
        xlabel="Threshold probability",
        ylabel="Net benefit"
    )
    axs[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(
        os.path.join(out, "external_curves.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()
    # operating points frozen from LIDC development data
    ops = []
    for opn, t in meta['operating_points'].items(): c = conf_metrics(y, pr, t); ops.append(dict(operating_point=opn, threshold=t, sens=c['sens'], spec=c['spec'], prec=c['prec']))
    print(pd.DataFrame(ops).round(3).to_string(index=False)); pd.DataFrame(ops).to_csv(os.path.join(out, 'operating_points.csv'), index=False)
    # subgroups + failure cases
    d = man.copy(); d['prob'] = pr; d['pred'] = (pr >= thr).astype(int); d['y'] = y
    d['solidity'] = [solidity_of(m) for m in M_]; d['size_bin'] = pd.cut(d.diameter_mm, [0, 6, 10, 20, 1e3], labels=['<6', '6-10', '10-20', '>20'])
    sg = []
    for col in [c for c in ('size_bin', 'scanner', 'slice_thickness_mm') if c in d.columns] + ['solidity_bin']:
        if col == 'solidity_bin': d['solidity_bin'] = pd.cut(d.solidity, [0, 0.6, 0.8, 1.01], labels=['low', 'mid', 'high'])
        for g, s in d.groupby(col, observed=True): sg.append(dict(variable=col, group=str(g), n=len(s), n_malignant=int(s.y.sum()), auc=fast_auc(s.y.values, s.prob.values), sens=conf_metrics(s.y.values, s.prob.values, thr)['sens'], spec=conf_metrics(s.y.values, s.prob.values, thr)['spec']))
    pd.DataFrame(sg).to_csv(os.path.join(out, 'subgroups.csv'), index=False); print(pd.DataFrame(sg).round(3).to_string(index=False))
    fails = d[d.pred != d.y].assign(conf=lambda x: (x.prob - thr).abs()).sort_values('conf', ascending=False); fails.to_csv(os.path.join(out, 'failure_cases.csv'), index=False)
    print(f'failure cases: {len(fails)} (most confident errors first) -> failure_cases.csv')
    for rank, (ix, r) in enumerate(fails.head(8).iterrows()):
        fig, ax = plt.subplots(1, 3, figsize=(9, 3)); p = P_[ix]
        for a_, sl_, ms_ in zip(ax, (p[32], p[:, 32], p[:, :, 32]), (M_[ix][32], M_[ix][:, 32], M_[ix][:, :, 32])): a_.imshow(sl_, cmap='gray', vmin=0, vmax=1); a_.contour(ms_.astype(float), [0.5], colors='r'); a_.axis('off')
        plt.suptitle(f"{r.case_id}: true {r.y}, P={r.prob:.2f}"); plt.savefig(os.path.join(out, f'failure_{rank:02d}_{r.case_id}.png'), dpi=100); plt.close()
    stmt = (f"EXTERNAL VALIDATION STATEMENT ({cohort})\nThe frozen LIDC-IDRI development ensemble (five fold models, seed {meta['seed']}, SHA-256 verified) and the fixed threshold {thr:.4f} "
            f"(Youden on LIDC development out-of-fold predictions) were applied without retraining, checkpoint selection, rescaling or threshold tuning. No data from {cohort} were used for training or tuning. "
            f"Radiomics were computed from automatic nodule masks (no radiologist masks available). Date: {datetime.datetime.now().date()}\n")
    open(os.path.join(out, 'external_validation_statement.txt'), 'w').write(stmt); print('\n' + stmt)
    return dict(cohort=cohort, n=len(man), auc=mci['auc'], patient_auc=(a, lo, hi), metrics={k: v for k, v in mci.items()}, cm=cm, calibration=calt)

EXT_RESULTS = {}
avail = [d for d in sorted(os.listdir(CFG['EXTERNAL_DIR'])) if os.path.exists(os.path.join(CFG['EXTERNAL_DIR'], d, 'manifest.csv'))] if os.path.isdir(CFG['EXTERNAL_DIR']) else []
if not avail: print('No external cohort found in', CFG['EXTERNAL_DIR'], '-> external validation NOT performed. Do not claim it. Add <cohort>/manifest.csv (see section 12 header) and re-run this cell.')
for c in avail: EXT_RESULTS[c] = evaluate_external(c)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, roc_auc_score

cohort = "LNDb"

cdir = os.path.join(CFG["EXTERNAL_DIR"], cohort)
out = os.path.join(EXT_DIR, cohort)
cache = os.path.join(out, "patches")

man = pd.read_csv(os.path.join(cdir, "manifest.csv"))
os.makedirs(out, exist_ok=True)

# 1. Ensure every nodule has a cached patch
missing = [
    str(r.case_id)
    for r in man.itertuples()
    if not os.path.isfile(
        os.path.join(cache, f"{r.case_id}.npz")
    )
]

if missing:
    raise FileNotFoundError(
        f"{len(missing)} cached patches are missing. "
        "Stopping rather than silently preprocessing again. "
        f"Examples: {missing[:5]}"
    )

print(f"All {len(man)} cached patches found.")

# 2. Load cached data; no CT preprocessing or radiomics extraction
P_, R_ = [], []

for r in tqdm(man.itertuples(), total=len(man),
              desc="Loading cached patches"):
    with np.load(os.path.join(cache, f"{r.case_id}.npz")) as z:
        P_.append(z["p64"].astype(np.float32))
        R_.append(z["rad"].astype(np.float32))

P_ = np.stack(P_)
R_ = np.stack(R_)
y = man["label"].to_numpy(dtype=int)

# 3. Predict using the existing frozen ensemble
meta = verify_frozen()
models = load_ensemble_from(FROZEN_DIR, meta["spec"])
pr = ens_predict(models, P_, R_)

# Apply Platt parameters learned on LIDC development data
pcal = 1 / (
    1 + np.exp(-(
        meta["platt_coef"] * _logit(pr)
        + meta["platt_intercept"]
    ))
)

# 4. Calibration bins that handle repeated probabilities
def calibration_points(y_true, probs, n_bins=8):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)

    edges = np.unique(
        np.quantile(probs, np.linspace(0, 1, n_bins + 1))
    )

    if len(edges) < 2:
        return [float(np.mean(probs))], [
            float(np.mean(y_true))
        ]

    ids = np.digitize(probs, edges[1:-1], right=True)

    mean_pred, mean_obs = [], []

    for b in range(len(edges) - 1):
        mask = ids == b
        if mask.any():
            mean_pred.append(float(probs[mask].mean()))
            mean_obs.append(float(y_true[mask].mean()))

    return mean_pred, mean_obs


# 5. Draw the three plots
fig, axs = plt.subplots(1, 3, figsize=(17, 5))

# Calibration: raw and calibrated probabilities
axs[0].plot([0, 1], [0, 1], "k--", label="Ideal")

for probs, label, marker in [
    (pr, "Raw", "o"),
    (pcal, "Platt calibrated", "s"),
]:
    xp, yp = calibration_points(y, probs)
    axs[0].plot(xp, yp, marker + "-", label=label)

axs[0].set(
    title=f"{cohort}: calibration",
    xlabel="Mean predicted probability",
    ylabel="Observed positive fraction",
    xlim=(0, 1),
    ylim=(0, 1),
)
axs[0].legend(fontsize=8)

# ROC curve
fpr, tpr, _ = roc_curve(y, pr)
auc_value = roc_auc_score(y, pr)

axs[1].plot(fpr, tpr, label=f"AUC = {auc_value:.3f}")
axs[1].plot([0, 1], [0, 1], "k--")
axs[1].set(
    title=f"{cohort}: ROC",
    xlabel="False positive rate",
    ylabel="True positive rate",
    xlim=(0, 1),
    ylim=(0, 1),
)
axs[1].legend()

# Decision curve
dc = decision_curve(y, pr)
visible = dc[
    (dc["threshold"] >= 0) &
    (dc["threshold"] <= 0.60)
]

axs[2].plot(
    visible["threshold"],
    visible["net_benefit_model"],
    label="Model",
)
axs[2].plot(
    visible["threshold"],
    visible["net_benefit_treat_all"],
    "k:",
    label="Treat all",
)
axs[2].axhline(0, color="gray", linestyle="--",
               label="Treat none")

values = np.concatenate([
    visible["net_benefit_model"].to_numpy(),
    visible["net_benefit_treat_all"].to_numpy(),
    np.array([0.0]),
])
values = values[np.isfinite(values)]

if len(values):
    pad = max((values.max() - values.min()) * 0.1, 0.01)
    axs[2].set_ylim(values.min() - pad, values.max() + pad)

axs[2].set(
    title=f"{cohort}: decision curve",
    xlabel="Threshold probability",
    ylabel="Net benefit",
    xlim=(0, 0.60),
)
axs[2].legend(fontsize=8)

plt.tight_layout()

figure_path = os.path.join(out, "external_curves.png")
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("Updated figure:", figure_path)
print(f"ROC AUC: {auc_value:.4f}")

In [ ]:
import numpy as np

print("Prediction quantiles:")
print(np.quantile(pr, [0, .01, .05, .25, .50, .75, .95, .99, 1]))

print("\nPredictions above frozen threshold:")
print(int((pr >= meta["threshold"]).sum()), "of", len(pr))

print("\nRounded prediction distribution:")
values, counts = np.unique(np.round(pr, 3), return_counts=True)

for value, count in zip(values, counts):
    print(f"{value:.3f}: {count}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Compare raw scores for the two label groups
for label, name in [(0, "Benign-labelled"), (1, "Malignant-labelled")]:
    scores = np.asarray(pr)[np.asarray(y) == label]

    print(f"\n{name}: n={len(scores)}")
    print(f"Mean probability:   {scores.mean():.4f}")
    print(f"Median probability: {np.median(scores):.4f}")
    print(f"10th percentile:    {np.quantile(scores, 0.10):.4f}")
    print(f"90th percentile:    {np.quantile(scores, 0.90):.4f}")

print("\nOverall AUC:", round(roc_auc_score(y, pr), 4))
print("Frozen threshold:", round(float(meta["threshold"]), 4))

# Compare raw and frozen Platt-calibrated predictions
if "pcal" in globals():
    print("\nCalibrated probability quantiles:")
    print(np.round(np.quantile(pcal, [0, .1, .25, .5, .75, .9, 1]), 4))

In [ ]:
print("DF rows:", len(DF))
print("DF_DEV rows:", len(DF_DEV))
print("DF_HOLD rows:", len(DF_HOLD))

print("\nUnique training patients:")
print("DF:", DF.patient.nunique())
print("DF_DEV:", DF_DEV.patient.nunique())
print("DF_HOLD:", DF_HOLD.patient.nunique())

print("\nTraining labels:")
print(DF_DEV.label.value_counts(dropna=False))

print("\nTraining dataframe columns:")
print(DF_DEV.columns.tolist())

print("\nExample radiomics vector length:")
print(len(DF_DEV.rad.iloc[0]))

## 13. Compact figure set for the paper

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
names = [v for v in ('full', 'img_3s_cbam', 'rad_only', 'size_only') if v in AVAILABLE]
for v in names:
    fp_, tp_, _ = roc_curve(y_hold, pred_set(v)[1]); ax[0].plot(fp_, tp_, label=f"{v} (AUC {fast_auc(y_hold, pred_set(v)[1]):.3f})")
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set_title('Holdout ROC (internal)'); ax[0].legend(); ax[0].grid(alpha=.3)
ci = CI_ALL.set_index('model').loc[names]; xs_ = np.arange(len(names))
ax[1].errorbar(xs_ - 0.1, ci.dev_auc, yerr=[ci.dev_auc - ci.dev_lo, ci.dev_hi - ci.dev_auc], fmt='o', label='dev out-of-fold'); ax[1].errorbar(xs_ + 0.1, ci.hold_auc, yerr=[ci.hold_auc - ci.hold_lo, ci.hold_hi - ci.hold_auc], fmt='s', label='holdout')
ax[1].set_xticks(xs_); ax[1].set_xticklabels(names, rotation=15); ax[1].set_ylabel('AUC (95% patient-bootstrap CI)'); ax[1].set_title('AUC with CIs: differences are within noise?'); ax[1].legend(); ax[1].grid(alpha=.3)
ab = [(l, v) for l, v, _ in ABLATION_TABLE if v in AVAILABLE]; vals = [CI_ALL.set_index('model').loc[v, 'hold_auc'] for _, v in ab]
ax[2].barh([l for l, _ in ab], vals); ax[2].set_xlim(max(0.5, min(vals) - 0.02), 1.0); ax[2].set_title('Ablation: holdout AUC'); ax[2].grid(alpha=.3, axis='x')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'summary_figure.png'), dpi=250); plt.show()

## 14. Public reproducibility package (reviewer item G)
Exports preprocessing/radiomics/model/training/statistics code pointers, exact fold and holdout assignments (patient IDs), seeds and configs, environment, frozen-model hashes, all result tables, a README with the command-to-table map, licence and citation template, then zips everything. **Upload the zip's content to GitHub and archive a release on Zenodo to obtain the DOI**, then replace "available on reasonable request" in the paper with that DOI.

In [ ]:
def export_repro_package():
    d = REPRO_DIR; os.makedirs(os.path.join(d, 'splits'), exist_ok=True); os.makedirs(os.path.join(d, 'results'), exist_ok=True)
    cfg = dict(CFG); cfg['VARIANTS'] = VARIANTS; cfg['PRIMARY_SEED'] = PS; cfg['HU_WINDOW'] = [HU_MIN, HU_MAX]; cfg['PATCH'] = PATCH; cfg['TASK'] = TASK_NAME; cfg['FINAL_THRESHOLD'] = FINAL_THRESHOLD
    json.dump(cfg, open(os.path.join(d, 'config.json'), 'w'), indent=2, default=str)
    roles = []
    for f, (tr, va, te) in enumerate(make_folds(DF_DEV, CFG['N_SPLITS'], CFG['SPLIT_SEED'])):
        for nm, ix in (('train', tr), ('val', va), ('test_oof', te)): roles += [dict(fold=f, role=nm, patient=DF_DEV.patient.values[i], series=DF_DEV.series.values[i], nod=DF_DEV.nod.values[i], label=DF_DEV.label.values[i]) for i in ix]
    pd.DataFrame(roles).to_csv(os.path.join(d, 'splits', 'development_folds.csv'), index=False)
    pd.DataFrame(dict(patient=DF_HOLD.patient, series=DF_HOLD.series, nod=DF_HOLD.nod, label=DF_HOLD.label)).to_csv(os.path.join(d, 'splits', 'final_holdout_nodules.csv'), index=False)
    ELIG.to_csv(os.path.join(d, 'splits', 'eligible_nodules_lidc.csv'), index=False)
    env = dict(python=sys.version, platform=platform.platform(), torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, numpy=np.__version__, pandas=pd.__version__)
    json.dump(env, open(os.path.join(d, 'environment.json'), 'w'), indent=2); os.system(f"pip freeze > {os.path.join(d, 'requirements.txt')}")
    for fn in glob.glob(os.path.join(STAT_DIR, '*.csv')) + glob.glob(os.path.join(EXPL_DIR, '*.csv')) + glob.glob(os.path.join(FIG_DIR, '*.png')) + [FROZEN_JSON]: shutil.copy(fn, os.path.join(d, 'results'))
    for sub in os.listdir(EXT_DIR):
        if os.path.isdir(os.path.join(EXT_DIR, sub)):
            for fn in glob.glob(os.path.join(EXT_DIR, sub, '*.csv')) + glob.glob(os.path.join(EXT_DIR, sub, '*.txt')): shutil.copy(fn, os.path.join(d, 'results', f'external_{sub}_{os.path.basename(fn)}'))
    open(os.path.join(d, 'LICENSE'), 'w').write('MIT License (placeholder: choose the licence you intend, and add your name/year)\n')
    open(os.path.join(d, 'CITATION.cff'), 'w').write('cff-version: 1.2.0\ntitle: "Multi-scale 3D CNN + radiomics + CBAM for radiologist-assessed lung-nodule malignancy risk (LIDC-IDRI)"\nauthors:\n  - family-names: "TO-FILL"\n    given-names: "TO-FILL"\nversion: "2.0"\ndoi: "TO-BE-ASSIGNED-BY-ZENODO"\nlicense: MIT\n')
    open(os.path.join(d, 'README.md'), 'w').write(f'''# Reproducibility package
Task: {TASK_NAME} on LIDC-IDRI (internal methodological validation; NOT clinical validation).
Data: LIDC-IDRI via NCI Imaging Data Commons (CC BY 3.0, cite Armato et al. Med Phys 2011; doi:10.7937/K9/TCIA.2015.LO9QL9SX). No patient data are redistributed here.

## How to reproduce
1. Open the notebook in Colab, set CFG['MODE']='full', run top to bottom (all steps are resumable). Seeds: {CFG['SEEDS']}; split seed {CFG['SPLIT_SEED']}.
2. Fold / holdout assignments are in splits/ (and are regenerated deterministically from SPLIT_SEED).
3. Model freeze + hashes: results/frozen.json. External validation: place cohorts per section 12 and re-run that cell; no retraining is performed.

## Notebook section -> table / figure
| Section | Output |
|---|---|
| 5 | core + ablation predictions (preds/) |
| 6 | ci_every_model.csv, paired_comparisons.csv, multi_seed_summary.csv |
| 7 | ablation_table.csv, size_overlap_analysis.csv |
| 8 | calibration_metrics.csv, operating_points.csv, threshold_stability.csv, calibration_curves.png, decision_curves.png |
| 9 | label_robustness.csv, pathology_subgroup.csv (if table supplied), patient_level_auc.csv, subgroups_lidc.csv, holdout_errors.csv |
| 11 | explainability_per_nodule.csv, attribution_vs_mask.csv, input_swap.csv, radiomics_permutation_importance.csv, radiologist_review/ |
| 12 | external_<cohort>_* files |

Environment: environment.json, requirements.txt. Add the notebook file itself and tag a release before archiving on Zenodo.
''')
    zp = os.path.join(CFG['WORK_DIR'], f'reproducibility_package_{SUB}.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for root_, _, fs in os.walk(d):
            for fn in fs: z.write(os.path.join(root_, fn), os.path.relpath(os.path.join(root_, fn), d))
    return zp
print('Package written:', export_repro_package())

## 15. Auto-generated, claim-safe results summary (writing guidance from the reviewer)
Generates a summary whose wording is *conditioned on the actual statistics*: it will not say "superior" unless the Holm-corrected test supports it, and it states external-validation status explicitly.

In [ ]:
def fmt_ci(v, lo, hi): return f"{v:.4f} (95% CI {lo:.4f}-{hi:.4f})"
r_full = CI_ALL[CI_ALL.model == 'full'].iloc[0]; L = []
L.append(f"# Results summary ({SUB} mode) — {TASK_NAME}\n")
L.append("## Evidence level\nInternal methodological evaluation on LIDC-IDRI only (patient-level development set + patient-level holdout; the two are *internal*, not independent validation). "
         + ("External validation performed on: " + ", ".join(EXT_RESULTS) + "." if EXT_RESULTS else "**No external validation was performed; do not claim generalisability or clinical applicability.**") + "\n")
L.append(f"## Headline\nFull model: development out-of-fold AUC {fmt_ci(r_full.dev_auc, r_full.dev_lo, r_full.dev_hi)}; holdout AUC {fmt_ci(r_full.hold_auc, r_full.hold_lo, r_full.hold_hi)} at the development-chosen threshold {FINAL_THRESHOLD:.4f} (sensitivity {r_full.hold_sens:.3f}, specificity {r_full.hold_spec:.3f}).\n")
L.append("## Is fusion better than simpler models?")
for r in CMP[(CMP.mode == 'primary') & (CMP.family == 'primary')].itertuples():
    verdict = 'significantly better' if (r.p_boot_holm < 0.05 and r.delta_AUC > 0) else 'NOT significantly different from'
    L.append(f"- [{r.dataset}] full vs {r.B}: ΔAUC {r.delta_AUC:+.4f} (95% CI {r.ci_lo:+.4f} to {r.ci_hi:+.4f}); Holm-adjusted bootstrap p={r.p_boot_holm:.3f}; DeLong p={r.p_delong:.3f} -> the full model is {verdict} {r.B}.")
if FUSION_SIG and not all(FUSION_SIG.values()): L.append("\n**Positioning:** the fused model is not demonstrably better than every simpler comparator -> present the work as a rigorous benchmark / evaluation study, not as a superior classifier.\n")
sec = CMP[(CMP.mode == 'primary') & (CMP.dataset == 'holdout') & (CMP.A.isin(['full', 'img_3s_cbam'])) & (CMP.B.isin(['full_no_cbam', 'img_3s_nocbam']))]
for r in sec.itertuples(): L.append(f"- CBAM ({r.A} vs {r.B}, holdout): ΔAUC {r.delta_AUC:+.4f} (CI {r.ci_lo:+.4f} to {r.ci_hi:+.4f}), Holm p={r.p_boot_holm:.3f} -> " + ("attention gain supported." if (r.p_boot_holm < 0.05 and r.delta_AUC > 0) else "attention gain NOT supported; do not claim CBAM improves performance."))
msc = CMP[(CMP.mode == 'primary') & (CMP.dataset == 'holdout') & (CMP.A == 'img_3s_cbam') & (CMP.B.isin(['img_32', 'img_48', 'img_64']))]
for r in msc.itertuples(): L.append(f"- Multiscale vs {r.B} (image-only, holdout): ΔAUC {r.delta_AUC:+.4f} (CI {r.ci_lo:+.4f} to {r.ci_hi:+.4f}), Holm p={r.p_boot_holm:.3f}.")
L.append("\n## Interpretation rules")
L += ["- Size is a strong shortcut (size-only AUC in the table above); the diameter-overlap analysis shows size does not *fully* explain performance, not that detection is size-independent.",
      "- Input-swap and permutation results are sensitivity diagnostics, not causal importance; radiomics dominance (if the swap table shows it) means do not claim equal image/radiomics complementarity.",
      "- Grad-CAM / occlusion / IG are compared with a centre prior (see attribution_vs_mask.csv): claim localisation only for methods that beat it with CIs excluding zero difference.",
      "- Calibration and decision-curve results are reported; the Youden threshold is statistical, not clinical.",
      "- Labels are radiologist-assessed; indeterminate nodules (mean score 2-4) are excluded, which simplifies the task.",
      "- Do not compare AUC with other LIDC-IDRI papers as proof of superiority (different cohorts, labels, preprocessing, validation).",
      "- Do not describe the model as suitable for clinical deployment before external and ideally prospective validation."]
L.append("\n## Suggested wording changes\nTitle/abstract: 'internal methodological evaluation'; replace 'highly accurate / best / superior' by the quantified, CI-qualified statements above; add 'Predefined statistical analysis plan', 'External validation' (or its explicit absence), and 'Reproducibility' (repository DOI, version, licence, environment) sections.")
open(os.path.join(ROOT, 'RESULTS_SUMMARY.md'), 'w').write('\n'.join(L)); print('\n'.join(L)); print('\nSaved:', os.path.join(ROOT, 'RESULTS_SUMMARY.md'))

**Citation (CC BY 3.0, required):** Armato III SG et al. *The Lung Image Database Consortium (LIDC) and Image Database Resource Initiative (IDRI)*. Med Phys 2011; TCIA: doi:10.7937/K9/TCIA.2015.LO9QL9SX. Data accessed via NCI Imaging Data Commons.